# BTL-K03 — Mở rộng hình học embedding bằng PH H₀/H₁

**Đúng phạm vi đề tài:** Diginetica; GRU4Rec, NARM, SASRec; seeds 2024/2025.
Phân tích popular/medium/rare, norm/category/cosine và cấu trúc lân cận nhiều thang đo.
Các đối chứng và phương pháp được liệt kê trong config khóa trước khi chạy. Đây là ứng viên nghiên cứu,
không bảo đảm cải thiện rare Recall hay giảm thiên lệch.

Mỗi run train đầy đủ, chọn checkpoint bằng validation MRR, sau đó đánh giá test.
Config khóa trước khi chạy, không dò tham số theo test. Cold/padding đóng băng.
Lịch regularization lấy từ config. Với interval > 1, regularization định kỳ không
hoàn toàn tương đương train PH mọi batch. Hai GPU chạy hai thí nghiệm độc lập.

Không có traffic allocation, audience discovery hoặc KuaiRand trong notebook này.
Lịch sử nghiên cứu đã dùng test Diginetica trước đây: benchmark này không phải một
holdout chưa từng được xem; cần dataset/holdout mới cho xác nhận độc lập sau cùng.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
WORK = Path('/kaggle/working/btl-k03-ph-study')
WORK.mkdir(parents=True, exist_ok=True)
PAYLOAD = {'BTL-K03-tong-quan-de-tai.md': "# BTL-K03: Thiên lệch độ phổ biến và hình học không gian embedding trong mô hình SBR\n\n## 1. Tóm tắt đề tài trong một câu\n\nĐề tài yêu cầu huấn luyện ba mô hình gợi ý theo phiên **GRU4Rec, NARM và SASRec** trên dữ liệu **Diginetica**, đo mức độ các mô hình ưu tiên item phổ biến ở danh sách gợi ý, sau đó phân tích embedding của item phổ biến, item hiếm và item cold để tìm hiểu xem thiên lệch ở đầu ra có đi kèm với những khác biệt hình học trong không gian embedding hay không.\n\nQuy trình tổng quát:\n\n```text\nDữ liệu Diginetica\n        ↓\nTiền xử lý và thiết lập giao thức cold-start 20%\n        ↓\nHuấn luyện GRU4Rec, NARM, SASRec với 2 seed\n        ↓\nSinh danh sách gợi ý top-20\n        ↓\nĐo popularity bias ở đầu ra\n        ↓\nTrích xuất và phân tích item embedding\n        ↓\nLiên hệ hình học embedding với tần suất item được gợi ý\n        ↓\nSo sánh ba mô hình và đưa ra kết luận chẩn đoán\n```\n\nĐây là đề tài **chẩn đoán và phân tích**, không phải đề tài xây dựng một thuật toán khử thiên lệch mới.\n\n---\n\n## 2. Các khái niệm nền tảng\n\n### 2.1. Item là gì?\n\nItem là đối tượng mà hệ thống muốn gợi ý. Trong Diginetica, item thường là một sản phẩm trên website thương mại điện tử.\n\nVí dụ:\n\n```text\nitem_101 = một chiếc điện thoại\nitem_205 = một bộ sạc\nitem_327 = một chiếc ốp lưng\n```\n\n### 2.2. Session là gì?\n\nSession là chuỗi hành động xảy ra trong một lần truy cập. Một session có thể chứa nhiều item theo thứ tự thời gian.\n\nVí dụ:\n\n```text\nSession 01: điện thoại → ốp lưng → sạc dự phòng → tai nghe\n```\n\nTừ ba item đầu, mô hình cố gắng dự đoán item tiếp theo là `tai nghe`.\n\n### 2.3. SBR là gì?\n\n**SBR — Session-Based Recommendation** là hệ thống gợi ý dựa trên chuỗi hành động trong phiên hiện tại. Mục tiêu thường là dự đoán item tiếp theo mà người dùng sẽ xem, nhấp hoặc mua.\n\nĐầu vào của mô hình có dạng:\n\n```text\n[item_12, item_35, item_08]\n```\n\nĐầu ra là điểm dự đoán cho các item ứng viên và một danh sách top-k, chẳng hạn top-20:\n\n```text\n[item_41, item_03, item_92, ..., item_17]\n```\n\n### 2.4. Popular, rare, warm và cold item\n\n- **Popular item:** xuất hiện nhiều lần trong tập huấn luyện.\n- **Rare item:** xuất hiện ít lần trong tập huấn luyện.\n- **Warm item:** đã từng xuất hiện trong dữ liệu huấn luyện.\n- **Cold item:** không xuất hiện trong dữ liệu huấn luyện, nên mô hình chưa có tương tác để học biểu diễn của item đó.\n\nCold item có thể được xem là trường hợp cực đoan của phổ tần suất:\n\n```text\nPopular: tần suất rất lớn\nMedium:  tần suất trung bình\nRare:    tần suất nhỏ nhưng lớn hơn 0\nCold:    tần suất bằng 0 trong train\n```\n\n### 2.5. Popularity bias là gì?\n\nPopularity bias, hay thiên lệch độ phổ biến, là hiện tượng mô hình có xu hướng gợi ý các item vốn đã phổ biến nhiều hơn mức hợp lý, trong khi item hiếm hoặc item mới ít được xuất hiện.\n\nVòng lặp có thể xảy ra:\n\n```text\nItem đã phổ biến\n      ↓\nĐược mô hình gợi ý nhiều\n      ↓\nNhận thêm nhiều tương tác\n      ↓\nCàng trở nên phổ biến\n```\n\nĐề tài không chỉ muốn biết mô hình dự đoán đúng hay sai, mà còn muốn biết **lượt hiển thị trong top-20 được phân phối cho các item như thế nào**.\n\n### 2.6. Embedding là gì?\n\nMỗi item ID được mô hình biểu diễn bằng một vector nhiều chiều:\n\n```text\nitem A = [0.2, 0.8, -0.1, ...]\nitem B = [0.4, 0.7,  0.0, ...]\n```\n\nNếu embedding dimension bằng 64 thì mỗi item có một vector gồm 64 số. Toàn bộ các vector tạo thành một không gian embedding 64 chiều.\n\nTrong quá trình huấn luyện, vector của các warm item được cập nhật bằng gradient. Item xuất hiện nhiều có thể nhận nhiều lần cập nhật hơn item hiếm. Cold item không xuất hiện trong train thường không nhận được cập nhật có ý nghĩa.\n\n---\n\n## 3. Câu chuyện nghiên cứu của đề tài\n\nĐề tài có hai góc nhìn chính.\n\n### Góc nhìn 1: Đầu ra của mô hình\n\nCâu hỏi là:\n\n> Danh sách top-20 có bị tập trung quá nhiều vào một số item phổ biến hay không?\n\nCác chỉ số chính:\n\n- Catalog Coverage@20.\n- Gini của tần suất được gợi ý.\n- Average Recommendation Popularity.\n\n### Góc nhìn 2: Bên trong mô hình\n\nCâu hỏi là:\n\n> Embedding của popular, rare và cold item có khác nhau như thế nào?\n\nCác phân tích chính:\n\n- Tương quan giữa tần suất train và norm embedding.\n- Cosine similarity của item cùng category và khác category.\n- Vị trí của cold item so với warm item cùng category.\n- Trực quan hóa bằng t-SNE.\n\n### Phần kết nối hai góc nhìn\n\nĐây là phần tạo chiều sâu cho đề tài:\n\n> Mô hình có popularity bias mạnh ở đầu ra có đồng thời thể hiện sự phân hóa mạnh hơn giữa embedding của popular và rare item hay không?\n\nCần lưu ý rằng mối liên hệ thống kê không tự động chứng minh quan hệ nhân quả. Kết luận nên dùng các cách diễn đạt như:\n\n- “Có mối liên hệ”.\n- “Đi kèm với”.\n- “Kết quả gợi ý một cơ chế khả dĩ”.\n\nKhông nên kết luận:\n\n- “Norm lớn chắc chắn gây ra popularity bias”.\n- “t-SNE chứng minh item phổ biến tạo thành cụm”.\n\n---\n\n## 4. Mục tiêu và câu hỏi nghiên cứu\n\n### 4.1. Mục tiêu tổng quát\n\nĐịnh lượng popularity bias ở đầu ra và liên hệ nó với cấu trúc hình học của item embedding trong ba mô hình SBR, bao gồm vị trí của cold item.\n\n### 4.2. Các câu hỏi bắt buộc từ mô tả đề tài\n\n1. Mô hình nào có popularity bias lớn nhất theo Gini và Coverage@20?\n2. Norm embedding có tương quan với tần suất xuất hiện trong train không? Mức độ tương quan ở mỗi mô hình ra sao?\n3. Cold item có vị trí như thế nào so với warm item cùng category?\n\n### 4.3. Các câu hỏi phân tích giúp kết nối kết quả\n\n1. Item xuất hiện nhiều trong train có được gợi ý nhiều hơn không?\n2. Item có norm embedding lớn có được gợi ý nhiều hơn không?\n3. Quan hệ giữa norm và số lần được gợi ý có còn tồn tại khi so sánh các item có tần suất train gần nhau không?\n4. Item cùng category có gần nhau hơn item khác category không?\n5. Rare item có nằm xa tâm category hơn popular item không?\n6. Các kết luận có ổn định qua hai random seed không?\n\nCác câu hỏi ở mục 4.3 làm phần phân tích chặt chẽ hơn, nhưng cần giữ phạm vi phù hợp với thời gian của bài tập lớn.\n\n---\n\n## 5. Dữ liệu Diginetica\n\n### 5.1. Diginetica là gì?\n\nDiginetica là dữ liệu hành vi thương mại điện tử được công bố cho **CIKM Cup 2016**. Dữ liệu gồm các session, lượt xem hoặc tương tác với sản phẩm, thời gian và metadata sản phẩm đã được ẩn danh.\n\nCác nguồn tham khảo:\n\n- [Trang CIKM Cup 2016 trên CodaLab](https://competitions.codalab.org/competitions/11161)\n- [Kho RecSysDatasets của RecBole](https://github.com/RUCAIBox/RecSysDatasets)\n- [Hướng dẫn chuyển đổi Diginetica](https://github.com/RUCAIBox/RecSysDatasets/blob/master/conversion_tools/usage/DIGINETICA.md)\n- [Bản dữ liệu được liên kết trong hướng dẫn RecBole](https://drive.google.com/open?id=0B7XZSACQf0KdXzZFS21DblRxQ3c)\n\n### 5.2. Các file quan trọng\n\nTheo hướng dẫn chuyển đổi của RecBole, ba file quan trọng là:\n\n| File | Nội dung chính | Mục đích sử dụng |\n|---|---|---|\n| `train-item-views.csv` | Lượt xem item theo session và thời gian | Tạo chuỗi để huấn luyện SBR |\n| `product-categories.csv` | Ánh xạ item với category | Phân tích cosine và category |\n| `products.csv` | Metadata sản phẩm | Kiểm tra, đối chiếu hoặc mở rộng |\n\nBiểu diễn khái quát của một lượt tương tác:\n\n```text\nsession_id | item_id | timestamp/order | category\n```\n\n### 5.3. Vì sao đề tài chọn Diginetica?\n\n#### Phù hợp với SBR\n\nDữ liệu chứa chuỗi hành động theo session, phù hợp trực tiếp với bài toán dự đoán item tiếp theo.\n\n#### Có phân bố popular–rare\n\nDữ liệu thương mại điện tử thường có phân bố đuôi dài: một số ít item xuất hiện nhiều, trong khi phần lớn item xuất hiện ít. Đây là điều kiện cần để nghiên cứu popularity bias.\n\n#### Có category\n\nÁnh xạ category cho phép kiểm tra xem không gian embedding có đặt các item cùng loại gần nhau hay không.\n\n#### Là benchmark phổ biến\n\nDiginetica được sử dụng trong nhiều nghiên cứu SBR, đồng thời có công cụ chuyển đổi để dùng với RecBole.\n\n#### Kích thước khả thi\n\nDữ liệu đủ lớn để thể hiện thiên lệch nhưng vẫn có thể chạy trên Colab hoặc Kaggle. Số session và item sau tiền xử lý có thể khác giữa các nghiên cứu vì quy tắc lọc khác nhau. Mô tả đề tài nhắc tới khoảng 40.000 item sau khi lọc.\n\n### 5.4. Nên dùng dữ liệu thô hay dữ liệu đã xử lý?\n\nĐề tài cần category và cần tự tạo giao thức cold-start 20%, vì vậy nên ưu tiên dữ liệu thô rồi tiền xử lý theo đúng tài liệu môn học.\n\nBản RecBole đã xử lý sẵn có thể tiện để chạy thử, nhưng có rủi ro:\n\n- Đã lọc item hoặc session theo quy tắc khác.\n- Không giữ đầy đủ category.\n- Không sử dụng đúng cold-start 20%.\n- Thay đổi tập item được dùng làm mẫu số của Coverage và Gini.\n\nNếu Mục 1.6 của tài liệu môn học cung cấp dữ liệu hoặc code chia tập riêng thì phải ưu tiên tài liệu đó.\n\n---\n\n## 6. Ba mô hình cần huấn luyện\n\n### 6.1. GRU4Rec\n\nGRU4Rec dùng mạng GRU để đọc chuỗi item trong session. Trạng thái ẩn của GRU tóm tắt những gì đã xảy ra trong phiên và được dùng để dự đoán item tiếp theo.\n\nKhái quát:\n\n```text\nitem_1 → item_2 → item_3\n             ↓ GRU\n       biểu diễn session\n             ↓\n      điểm cho các item\n```\n\n### 6.2. NARM\n\nNARM kết hợp GRU và attention. Ngoài việc tóm tắt toàn bộ session, mô hình cố gắng nhận ra item nào trong phiên có ảnh hưởng lớn hơn đến ý định hiện tại.\n\n### 6.3. SASRec\n\nSASRec dùng self-attention để mô hình hóa quan hệ giữa các vị trí trong chuỗi. Nó có thể tập trung vào nhiều vị trí khác nhau mà không chỉ phụ thuộc vào trạng thái tuần tự như GRU.\n\n### 6.4. Yêu cầu so sánh công bằng\n\nBa mô hình cần được huấn luyện trên cùng:\n\n- Tập train, validation và test.\n- Candidate set.\n- Quy tắc lọc item/session.\n- Cách định nghĩa popular, medium, rare và cold.\n- Embedding dimension nếu có thể so sánh thống nhất.\n- Top-k bằng 20.\n- Hai random seed đã xác định trước.\n\nCác siêu tham số riêng của mô hình có thể khác, nhưng mọi khác biệt phải được ghi lại.\n\n---\n\n## 7. Tiền xử lý dữ liệu và giao thức cold-start\n\n### 7.1. Tiền xử lý session\n\nCác bước thường gặp:\n\n1. Đọc `train-item-views.csv`.\n2. Sắp xếp tương tác theo thời gian trong từng session.\n3. Loại các bản ghi lỗi hoặc thiếu item ID.\n4. Loại session chỉ có một tương tác vì không tạo được cặp đầu vào–mục tiêu.\n5. Có thể loại item quá hiếm nếu tài liệu yêu cầu.\n6. Ánh xạ item sang category.\n7. Chia train, validation và test theo đúng giao thức.\n\nMột session:\n\n```text\n[A, B, C, D]\n```\n\ncó thể sinh các mẫu next-item:\n\n```text\n[A]       → B\n[A, B]    → C\n[A, B, C] → D\n```\n\n### 7.2. Không được tự chọn cách chia cold-start tùy ý\n\n“Cold-start 20%” có thể được hiểu theo nhiều cách:\n\n- Chọn 20% item làm cold rồi loại mọi tương tác của chúng khỏi train.\n- Dùng chia theo thời gian và xem item chỉ xuất hiện ở test là cold.\n- Chọn cold theo category hoặc theo một quy tắc khác.\n\nDo đó phải kiểm tra **Mục 1.6 và Option B** trong tài liệu gốc. Đây là chi tiết quyết định tính đúng đắn của toàn bộ thí nghiệm.\n\n### 7.3. Vấn đề kỹ thuật của cold item trong mô hình ID-based\n\nNếu một item hoàn toàn không có trong train:\n\n- Nó có thể không được RecBole đưa vào vocabulary.\n- Nếu vẫn được đăng ký trong vocabulary, embedding của nó thường chỉ là giá trị khởi tạo vì không nhận gradient.\n- Mô hình không có thông tin để hiểu quan hệ giữa cold item và các item khác.\n- Việc đưa cold item vào danh sách ứng viên có thể không có ý nghĩa đối với baseline chỉ dựa trên ID.\n\nVì mô tả đề tài ghi “cold item chỉ để minh họa Option B”, nên cách hiểu an toàn là:\n\n- Phân tích định lượng chính tập trung vào warm item.\n- Cold item được giữ để đánh dấu và minh họa trong không gian embedding.\n- Không dùng hiệu suất gợi ý cold item làm kết luận chính nếu giao thức không hỗ trợ hợp lệ.\n\n---\n\n## 8. Đánh giá chất lượng dự đoán\n\nĐây không phải trọng tâm chính của đề tài, nhưng nên có một số chỉ số làm **sanity check** để chứng minh mô hình hoạt động bình thường.\n\nCác chỉ số có thể dùng:\n\n- Recall@20.\n- MRR@20.\n- NDCG@20.\n\nLý do cần sanity check:\n\n> Một mô hình có Gini thấp chưa chắc tốt; nó có thể gợi ý gần như ngẫu nhiên nên phân phối rất đều nhưng dự đoán không chính xác.\n\nCần báo cáo đồng thời chất lượng và thiên lệch, nhưng không biến bài toán thành cuộc thi tối ưu accuracy.\n\n---\n\n## 9. Đo popularity bias ở đầu ra\n\nSau khi huấn luyện, mỗi mô hình sinh top-20 cho các session test. Từ toàn bộ danh sách này, đếm số lần mỗi item được gợi ý.\n\n### 9.1. Recommendation count và exposure share\n\nVới item \\(i\\):\n\n\\[\nrec\\_count_i = \\text{số lần item } i \\text{ xuất hiện trong tất cả danh sách top-20}\n\\]\n\nNếu có \\(N\\) session test thì tổng số vị trí gợi ý là:\n\n\\[\nN \\times 20\n\\]\n\nExposure share:\n\n\\[\nexposure\\_share_i = \\frac{rec\\_count_i}{N \\times 20}\n\\]\n\nVí dụ có 10.000 session test:\n\n```text\nTổng vị trí gợi ý = 10.000 × 20 = 200.000\n```\n\nNếu item A xuất hiện 8.000 lần thì nó chiếm 4% tổng lượt hiển thị.\n\n### 9.2. Catalog Coverage@20\n\n\\[\nCoverage@20 =\n\\frac{\\text{số item khác nhau xuất hiện trong các top-20}}\n{\\text{tổng số item hợp lệ trong catalog}}\n\\]\n\nDiễn giải:\n\n- Coverage cao: mô hình sử dụng nhiều item trong catalog.\n- Coverage thấp: mô hình lặp lại một nhóm nhỏ item.\n\nCần ghi rõ mẫu số là:\n\n- Toàn bộ warm candidate item, hay\n- Toàn bộ item catalog bao gồm cold item.\n\nTrong baseline ID-based, dùng tập candidate hợp lệ thường dễ diễn giải hơn. Phải áp dụng cùng một định nghĩa cho cả ba mô hình.\n\n### 9.3. Gini của tần suất được gợi ý\n\nGini đo mức độ tập trung của `rec_count`:\n\n- Gini gần 0: exposure tương đối đồng đều.\n- Gini cao: exposure tập trung vào một số ít item.\n\nKhi tính Gini, cần quyết định có đưa các item được gợi ý 0 lần vào vector hay không. Đối với đo catalog concentration, nên tính cả các candidate item hợp lệ có `rec_count = 0`, đồng thời ghi rõ định nghĩa trong báo cáo.\n\n### 9.4. Average Recommendation Popularity\n\nChỉ số này đo độ phổ biến trong tập train của các item được gợi ý:\n\n\\[\nARP = \\frac{1}{N\\times K}\n\\sum_{u=1}^{N}\\sum_{i\\in Rec_K(u)} popularity(i)\n\\]\n\nTrong đó `popularity(i)` thường là số lần item xuất hiện trong train. Nếu dùng `log(1 + frequency)` để giảm độ lệch, phải ghi rõ và dùng thống nhất.\n\nDiễn giải:\n\n- ARP cao: mô hình thường gợi ý item vốn đã phổ biến.\n- ARP thấp: danh sách có nhiều item ít phổ biến hơn.\n\n### 9.5. Cách kết luận popularity bias\n\nMột mô hình thường được xem là thiên lệch mạnh hơn nếu nó có tổ hợp:\n\n```text\nCoverage thấp\n+ Gini cao\n+ Average Recommendation Popularity cao\n```\n\nKhông nên dựa vào một chỉ số duy nhất.\n\n---\n\n## 10. Hình học không gian embedding\n\n### 10.1. Trích xuất embedding\n\nSau khi huấn luyện, lấy ma trận item embedding từ checkpoint của từng mô hình. Trong RecBole, các mô hình thường có tham số tương tự `item_embedding.weight`, nhưng cần kiểm tra đúng implementation và loại bỏ hàng padding nếu có.\n\nCần bảo đảm:\n\n- Ánh xạ từ internal item ID về raw item ID là chính xác.\n- Không nhầm item embedding với position embedding hoặc session representation.\n- Embedding được lấy từ đúng checkpoint tốt nhất.\n- Embedding của ba mô hình được phân tích theo cùng quy trình.\n\n### 10.2. Norm embedding\n\nVới embedding \\(e_i\\) có \\(d\\) chiều:\n\n\\[\n\\lVert e_i\\rVert_2 =\n\\sqrt{e_{i1}^2 + e_{i2}^2 + \\dots + e_{id}^2}\n\\]\n\nNorm là độ dài của vector. Phân tích chính:\n\n\\[\nSpearman(train\\_frequency_i, \\lVert e_i\\rVert_2)\n\\]\n\nDiễn giải hệ số Spearman \\(\\rho\\):\n\n- Gần `+1`: item phổ biến hơn có xu hướng có norm lớn hơn.\n- Gần `0`: chưa thấy quan hệ đơn điệu rõ ràng.\n- Gần `-1`: item phổ biến hơn có xu hướng có norm nhỏ hơn.\n\nCần báo cáo `p-value` hoặc khoảng tin cậy. Với số lượng item rất lớn, `p-value` có thể rất nhỏ ngay cả khi hiệu ứng yếu, nên phải báo cáo cả độ lớn của \\(\\rho\\).\n\n### 10.3. Vì sao norm có thể liên quan đến điểm gợi ý?\n\nNhiều mô hình tính điểm gần với dot product:\n\n\\[\nscore(i) = h^\\top e_i\n\\]\n\nTrong đó:\n\n- \\(h\\) là vector biểu diễn session.\n- \\(e_i\\) là item embedding.\n\nVề mặt hình học:\n\n\\[\nh^\\top e_i = \\lVert h\\rVert \\lVert e_i\\rVert \\cos(\\theta)\n\\]\n\nVì vậy, nếu hướng tương đối thuận lợi, item có norm lớn có thể dễ đạt điểm cao hơn. Tuy nhiên, đây chỉ là cơ chế khả dĩ; score còn phụ thuộc vào hướng vector, kiến trúc, bias term và quá trình huấn luyện.\n\n### 10.4. Cosine similarity\n\nCosine similarity giữa hai embedding:\n\n\\[\ncos(e_i,e_j)=\n\\frac{e_i^\\top e_j}\n{\\lVert e_i\\rVert\\lVert e_j\\rVert}\n\\]\n\nSo sánh tối thiểu:\n\n- Warm–warm cùng category.\n- Warm–warm khác category.\n\nCó thể bổ sung để minh họa:\n\n- Cold–warm cùng category.\n\nNếu cosine trung bình của cặp cùng category cao hơn cặp khác category, mô hình đã học được một phần cấu trúc category từ dữ liệu tương tác.\n\nKhông nên tạo toàn bộ ma trận cosine cho khoảng 40.000 item:\n\n\\[\n40.000 \\times 40.000 = 1{,}6 \\text{ tỷ cặp}\n\\]\n\nNên lấy mẫu số lượng cặp bằng nhau giữa các nhóm, cố định seed lấy mẫu và báo cáo khoảng tin cậy.\n\n### 10.5. Khoảng cách đến tâm category\n\nĐây là phân tích bổ sung hữu ích. Tâm của category \\(c\\):\n\n\\[\ncentroid_c = \\frac{1}{N_c}\\sum_{i\\in c}e_i\n\\]\n\nSau đó đo khoảng cách hoặc cosine giữa từng item với tâm category của nó.\n\nVí dụ giả định:\n\n| Nhóm | Khoảng cách trung bình đến tâm category |\n|---|---:|\n| Popular warm | 0.42 |\n| Medium warm | 0.58 |\n| Rare warm | 0.79 |\n| Cold | 1.31 |\n\nNếu popular item gần tâm category hơn rare item, điều đó cho thấy chất lượng hoặc mức độ ổn định của biểu diễn có thể khác theo tần suất.\n\n### 10.6. t-SNE\n\nt-SNE giảm embedding 64/128 chiều xuống hai chiều để trực quan hóa. Mỗi điểm là một item.\n\nNên tạo ít nhất:\n\n1. Hình tô màu theo bin tần suất: popular, medium, rare.\n2. Hình tô màu theo category.\n3. Đánh dấu cold item bằng ký hiệu riêng.\n\nChỉ lấy mẫu khoảng 5.000–10.000 item. Cần ghi:\n\n- Số điểm.\n- Perplexity.\n- Random seed.\n- Các bước tiền xử lý như PCA trước t-SNE nếu có.\n\nGiới hạn diễn giải:\n\n- t-SNE ưu tiên bảo toàn cấu trúc lân cận cục bộ.\n- Khoảng cách giữa hai cụm trên hình không nhất thiết phản ánh đúng khoảng cách trong không gian gốc.\n- Hình có thể thay đổi theo seed và perplexity.\n- Kết luận định lượng phải dựa vào norm, cosine, centroid hoặc các số đo trong không gian gốc.\n\n---\n\n## 11. Bảng phân tích item-level\n\nSau khi huấn luyện mỗi mô hình, nên tạo một bảng trung tâm. Đây là dữ liệu nối phần đầu ra với phần embedding.\n\n| Cột | Ý nghĩa |\n|---|---|\n| `model` | GRU4Rec, NARM hoặc SASRec |\n| `seed` | Seed thí nghiệm |\n| `item_id` | ID item gốc |\n| `category_id` | Category của item |\n| `train_freq` | Số lần xuất hiện trong train |\n| `popularity_group` | popular, medium, rare hoặc cold |\n| `is_cold` | Item có cold hay không |\n| `embedding_norm` | Norm L2 của embedding |\n| `rec_count` | Số lần xuất hiện trong các top-20 |\n| `exposure_share` | Tỷ trọng trên toàn bộ vị trí top-20 |\n| `distance_to_category_centroid` | Khoảng cách đến tâm category, nếu tính |\n\nVí dụ minh họa:\n\n| item_id | category | train_freq | group | cold | norm | rec_count | exposure |\n|---|---|---:|---|---:|---:|---:|---:|\n| 101 | phone | 5.000 | popular | 0 | 4.82 | 8.200 | 0.0410 |\n| 102 | phone | 120 | medium | 0 | 2.10 | 250 | 0.00125 |\n| 103 | phone | 5 | rare | 0 | 0.91 | 2 | 0.00001 |\n| 104 | phone | 0 | cold | 1 | 0.36 | NA | NA |\n\n### Quy tắc chia popularity group\n\nKhông có một cách duy nhất. Có thể dùng:\n\n- Quantile theo `train_freq`.\n- Quantile theo `log1p(train_freq)`.\n- Nhóm head/middle/tail theo tỷ trọng tương tác tích lũy.\n- Các khoảng tần suất cố định nếu tài liệu môn học quy định.\n\nĐiều quan trọng là:\n\n- Xác định quy tắc trước khi so sánh mô hình.\n- Dùng cùng quy tắc cho cả ba mô hình và hai seed.\n- Cold luôn có `train_freq = 0` và tách khỏi các quantile warm.\n- Ghi rõ số item và số tương tác trong mỗi nhóm.\n\n---\n\n## 12. Liên hệ hình học embedding với tần suất được gợi ý\n\n### 12.1. Ba quan hệ cốt lõi\n\nTừ bảng item-level, tính ba tương quan Spearman:\n\n| Tương quan | Câu hỏi |\n|---|---|\n| `train_freq ↔ embedding_norm` | Item phổ biến có norm lớn hơn không? |\n| `train_freq ↔ rec_count` | Item phổ biến có được gợi ý nhiều hơn không? |\n| `embedding_norm ↔ rec_count` | Item có norm lớn có được gợi ý nhiều hơn không? |\n\nVí dụ giả định:\n\n| Mô hình | freq–norm | freq–rec | norm–rec |\n|---|---:|---:|---:|\n| GRU4Rec | 0.31 | 0.61 | 0.35 |\n| NARM | 0.52 | 0.70 | 0.54 |\n| SASRec | 0.76 | 0.84 | 0.73 |\n\nNếu SASRec đồng thời có:\n\n- Gini cao nhất.\n- Coverage thấp nhất.\n- Tương quan `freq–norm` mạnh nhất.\n- Tương quan `norm–rec` mạnh nhất.\n\nthì có thể nói popularity bias ở đầu ra **đi kèm với** sự phân hóa embedding mạnh hơn theo tần suất.\n\n### 12.2. Biến gây nhiễu: train frequency\n\nMột item phổ biến có thể vừa có norm lớn vừa được gợi ý nhiều chỉ vì nó xuất hiện nhiều trong train:\n\n```text\ntrain_freq ──→ embedding_norm\n     │\n     └───────→ rec_count\n```\n\nKhi đó tương quan `norm–rec` chưa chứng minh norm có vai trò độc lập.\n\n#### Cách kiểm tra đơn giản\n\nTrong từng popularity bin, tính lại tương quan giữa norm và rec_count. Khi các item có tần suất gần nhau, ảnh hưởng của frequency được giảm bớt.\n\n#### Cách kiểm tra nâng cao\n\nDùng mô hình hồi quy:\n\n\\[\n\\log(1+rec\\_count_i)=\n\\beta_0+\n\\beta_1\\log(1+train\\_freq_i)+\n\\beta_2 norm_i\n\\]\n\nNếu \\(\\beta_2\\) vẫn dương sau khi kiểm soát frequency, có bằng chứng mạnh hơn rằng norm chứa thông tin liên quan đến mức exposure. Tuy nhiên, đây vẫn là nghiên cứu quan sát, không phải thí nghiệm nhân quả.\n\n### 12.3. Các biểu đồ nên có\n\n1. Scatter plot `log train_freq` và `embedding_norm`.\n2. Scatter plot `log train_freq` và `log(1 + rec_count)`.\n3. Scatter plot `embedding_norm` và `log(1 + rec_count)`, tô màu theo popularity group.\n4. Boxplot norm theo popular/medium/rare/cold.\n5. Boxplot rec_count hoặc exposure theo popularity group.\n6. Biểu đồ t-SNE theo frequency bin.\n7. Biểu đồ t-SNE theo category, đánh dấu cold item.\n\n---\n\n## 13. Cách đọc kết quả cold item\n\nCold item không được học từ tương tác nên embedding của nó thường là vector khởi tạo. Vì vậy:\n\n- Norm của cold item có thể phản ánh initializer hơn là dữ liệu.\n- Vị trí t-SNE có thể ngẫu nhiên.\n- Cosine với warm item cùng category có thể gần 0.\n- Category của cold item không tự động được mô hình ID-based sử dụng.\n\nMột kết luận phù hợp:\n\n> Cold item không nằm gần các warm item cùng category và không hình thành cấu trúc category rõ ràng. Kết quả minh họa giới hạn của baseline chỉ dựa trên item ID: khi item không xuất hiện trong train, mô hình không có tín hiệu để học một biểu diễn có ý nghĩa cho nó.\n\nKhông nên kết luận:\n\n> Cold item nằm xa vì mô hình cố tình thiên vị nó.\n\nCold-start và popularity bias liên quan nhưng không hoàn toàn giống nhau:\n\n- Popularity bias nói về sự phân bổ attention/exposure theo độ phổ biến.\n- Cold-start nói về việc mô hình thiếu thông tin để biểu diễn hoặc dự đoán item hoàn toàn mới.\n\n---\n\n## 14. Ví dụ về cách tổng hợp kết quả\n\nGiả sử thí nghiệm cho bảng sau:\n\n| Mô hình | Recall@20 | Coverage@20 ↑ | Gini ↓ | ARP ↓ | Spearman freq–norm |\n|---|---:|---:|---:|---:|---:|\n| GRU4Rec | 0.41 | 0.32 | 0.62 | 125 | 0.31 |\n| NARM | 0.44 | 0.25 | 0.70 | 158 | 0.52 |\n| SASRec | 0.46 | 0.17 | 0.81 | 213 | 0.76 |\n\nMột kết luận hợp lý:\n\n> Trong kết quả giả định, SASRec đạt Recall@20 cao nhất nhưng có Coverage@20 thấp nhất, Gini và ARP cao nhất. Điều này cho thấy mô hình tạo danh sách chính xác hơn nhưng tập trung mạnh hơn vào item phổ biến. SASRec cũng có tương quan mạnh nhất giữa tần suất train và norm embedding. Phân tích item-level cho thấy item có norm lớn xuất hiện thường xuyên hơn trong top-20, kể cả khi quan hệ suy yếu sau khi kiểm soát tần suất. Các bằng chứng cho thấy popularity bias ở đầu ra đi kèm với sự phân hóa embedding theo độ phổ biến, nhưng chưa đủ để kết luận norm là nguyên nhân trực tiếp.\n\nLưu ý toàn bộ số trong bảng chỉ là ví dụ minh họa, không phải kết quả dự kiến phải đạt được.\n\n---\n\n## 15. Hai seed và độ ổn định\n\nĐề tài yêu cầu kiểm tra trên hai seed. Mỗi mô hình cần chạy ít nhất hai lần với cùng cấu hình nhưng seed khác nhau.\n\nVí dụ:\n\n```text\nseed = 2024\nseed = 2025\n```\n\nCần kiểm tra:\n\n- Thứ hạng mô hình theo Gini có đổi không?\n- Thứ hạng theo Coverage có đổi không?\n- Dấu và độ lớn của Spearman có ổn định không?\n- Hình t-SNE có thay đổi quá nhiều không?\n\nBáo cáo có thể dùng:\n\n```text\nmean ± standard deviation qua 2 seed\n```\n\nVới chỉ hai seed, độ lệch chuẩn chỉ cung cấp dấu hiệu ban đầu, không phải ước lượng độ bất định rất chắc chắn. Có thể dùng bootstrap trên item để bổ sung khoảng tin cậy cho các thống kê item-level.\n\n---\n\n## 16. Công cụ dự kiến\n\n### RecBole/PyTorch\n\n- Huấn luyện GRU4Rec, NARM, SASRec.\n- Sinh top-20.\n- Lưu checkpoint.\n- Trích xuất embedding.\n\n### pandas và NumPy\n\n- Tiền xử lý dữ liệu.\n- Đếm tần suất train và recommendation count.\n- Tạo bảng item-level.\n- Tính Coverage và Gini.\n\n### SciPy\n\n- `scipy.stats.spearmanr` để tính Spearman và p-value.\n\n### scikit-learn\n\n- PCA nếu cần giảm chiều sơ bộ.\n- t-SNE để trực quan hóa.\n- Cosine similarity hoặc nearest neighbors.\n\n### Matplotlib/Seaborn\n\n- Boxplot, scatter plot, histogram và t-SNE plot.\n\n### Môi trường chạy\n\n- GPU Colab hoặc Kaggle cho huấn luyện.\n- CPU có thể chạy phần thống kê và t-SNE mẫu 5.000–10.000 item.\n\nƯớc lượng trong đề bài: ba mô hình × hai seed khoảng 5–10 giờ GPU, nhưng thời gian thực tế phụ thuộc cấu hình và cách tiền xử lý.\n\n---\n\n## 17. Kế hoạch thực hiện trong 10 tuần\n\n### Tuần 1–2: Dữ liệu và thiết kế thí nghiệm\n\n- Tải dữ liệu thô.\n- Đọc Mục 1.5, 1.6 và Option B.\n- Khảo sát schema và thống kê dữ liệu.\n- Xác định cách chia train/validation/test.\n- Xác định cold-start 20%.\n- Xác định popularity bins và candidate catalog.\n- Viết tài liệu định nghĩa các chỉ số trước khi chạy.\n\n### Tuần 3: Mô hình đầu tiên\n\n- Chuyển dữ liệu sang định dạng RecBole.\n- Huấn luyện một mô hình, chẳng hạn GRU4Rec.\n- Sinh top-20.\n- Tính Coverage, Gini và ARP.\n- Kiểm tra Recall/MRR/NDCG như sanity check.\n\nMốc 1: một mô hình chạy hoàn chỉnh và có chỉ số đầu ra.\n\n### Tuần 4–5: Hoàn thành ba mô hình\n\n- Huấn luyện NARM và SASRec.\n- Chuẩn hóa việc lưu checkpoint, top-20 và log.\n- Chạy seed thứ nhất cho cả ba mô hình.\n- Kiểm tra sự công bằng của candidate set và cách đánh giá.\n\n### Tuần 6: Norm và tần suất\n\n- Trích xuất embedding đúng item mapping.\n- Tính norm.\n- Tạo bảng item-level.\n- Tính Spearman frequency–norm.\n- Vẽ scatter và boxplot.\n\nMốc 2: đủ ba mô hình và có phân tích norm–frequency.\n\n### Tuần 7: Cosine và category\n\n- Lấy mẫu cặp cùng/khác category.\n- Tính cosine và khoảng tin cậy.\n- Nếu đủ thời gian, tính khoảng cách đến category centroid.\n- Phân tích warm/rare/cold một cách thận trọng.\n\n### Tuần 8: Liên hệ embedding–recommendation\n\n- Tính rec_count và exposure share theo item.\n- Tính ba tương quan cốt lõi.\n- Kiểm tra `norm–rec_count` trong từng frequency bin.\n- Tổng hợp bảng so sánh ba mô hình.\n\n### Tuần 9: t-SNE và seed thứ hai\n\n- Chạy đủ seed thứ hai.\n- Tạo t-SNE với tham số được ghi lại.\n- Kiểm tra kết luận có ổn định không.\n- Chọn hình và bảng quan trọng.\n\n### Tuần 10: Báo cáo và slide\n\n- Viết báo cáo 12–16 trang.\n- Hoàn thiện notebook.\n- Viết giới hạn nghiên cứu.\n- Chuẩn bị slide và kịch bản thuyết trình.\n\n---\n\n## 18. Phân công cho hai sinh viên\n\nTheo mô tả đề tài:\n\n### Sinh viên 1: Đầu ra và popularity bias\n\n- Tiền xử lý và huấn luyện GRU4Rec, NARM, SASRec.\n- Sinh top-20.\n- Tính Coverage@20, Gini và ARP.\n- Phân tích chỉ số theo nhóm item.\n- Lưu kết quả theo mô hình và seed.\n\n### Sinh viên 2: Embedding geometry\n\n- Trích xuất item embedding.\n- Tính norm và Spearman.\n- Phân tích cosine cùng/khác category.\n- Trực quan hóa t-SNE.\n- Đánh dấu và diễn giải cold item.\n\n### Làm chung\n\n- Chốt protocol trước khi chạy.\n- Xây dựng bảng item-level thống nhất.\n- Liên hệ output bias với embedding geometry.\n- Kiểm tra kết luận qua hai seed.\n- Viết báo cáo và slide.\n\nHai phần không được làm độc lập hoàn toàn. Cả hai thành viên phải thống nhất item mapping, split, popularity group và seed để có thể nối kết quả.\n\n---\n\n## 19. Sản phẩm cần nộp\n\nTheo mô tả đề tài:\n\n1. Notebook tính chỉ số popularity bias.\n2. Notebook phân tích embedding và tạo hình t-SNE.\n3. Báo cáo khoảng 12–16 trang.\n4. Slide thuyết trình.\n\nNên lưu thêm để tái lập:\n\n- File cấu hình từng mô hình.\n- Danh sách seed.\n- Thống kê split.\n- Danh sách cold item.\n- Bảng item-level của từng mô hình/seed.\n- File top-20 hoặc recommendation count.\n- Phiên bản thư viện và môi trường chạy.\n\n---\n\n## 20. Cấu trúc báo cáo đề xuất\n\n### 1. Giới thiệu\n\n- SBR là gì?\n- Popularity bias là gì?\n- Vì sao cần nhìn cả đầu ra và embedding?\n- Mục tiêu và câu hỏi nghiên cứu.\n\n### 2. Kiến thức liên quan\n\n- GRU4Rec, NARM, SASRec.\n- Item embedding.\n- Popularity bias.\n- Norm, cosine, Spearman và t-SNE.\n\n### 3. Dữ liệu và phương pháp\n\n- Diginetica.\n- Tiền xử lý.\n- Giao thức cold-start 20%.\n- Train/validation/test.\n- Định nghĩa popularity groups.\n- Cấu hình mô hình và seed.\n- Định nghĩa chính xác từng chỉ số.\n\n### 4. Kết quả đầu ra\n\n- Sanity-check accuracy.\n- Coverage@20.\n- Gini.\n- ARP.\n- So sánh ba mô hình và hai seed.\n\n### 5. Phân tích embedding\n\n- Norm–frequency.\n- Cosine cùng/khác category.\n- Khoảng cách centroid nếu có.\n- t-SNE.\n- Cold item.\n\n### 6. Liên hệ hai góc nhìn\n\n- Frequency–recommendation count.\n- Norm–recommendation count.\n- Phân tích trong từng frequency bin.\n- Mô hình nào có cả output bias và embedding separation mạnh nhất?\n\n### 7. Thảo luận và giới hạn\n\n- Không khẳng định nhân quả.\n- Hạn chế của t-SNE.\n- Cold embedding chưa được học.\n- Chỉ hai seed.\n- Kết quả phụ thuộc preprocessing và định nghĩa catalog.\n\n### 8. Kết luận\n\n- Trả lời trực tiếp ba câu hỏi bắt buộc.\n- Tóm tắt bằng số liệu.\n- Không tuyên bố vượt quá bằng chứng.\n\n---\n\n## 21. Các hướng cải tiến có thể suy ra nhưng nằm ngoài phạm vi\n\nMục này dùng để hiểu ý nghĩa ứng dụng của kết quả. Mô tả đề tài ghi rõ **không đề xuất hoặc triển khai phương pháp khử thiên lệch và không hiệu chỉnh embedding**. Vì vậy không đưa các hướng sau thành đóng góp chính nếu chưa được giảng viên cho phép.\n\n### Nếu popular item có norm quá lớn\n\nCác hướng nghiên cứu sau này có thể gồm:\n\n- Chuẩn hóa embedding và dùng cosine scoring.\n- Regularization norm.\n- Ràng buộc độ lớn embedding.\n\n### Nếu bias mạnh nhưng không liên quan đến norm\n\nCó thể nghiên cứu:\n\n- Frequency-aware sampling.\n- Weighted loss cho rare item.\n- Inverse-propensity weighting.\n- Điều chỉnh negative sampling.\n\n### Nếu top-20 quá tập trung\n\nCó thể dùng re-ranking với penalty độ phổ biến:\n\n\\[\nscore'(i)=score(i)-\\lambda\\log(1+frequency_i)\n\\]\n\nHoặc tối ưu đồng thời accuracy, diversity và catalog exposure.\n\n### Nếu rare item có cấu trúc category yếu\n\nCó thể nghiên cứu:\n\n- Auxiliary category loss.\n- Contrastive learning cho item cùng category.\n- Chia sẻ biểu diễn qua metadata.\n\n### Nếu cold item có embedding ngẫu nhiên\n\nMô hình chỉ dùng item ID không thể xử lý triệt để item hoàn toàn mới. Cần thêm side information:\n\n- Category.\n- Tên hoặc mô tả sản phẩm.\n- Giá, brand và thuộc tính.\n- Hình ảnh sản phẩm.\n\nKhi đó có thể dùng mô hình hybrid hoặc content-aware để sinh embedding cho item mới.\n\nCác hướng trên là bước nghiên cứu tiếp theo, không thuộc sản phẩm bắt buộc của BTL-K03.\n\n---\n\n## 22. Những lỗi dễ gặp\n\n### Lỗi 1: Gọi mọi item hiếm là cold\n\nRare item vẫn có mặt trong train; cold item không có mặt trong train. Hai nhóm không giống nhau.\n\n### Lỗi 2: Để cold item lọt vào train\n\nNếu item được chọn là cold nhưng vẫn có một số tương tác trong train, giao thức bị rò rỉ.\n\n### Lỗi 3: RecBole loại bỏ cold item khỏi vocabulary\n\nKhi đó không thể trích embedding để minh họa. Phải thiết kế data pipeline theo Option B.\n\n### Lỗi 4: So sánh ba mô hình trên candidate set khác nhau\n\nKết quả Coverage và Gini sẽ không còn công bằng.\n\n### Lỗi 5: Tính Gini nhưng bỏ các item có rec_count bằng 0\n\nViệc này có thể làm thay đổi mạnh ý nghĩa đo catalog concentration.\n\n### Lỗi 6: Kết luận chỉ dựa trên t-SNE\n\nt-SNE chỉ là hình minh họa. Kết luận phải dựa vào số đo trong không gian gốc.\n\n### Lỗi 7: Xem p-value nhỏ là hiệu ứng mạnh\n\nVới hàng chục nghìn item, một tương quan rất nhỏ vẫn có thể có p-value nhỏ. Phải nhìn cả \\(\\rho\\) và khoảng tin cậy.\n\n### Lỗi 8: Khẳng định quan hệ nhân quả\n\nTương quan giữa frequency, norm và rec_count không chứng minh norm gây ra bias.\n\n### Lỗi 9: Chỉ báo cáo bias mà không kiểm tra mô hình có hoạt động\n\nNên có Recall/MRR/NDCG làm sanity check để tránh khen một mô hình “ít bias” chỉ vì gợi ý kém.\n\n### Lỗi 10: Vượt phạm vi sang xây dựng phương pháp debias\n\nĐề tài chỉ chẩn đoán. Không triển khai debiasing trước khi hoàn thành đầy đủ ba mốc bắt buộc và có sự cho phép của giảng viên.\n\n---\n\n## 23. Checklist hoàn thành\n\n### Dữ liệu và protocol\n\n- [ ] Có dữ liệu Diginetica và ánh xạ category.\n- [ ] Đã đọc Mục 1.5, Mục 1.6 và Option B.\n- [ ] Có mô tả rõ train/validation/test.\n- [ ] Có danh sách cold item và xác nhận không rò rỉ vào train.\n- [ ] Có định nghĩa candidate catalog.\n- [ ] Có định nghĩa popular/medium/rare.\n\n### Huấn luyện\n\n- [ ] GRU4Rec chạy đủ hai seed.\n- [ ] NARM chạy đủ hai seed.\n- [ ] SASRec chạy đủ hai seed.\n- [ ] Lưu checkpoint, config và log.\n- [ ] Có sanity-check accuracy.\n\n### Popularity bias\n\n- [ ] Coverage@20 cho ba mô hình.\n- [ ] Gini cho ba mô hình.\n- [ ] Average Recommendation Popularity cho ba mô hình.\n- [ ] Mọi chỉ số dùng cùng catalog và định nghĩa.\n\n### Embedding\n\n- [ ] Ánh xạ embedding về đúng raw item ID.\n- [ ] Tính norm cho từng item.\n- [ ] Spearman norm–frequency kèm p-value hoặc khoảng tin cậy.\n- [ ] Cosine cùng category và khác category.\n- [ ] t-SNE có perplexity, seed và chú thích giới hạn.\n- [ ] Cold item được đánh dấu và diễn giải thận trọng.\n\n### Liên hệ hai góc nhìn\n\n- [ ] Có train frequency, norm và rec_count trên cùng bảng item-level.\n- [ ] Có tương quan frequency–rec_count.\n- [ ] Có tương quan norm–rec_count.\n- [ ] Nếu có thể, kiểm tra norm–rec_count trong cùng frequency bin.\n- [ ] So sánh kết luận qua hai seed.\n- [ ] Không khẳng định quan hệ nhân quả.\n\n### Sản phẩm\n\n- [ ] Notebook popularity bias.\n- [ ] Notebook embedding geometry và t-SNE.\n- [ ] Báo cáo 12–16 trang.\n- [ ] Slide.\n- [ ] Kết luận trả lời trực tiếp ba câu hỏi bắt buộc.\n\n---\n\n## 24. Phát biểu cuối cùng để ghi nhớ\n\nĐề tài không chỉ làm:\n\n```text\nTrain 3 mô hình → tính vài chỉ số → vẽ t-SNE\n```\n\nMà phải làm thành chuỗi lập luận:\n\n```text\nTần suất xuất hiện trong train\n            ↓\nĐặc điểm hình học của embedding\n            ↓\nTần suất xuất hiện trong top-20\n            ↓\nCoverage, Gini và độ phổ biến trung bình\n```\n\nMục tiêu cuối là xác định:\n\n1. Mô hình nào thiên lệch nhất ở đầu ra?\n2. Popular, rare và cold item khác nhau như thế nào trong không gian embedding?\n3. Những khác biệt embedding đó có đi kèm với số lần item được gợi ý hay không?\n4. Kết luận có ổn định qua hai seed không?\n\nĐề tài dừng ở việc cung cấp bằng chứng chẩn đoán. Các giải pháp như normalization, weighted loss, re-ranking hoặc mô hình content-aware chỉ là hướng nghiên cứu tiếp theo và không thuộc phạm vi triển khai bắt buộc.\n", 'requirements.txt': 'numpy>=1.26.0\npandas>=2.1.0\nscipy>=1.11.0\nscikit-learn>=1.3.0\nmatplotlib>=3.8.0\nseaborn>=0.13.0\npyarrow>=14.0.0\nPyYAML>=6.0.1\ntqdm>=4.66.0\ntorch>=2.1.0\njupyter>=1.0.0\nnotebook>=7.0.0\nipykernel>=6.26.0\nnbformat>=5.9.0\npytest>=7.4.0\n', 'src/btl_k03/__init__.py': '"""\nPackage btl_k03 — Hỗ trợ huấn luyện GRU4Rec, NARM, SASRec và phân tích\nPopularity Bias + Hình học không gian Embedding trên bộ dữ liệu Diginetica.\n"""\n\nfrom .paths import (\n    CONFIGS_DIR,\n    DATA_DIR,\n    FIGURES_DIR,\n    INTERIM_DIGI_DIR,\n    ITEM_LEVEL_DIR,\n    PROCESSED_DIGI_DIR,\n    PROJECT_ROOT,\n    RAW_DIGI_DIR,\n    RECBOLE_DIGI_DIR,\n    RESULTS_DIR,\n    RUNS_DIR,\n    STAT_TESTS_DIR,\n    TABLES_DIR,\n    ensure_output_dirs,\n    find_project_root,\n    get_run_dir,\n)\nfrom .popularity_metrics import (\n    compute_arp,\n    compute_catalog_coverage,\n    compute_cold_start_test_diagnostic,\n    compute_gini_coefficient,\n    compute_lorenz_curve,\n    compute_ranking_metrics_from_predictions,\n    compute_stratified_ranking_metrics,\n    evaluate_top20_popularity_bias,\n)\nfrom .utils import (\n    create_run_logger,\n    load_experiment_config,\n    load_item_mapping,\n    load_json,\n    load_yaml,\n    save_json,\n    save_yaml,\n    set_global_seed,\n)\n\n__all__ = [\n    "PROJECT_ROOT",\n    "CONFIGS_DIR",\n    "DATA_DIR",\n    "RAW_DIGI_DIR",\n    "INTERIM_DIGI_DIR",\n    "PROCESSED_DIGI_DIR",\n    "RECBOLE_DIGI_DIR",\n    "RUNS_DIR",\n    "RESULTS_DIR",\n    "ITEM_LEVEL_DIR",\n    "TABLES_DIR",\n    "FIGURES_DIR",\n    "STAT_TESTS_DIR",\n    "find_project_root",\n    "get_run_dir",\n    "ensure_output_dirs",\n    "set_global_seed",\n    "load_yaml",\n    "save_yaml",\n    "load_json",\n    "save_json",\n    "load_experiment_config",\n    "load_item_mapping",\n    "create_run_logger",\n    "compute_gini_coefficient",\n    "compute_lorenz_curve",\n    "compute_catalog_coverage",\n    "compute_arp",\n    "compute_ranking_metrics_from_predictions",\n    "compute_stratified_ranking_metrics",\n    "evaluate_top20_popularity_bias",\n    "compute_cold_start_test_diagnostic",\n]\n', 'src/btl_k03/models.py': 'from __future__ import annotations\n\nimport copy\nimport math\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\n\nfrom .embedding_analysis import build_item_level_analysis_table\nfrom .paths import ITEM_LEVEL_DIR, PROCESSED_DIGI_DIR, ensure_output_dirs, get_run_dir\nfrom .popularity_metrics import (\n    compute_cold_start_test_diagnostic,\n    compute_ranking_metrics_from_predictions,\n    compute_stratified_ranking_metrics,\n    evaluate_top20_popularity_bias,\n)\nfrom .utils import (\n    create_run_logger,\n    load_experiment_config,\n    load_item_mapping,\n    save_json,\n    save_yaml,\n    set_global_seed,\n)\n\n\nclass SessionPrefixDataset(Dataset):\n    """\n    Dataset tiền xử lý sẵn chuỗi prefix sang ma trận numpy cố định (N, max_seq_length)\n    để tốc độ nạp batch vào GPU/CPU nhanh gấp 10 lần so với duyệt list Python ở mỗi batch.\n    """\n\n    def __init__(self, examples_df: pd.DataFrame, max_seq_length: int = 50):\n        super().__init__()\n        self.max_seq_length = int(max_seq_length)\n        n = len(examples_df)\n\n        self.session_ids = examples_df["session_id"].to_numpy(dtype=np.int64)\n        self.targets = examples_df["target_internal_item_id"].to_numpy(dtype=np.int64)\n        self.target_raw_ids = examples_df["target_item_id"].to_numpy(dtype=np.int64)\n        self.target_groups = examples_df["target_popularity_group"].astype(str).to_numpy()\n        self.query_ids = np.arange(n, dtype=np.int64)\n\n        sequences = np.zeros((n, self.max_seq_length), dtype=np.int64)\n        lengths = np.empty(n, dtype=np.int64)\n\n        for idx, seq in enumerate(examples_df["prefix_items"]):\n            arr = np.asarray(seq, dtype=np.int64)\n            if len(arr) > self.max_seq_length:\n                arr = arr[-self.max_seq_length :]\n            length = max(1, len(arr))\n            sequences[idx, :length] = arr[:length]\n            lengths[idx] = length\n\n        self.sequences = torch.from_numpy(sequences)\n        self.lengths = torch.from_numpy(lengths)\n        self.targets_tensor = torch.from_numpy(self.targets)\n        self.query_ids_tensor = torch.from_numpy(self.query_ids)\n\n    def __len__(self) -> int:\n        return len(self.targets)\n\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:\n        return {\n            "query_id": self.query_ids_tensor[idx],\n            "item_seq": self.sequences[idx],\n            "item_seq_len": self.lengths[idx],\n            "target": self.targets_tensor[idx],\n        }\n\n\nclass BaseSBRModel(nn.Module):\n    """\n    Lớp cơ sở cho cả 3 mô hình SBR (GRU4Rec, NARM, SASRec).\n    Đảm bảo:\n    1. Bảng item_embedding có đúng kích thước (num_items + 1, embedding_size) với padding_idx=0.\n    2. Cố định và đóng băng tuyệt đối gradient của padding (ID 0) và 7,647 Controlled Cold Items.\n    3. Hỗ trợ chấm điểm full-catalog kèm warm_mask để chỉ gợi ý trong 29,936 warm items.\n    """\n\n    def __init__(\n        self,\n        num_items_with_pad: int,\n        embedding_size: int,\n        trainable_warm_mask: torch.Tensor,\n        padding_idx: int = 0,\n    ):\n        super().__init__()\n        self.num_items_with_pad = int(num_items_with_pad)\n        self.embedding_size = int(embedding_size)\n        self.padding_idx = int(padding_idx)\n\n        self.item_embedding = nn.Embedding(\n            self.num_items_with_pad,\n            self.embedding_size,\n            padding_idx=self.padding_idx,\n        )\n\n        # Đăng ký buffer mask boolean cho các warm item hợp lệ (ID 1..N có is_trainable_warm == True)\n        mask_bool = trainable_warm_mask.to(dtype=torch.bool).clone()\n        mask_bool[self.padding_idx] = False\n        self.register_buffer("trainable_warm_mask", mask_bool)\n\n        # Hook khóa cứng gradient = 0 đối với hàng padding (0) và toàn bộ 7,647 hàng Controlled Cold\n        self._register_cold_gradient_freeze_hook()\n\n    def _register_cold_gradient_freeze_hook(self) -> None:\n        def _hook(grad: torch.Tensor) -> torch.Tensor:\n            grad_masked = grad.clone()\n            grad_masked[~self.trainable_warm_mask] = 0.0\n            return grad_masked\n\n        self.item_embedding.weight.register_hook(_hook)\n\n    def gather_indexes(self, output: torch.Tensor, gather_index: torch.Tensor) -> torch.Tensor:\n        """Trích xuất vector ẩn tại vị trí kết thúc thực tế (seq_len - 1) của mỗi phiên."""\n        gather_index = gather_index.view(-1, 1, 1).expand(-1, -1, output.shape[-1])\n        output_tensor = output.gather(dim=1, index=gather_index)\n        return output_tensor.squeeze(1)\n\n    def encode_session(\n        self, item_seq: torch.Tensor, item_seq_len: torch.Tensor\n    ) -> torch.Tensor:\n        raise NotImplementedError\n\n    def full_sort_logits(\n        self,\n        item_seq: torch.Tensor,\n        item_seq_len: torch.Tensor,\n        mask_cold_and_pad: bool = True,\n        mask_fill_value: float = -1.0e9,\n    ) -> torch.Tensor:\n        """Tính điểm tích vô hướng h_s @ E^T cho toàn bộ catalog và che điểm của cold/padding."""\n        seq_output = self.encode_session(item_seq, item_seq_len)\n        logits = torch.matmul(seq_output, self.item_embedding.weight.transpose(0, 1))\n        if mask_cold_and_pad:\n            fill_val = mask_fill_value\n            if logits.dtype == torch.float16 and fill_val < -60000.0:\n                fill_val = -10000.0\n            logits = logits.masked_fill(~self.trainable_warm_mask.unsqueeze(0), fill_val)\n        return logits\n\n\nclass GRU4RecModel(BaseSBRModel):\n    """\n    GRU4Rec (Hidasi et al., ICLR 2016) — Kiến trúc chuẩn RecBole:\n    Item Embedding -> Dropout -> GRU -> Dense Projection -> Dot Product với Item Embedding.\n    """\n\n    def __init__(\n        self,\n        num_items_with_pad: int,\n        trainable_warm_mask: torch.Tensor,\n        embedding_size: int = 64,\n        hidden_size: int = 128,\n        num_layers: int = 1,\n        embedding_dropout: float = 0.25,\n        hidden_dropout: float = 0.20,\n        init_std: float = 0.02,\n        padding_idx: int = 0,\n    ):\n        super().__init__(\n            num_items_with_pad=num_items_with_pad,\n            embedding_size=embedding_size,\n            trainable_warm_mask=trainable_warm_mask,\n            padding_idx=padding_idx,\n        )\n        self.hidden_size = int(hidden_size)\n        self.num_layers = int(num_layers)\n        self.init_std = float(init_std)\n\n        self.emb_dropout = nn.Dropout(embedding_dropout)\n        self.gru_layers = nn.GRU(\n            input_size=self.embedding_size,\n            hidden_size=self.hidden_size,\n            num_layers=self.num_layers,\n            bias=False,\n            batch_first=True,\n            dropout=hidden_dropout if self.num_layers > 1 else 0.0,\n        )\n        self.out_dropout = nn.Dropout(hidden_dropout)\n        self.dense = nn.Linear(self.hidden_size, self.embedding_size)\n\n        self._init_weights()\n\n    def _init_weights(self) -> None:\n        nn.init.normal_(self.item_embedding.weight, mean=0.0, std=self.init_std)\n        with torch.no_grad():\n            self.item_embedding.weight[self.padding_idx].zero_()\n        for name, param in self.gru_layers.named_parameters():\n            if "weight_ih" in name:\n                nn.init.xavier_uniform_(param.data)\n            elif "weight_hh" in name:\n                nn.init.orthogonal_(param.data)\n        nn.init.xavier_normal_(self.dense.weight)\n        if self.dense.bias is not None:\n            nn.init.constant_(self.dense.bias, 0.0)\n\n    def encode_session(\n        self, item_seq: torch.Tensor, item_seq_len: torch.Tensor\n    ) -> torch.Tensor:\n        item_seq_emb = self.item_embedding(item_seq)\n        item_seq_emb_dropout = self.emb_dropout(item_seq_emb)\n        gru_output, _ = self.gru_layers(item_seq_emb_dropout)\n        last_hidden = self.gather_indexes(gru_output, item_seq_len - 1)\n        last_hidden = self.out_dropout(last_hidden)\n        seq_output = self.dense(last_hidden)\n        return seq_output\n\n\nclass NARMModel(BaseSBRModel):\n    """\n    NARM (Li et al., CIKM 2017) — Kiến trúc chuẩn RecBole:\n    Kết hợp Global Encoder (trạng thái cuối GRU h_t) và Local Encoder (Attention trên toàn chuỗi\n    để nắm bắt ý định chính của người dùng trong phiên) -> Ghép [c_local; c_global] -> Bi-linear Dense.\n    """\n\n    def __init__(\n        self,\n        num_items_with_pad: int,\n        trainable_warm_mask: torch.Tensor,\n        embedding_size: int = 64,\n        hidden_size: int = 128,\n        n_layers: int = 1,\n        embedding_dropout: float = 0.25,\n        ct_dropout: float = 0.50,\n        init_std: float = 0.02,\n        padding_idx: int = 0,\n    ):\n        super().__init__(\n            num_items_with_pad=num_items_with_pad,\n            embedding_size=embedding_size,\n            trainable_warm_mask=trainable_warm_mask,\n            padding_idx=padding_idx,\n        )\n        self.hidden_size = int(hidden_size)\n        self.n_layers = int(n_layers)\n        self.init_std = float(init_std)\n\n        self.emb_dropout = nn.Dropout(embedding_dropout)\n        self.gru = nn.GRU(\n            input_size=self.embedding_size,\n            hidden_size=self.hidden_size,\n            num_layers=self.n_layers,\n            bias=False,\n            batch_first=True,\n        )\n        self.a_1 = nn.Linear(self.hidden_size, self.hidden_size, bias=False)\n        self.a_2 = nn.Linear(self.hidden_size, self.hidden_size, bias=False)\n        self.v_t = nn.Linear(self.hidden_size, 1, bias=False)\n        self.ct_dropout = nn.Dropout(ct_dropout)\n        self.b = nn.Linear(2 * self.hidden_size, self.embedding_size, bias=False)\n\n        self._init_weights()\n\n    def _init_weights(self) -> None:\n        nn.init.normal_(self.item_embedding.weight, mean=0.0, std=self.init_std)\n        with torch.no_grad():\n            self.item_embedding.weight[self.padding_idx].zero_()\n        for name, param in self.gru.named_parameters():\n            if "weight_ih" in name:\n                nn.init.xavier_uniform_(param.data)\n            elif "weight_hh" in name:\n                nn.init.orthogonal_(param.data)\n        for linear in (self.a_1, self.a_2, self.v_t, self.b):\n            nn.init.xavier_normal_(linear.weight)\n\n    def encode_session(\n        self, item_seq: torch.Tensor, item_seq_len: torch.Tensor\n    ) -> torch.Tensor:\n        item_seq_emb = self.item_embedding(item_seq)\n        item_seq_emb_dropout = self.emb_dropout(item_seq_emb)\n        gru_out, _ = self.gru(item_seq_emb_dropout)\n\n        # Global representation: trạng thái ẩn ở bước cuối của chuỗi\n        c_global = h_t = self.gather_indexes(gru_out, item_seq_len - 1)\n\n        # Local representation: cơ chế Attention tính trọng số của từng item trong phiên với h_t\n        q1 = self.a_1(gru_out)\n        q2 = self.a_2(h_t).unsqueeze(1)\n        alpha = self.v_t(torch.sigmoid(q1 + q2))  # (B, L, 1)\n\n        mask = item_seq.gt(0).unsqueeze(2)  # (B, L, 1)\n        alpha = alpha.masked_fill(~mask, -1.0e9)\n        attn_weights = torch.softmax(alpha, dim=1)\n        c_local = torch.sum(attn_weights * gru_out, dim=1)\n\n        c_t = torch.cat([c_local, c_global], dim=1)\n        c_t = self.ct_dropout(c_t)\n        seq_output = self.b(c_t)\n        return seq_output\n\n\nclass SASRecTransformerBlock(nn.Module):\n    """Một tầng Causal Self-Attention + Point-wise Feed-Forward chuẩn SASRec / RecBole."""\n\n    def __init__(\n        self,\n        hidden_size: int,\n        n_heads: int,\n        inner_size: int,\n        hidden_dropout_prob: float,\n        attn_dropout_prob: float,\n        layer_norm_eps: float = 1e-12,\n        hidden_act: str = "gelu",\n    ):\n        super().__init__()\n        self.num_attention_heads = int(n_heads)\n        self.attention_head_size = int(hidden_size // n_heads)\n        self.all_head_size = self.num_attention_heads * self.attention_head_size\n\n        self.query = nn.Linear(hidden_size, self.all_head_size)\n        self.key = nn.Linear(hidden_size, self.all_head_size)\n        self.value = nn.Linear(hidden_size, self.all_head_size)\n\n        self.attn_dropout = nn.Dropout(attn_dropout_prob)\n        self.out_proj = nn.Linear(hidden_size, hidden_size)\n        self.out_dropout = nn.Dropout(hidden_dropout_prob)\n        self.layer_norm_1 = nn.LayerNorm(hidden_size, eps=layer_norm_eps)\n\n        self.ffn_1 = nn.Linear(hidden_size, inner_size)\n        self.ffn_2 = nn.Linear(inner_size, hidden_size)\n        self.ffn_dropout = nn.Dropout(hidden_dropout_prob)\n        self.layer_norm_2 = nn.LayerNorm(hidden_size, eps=layer_norm_eps)\n        self.act_fn = F.gelu if hidden_act.lower() == "gelu" else F.relu\n\n    def _transpose_for_scores(self, x: torch.Tensor) -> torch.Tensor:\n        new_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)\n        x = x.view(*new_shape)\n        return x.permute(0, 2, 1, 3)\n\n    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        q = self._transpose_for_scores(self.query(hidden_states))\n        k = self._transpose_for_scores(self.key(hidden_states))\n        v = self._transpose_for_scores(self.value(hidden_states))\n\n        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(\n            self.attention_head_size\n        )\n        attn_scores = attn_scores + attention_mask\n        attn_probs = nn.Softmax(dim=-1)(attn_scores)\n        attn_probs = self.attn_dropout(attn_probs)\n\n        context = torch.matmul(attn_probs, v)\n        context = context.permute(0, 2, 1, 3).contiguous()\n        context = context.view(context.size()[:-2] + (self.all_head_size,))\n\n        attn_out = self.out_proj(context)\n        attn_out = self.out_dropout(attn_out)\n        hidden_states = self.layer_norm_1(hidden_states + attn_out)\n\n        ffn_out = self.ffn_2(self.act_fn(self.ffn_1(hidden_states)))\n        ffn_out = self.ffn_dropout(ffn_out)\n        hidden_states = self.layer_norm_2(hidden_states + ffn_out)\n        return hidden_states\n\n\nclass SASRecModel(BaseSBRModel):\n    """\n    SASRec (Kang & McAuley, ICDM 2018) — Kiến trúc chuẩn RecBole:\n    Item Embedding + Positional Embedding -> LayerNorm + Dropout -> Causal Multi-Head Transformer -> Gather Last Position.\n    """\n\n    def __init__(\n        self,\n        num_items_with_pad: int,\n        trainable_warm_mask: torch.Tensor,\n        max_seq_length: int = 50,\n        embedding_size: int = 64,\n        n_layers: int = 2,\n        n_heads: int = 2,\n        inner_size: int = 256,\n        hidden_dropout_prob: float = 0.20,\n        attn_dropout_prob: float = 0.20,\n        hidden_act: str = "gelu",\n        layer_norm_eps: float = 1.0e-12,\n        initializer_range: float = 0.02,\n        padding_idx: int = 0,\n    ):\n        super().__init__(\n            num_items_with_pad=num_items_with_pad,\n            embedding_size=embedding_size,\n            trainable_warm_mask=trainable_warm_mask,\n            padding_idx=padding_idx,\n        )\n        self.max_seq_length = int(max_seq_length)\n        self.initializer_range = float(initializer_range)\n\n        self.position_embedding = nn.Embedding(self.max_seq_length, self.embedding_size)\n        self.layer_norm = nn.LayerNorm(self.embedding_size, eps=layer_norm_eps)\n        self.dropout = nn.Dropout(hidden_dropout_prob)\n\n        self.blocks = nn.ModuleList(\n            [\n                SASRecTransformerBlock(\n                    hidden_size=self.embedding_size,\n                    n_heads=n_heads,\n                    inner_size=inner_size,\n                    hidden_dropout_prob=hidden_dropout_prob,\n                    attn_dropout_prob=attn_dropout_prob,\n                    layer_norm_eps=layer_norm_eps,\n                    hidden_act=hidden_act,\n                )\n                for _ in range(int(n_layers))\n            ]\n        )\n\n        self._init_weights()\n\n    def _init_weights(self) -> None:\n        for module in self.modules():\n            if isinstance(module, (nn.Linear, nn.Embedding)):\n                module.weight.data.normal_(mean=0.0, std=self.initializer_range)\n            elif isinstance(module, nn.LayerNorm):\n                module.bias.data.zero_()\n                module.weight.data.fill_(1.0)\n            if isinstance(module, nn.Linear) and module.bias is not None:\n                module.bias.data.zero_()\n        with torch.no_grad():\n            self.item_embedding.weight[self.padding_idx].zero_()\n\n    def _get_causal_attention_mask(self, item_seq: torch.Tensor) -> torch.Tensor:\n        """Tạo mặt nạ Causal (tam giác dưới) kết hợp mặt nạ Padding cho Self-Attention."""\n        attention_mask = (item_seq > 0).long()  # (B, L)\n        extended_mask = attention_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, L)\n        max_len = attention_mask.size(-1)\n        subsequent_mask = torch.triu(\n            torch.ones((1, max_len, max_len), device=item_seq.device), diagonal=1\n        ).eq(0)\n        subsequent_mask = subsequent_mask.unsqueeze(1).long()  # (1, 1, L, L)\n\n        combined_mask = extended_mask * subsequent_mask\n        combined_mask = (1.0 - combined_mask.to(dtype=torch.float32)) * -10000.0\n        return combined_mask\n\n    def encode_session(\n        self, item_seq: torch.Tensor, item_seq_len: torch.Tensor\n    ) -> torch.Tensor:\n        seq_len = item_seq.size(1)\n        position_ids = (\n            torch.arange(seq_len, dtype=torch.long, device=item_seq.device)\n            .unsqueeze(0)\n            .expand_as(item_seq)\n        )\n        hidden_states = self.item_embedding(item_seq) + self.position_embedding(position_ids)\n        hidden_states = self.layer_norm(hidden_states)\n        hidden_states = self.dropout(hidden_states)\n\n        attention_mask = self._get_causal_attention_mask(item_seq)\n        for block in self.blocks:\n            hidden_states = block(hidden_states, attention_mask)\n\n        seq_output = self.gather_indexes(hidden_states, item_seq_len - 1)\n        return seq_output\n\n\ndef build_model_from_config(\n    config: dict[str, Any],\n    item_mapping: pd.DataFrame,\n    device: torch.device,\n) -> BaseSBRModel:\n    """Khởi tạo đúng mô hình (GRU4Rec, NARM hoặc SASRec) cùng mask warm/cold chuẩn từ item_mapping."""\n    num_items_with_pad = int(item_mapping["internal_item_id"].max()) + 1\n    warm_mask = torch.zeros(num_items_with_pad, dtype=torch.bool)\n    warm_ids = item_mapping.loc[\n        item_mapping["is_trainable_warm"], "internal_item_id"\n    ].to_numpy(dtype=np.int64)\n    warm_mask[warm_ids] = True\n\n    model_name = config["model_name"].lower()\n    params = config["model_params"]\n    padding_idx = int(config.get("padding_idx", 0))\n    max_seq_len = int(config.get("max_seq_length", 50))\n\n    if model_name == "gru4rec":\n        model = GRU4RecModel(\n            num_items_with_pad=num_items_with_pad,\n            trainable_warm_mask=warm_mask,\n            embedding_size=int(params.get("embedding_size", 64)),\n            hidden_size=int(params.get("hidden_size", 128)),\n            num_layers=int(params.get("num_layers", 1)),\n            embedding_dropout=float(params.get("embedding_dropout", 0.25)),\n            hidden_dropout=float(params.get("hidden_dropout", 0.20)),\n            init_std=float(params.get("init_std", 0.02)),\n            padding_idx=padding_idx,\n        )\n    elif model_name == "narm":\n        model = NARMModel(\n            num_items_with_pad=num_items_with_pad,\n            trainable_warm_mask=warm_mask,\n            embedding_size=int(params.get("embedding_size", 64)),\n            hidden_size=int(params.get("hidden_size", 128)),\n            n_layers=int(params.get("n_layers", 1)),\n            embedding_dropout=float(params.get("embedding_dropout", 0.25)),\n            ct_dropout=float(params.get("ct_dropout", 0.50)),\n            init_std=float(params.get("init_std", 0.02)),\n            padding_idx=padding_idx,\n        )\n    elif model_name == "sasrec":\n        model = SASRecModel(\n            num_items_with_pad=num_items_with_pad,\n            trainable_warm_mask=warm_mask,\n            max_seq_length=max_seq_len,\n            embedding_size=int(params.get("embedding_size", 64)),\n            n_layers=int(params.get("n_layers", 2)),\n            n_heads=int(params.get("n_heads", 2)),\n            inner_size=int(params.get("inner_size", 256)),\n            hidden_dropout_prob=float(params.get("hidden_dropout_prob", 0.20)),\n            attn_dropout_prob=float(params.get("attn_dropout_prob", 0.20)),\n            hidden_act=str(params.get("hidden_act", "gelu")),\n            layer_norm_eps=float(params.get("layer_norm_eps", 1e-12)),\n            initializer_range=float(params.get("initializer_range", 0.02)),\n            padding_idx=padding_idx,\n        )\n    else:\n        raise ValueError(f"Mô hình không hợp lệ: {model_name}")\n\n    return model.to(device)\n\n\n@torch.no_grad()\ndef predict_topk_for_dataset(\n    model: BaseSBRModel,\n    dataset: SessionPrefixDataset,\n    device: torch.device,\n    batch_size: int = 1024,\n    topk: int = 20,\n) -> tuple[np.ndarray, np.ndarray]:\n    """\n    Sinh dự đoán Top-K (internal_item_id và score) trên toàn bộ dataset.\n    Chỉ cho phép chọn ứng viên trong tập 29,936 warm items (mask cold items & padding = -1e9).\n    """\n    model.eval()\n    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)\n\n    all_topk_ids = np.empty((len(dataset), topk), dtype=np.int64)\n    all_topk_scores = np.empty((len(dataset), topk), dtype=np.float32)\n\n    offset = 0\n    for batch in loader:\n        item_seq = batch["item_seq"].to(device, non_blocking=True)\n        item_seq_len = batch["item_seq_len"].to(device, non_blocking=True)\n\n        logits = model.full_sort_logits(\n            item_seq=item_seq,\n            item_seq_len=item_seq_len,\n            mask_cold_and_pad=True,\n        )\n        scores, top_indices = torch.topk(logits, k=topk, dim=-1)\n\n        bsz = item_seq.size(0)\n        all_topk_ids[offset : offset + bsz] = top_indices.cpu().numpy()\n        all_topk_scores[offset : offset + bsz] = scores.cpu().numpy()\n        offset += bsz\n\n    return all_topk_ids, all_topk_scores\n\n\ndef build_top20_dataframe(\n    dataset: SessionPrefixDataset,\n    topk_internal_ids: np.ndarray,\n    topk_scores: np.ndarray,\n    internal_to_raw_id: dict[int, int] | np.ndarray,\n) -> pd.DataFrame:\n    """\n    Xuất bảng top20.parquet theo đúng quy ước tại Mục 9.6 tài liệu CAU-TRUC-DU-AN.md:\n    Các cột: session_id, query_id, rank (1..20), item_id (raw), internal_item_id, score.\n    """\n    n_queries, k = topk_internal_ids.shape\n    session_col = np.repeat(dataset.session_ids, k)\n    query_col = np.repeat(dataset.query_ids, k)\n    rank_col = np.tile(np.arange(1, k + 1, dtype=np.int32), n_queries)\n    internal_col = topk_internal_ids.ravel()\n    score_col = topk_scores.ravel()\n\n    if isinstance(internal_to_raw_id, np.ndarray):\n        raw_col = internal_to_raw_id[internal_col]\n    else:\n        raw_col = np.array([internal_to_raw_id[int(i)] for i in internal_col], dtype=np.int64)\n\n    return pd.DataFrame(\n        {\n            "session_id": session_col,\n            "query_id": query_col,\n            "rank": rank_col,\n            "item_id": raw_col,\n            "internal_item_id": internal_col,\n            "score": score_col,\n        }\n    )\n\n\ndef train_and_evaluate_seed(\n    model_name: str,\n    seed: int,\n    config: dict[str, Any] | None = None,\n    processed_dir: Path = PROCESSED_DIGI_DIR,\n    max_epochs_override: int | None = None,\n    device_str: str | None = None,\n) -> dict[str, Any]:\n    """\n    Thực thi trọn vẹn chu trình Huấn luyện -> Early Stopping trên Validation -> Đánh giá trên Test ->\n    Sinh top20.parquet -> Trích xuất item_embeddings.npy -> Kiểm định Cold-start Option B ->\n    Xuất bảng phân tích item-level cho một cặp (model_name, seed).\n    """\n    ensure_output_dirs()\n    cfg = copy.deepcopy(config) if config is not None else load_experiment_config(model_name)\n    cfg["seed"] = int(seed)\n    if max_epochs_override is not None:\n        cfg["training"]["epochs"] = int(max_epochs_override)\n\n    set_global_seed(seed)\n\n    if device_str is not None:\n        device = torch.device(device_str)\n    else:\n        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    run_dir = get_run_dir(model_name, seed)\n    logger = create_run_logger(run_dir / "training.log", f"{model_name}_seed_{seed}")\n\n    logger.info("=" * 72)\n    logger.info(f"BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH {model_name.upper()} | SEED = {seed}")\n    logger.info(f"Thiết bị chạy (Device) : {device}")\n    logger.info(f"Thư mục lưu kết quả    : {run_dir}")\n    logger.info("=" * 72)\n\n    # 1. Đọc item_mapping và dữ liệu train/val/test\n    item_mapping = load_item_mapping(processed_dir)\n    max_internal_id = int(item_mapping["internal_item_id"].max())\n    internal_to_raw = np.zeros(max_internal_id + 1, dtype=np.int64)\n    internal_to_raw[item_mapping["internal_item_id"].to_numpy(dtype=np.int64)] = (\n        item_mapping["item_id"].to_numpy(dtype=np.int64)\n    )\n\n    max_seq_len = int(cfg.get("max_seq_length", 50))\n    topk = int(cfg["evaluation"].get("topk", 20))\n\n    train_df = pd.read_parquet(processed_dir / "train_examples.parquet")\n    val_df = pd.read_parquet(processed_dir / "validation_examples.parquet")\n    test_df = pd.read_parquet(processed_dir / "test_examples.parquet")\n    test_last_df = pd.read_parquet(processed_dir / "test_last_item_examples.parquet")\n    test_events_full = pd.read_parquet(processed_dir / "test_events_full.parquet")\n\n    train_dataset = SessionPrefixDataset(train_df, max_seq_length=max_seq_len)\n    val_dataset = SessionPrefixDataset(val_df, max_seq_length=max_seq_len)\n    test_dataset = SessionPrefixDataset(test_df, max_seq_length=max_seq_len)\n    test_last_dataset = SessionPrefixDataset(test_last_df, max_seq_length=max_seq_len)\n\n    train_batch_size = int(cfg["training"].get("train_batch_size", 512))\n    eval_batch_size = int(cfg["training"].get("eval_batch_size", 1024))\n\n    g = torch.Generator()\n    g.manual_seed(int(seed))\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=train_batch_size,\n        shuffle=True,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n        generator=g,\n    )\n\n    # 2. Khởi tạo mô hình và lưu lại bản chụp embedding khởi tạo của Controlled Cold Items\n    model = build_model_from_config(cfg, item_mapping, device)\n    cold_internal_ids = item_mapping.loc[\n        item_mapping["is_controlled_cold"], "internal_item_id"\n    ].to_numpy(dtype=np.int64)\n\n    initial_cold_embeddings = (\n        model.item_embedding.weight.detach().cpu().numpy()[cold_internal_ids].copy()\n    )\n\n    total_params = sum(p.numel() for p in model.parameters())\n    logger.info(\n        f"Tổng số item trong bảng Embedding: {max_internal_id:,} (+1 hàng padding 0) | "\n        f"Warm trainable: {int(item_mapping[\'is_trainable_warm\'].sum()):,} | "\n        f"Controlled cold (đóng băng): {len(cold_internal_ids):,}"\n    )\n    logger.info(f"Tổng số tham số mô hình: {total_params:,}")\n\n    lr = float(cfg["training"].get("learning_rate", 0.001))\n    weight_decay = float(cfg["training"].get("weight_decay", 0.0))\n    grad_clip = float(cfg["training"].get("grad_clip_norm", 5.0))\n    epochs = int(cfg["training"].get("epochs", 30))\n    use_amp = bool(cfg["training"].get("use_amp", True)) and (device.type == "cuda")\n\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n\n    monitor_metric = str(cfg["early_stopping"].get("monitor", "mrr@20")).lower()\n    patience = int(cfg["early_stopping"].get("patience", 5))\n    min_delta = float(cfg["early_stopping"].get("min_delta", 0.0001))\n\n    best_score = -float("inf")\n    best_epoch = 0\n    epochs_no_improve = 0\n    history_rows: list[dict[str, Any]] = []\n    checkpoint_path = run_dir / "checkpoint.pth"\n\n    # 3. Vòng lặp huấn luyện qua các Epoch\n    for epoch in range(1, epochs + 1):\n        t0 = time.perf_counter()\n        model.train()\n        running_loss = 0.0\n        total_samples = 0\n\n        progress = tqdm(\n            train_loader,\n            desc=f"[{model_name.upper()} seed={seed}] Epoch {epoch:02d}/{epochs:02d}",\n            leave=False,\n        )\n        for batch in progress:\n            item_seq = batch["item_seq"].to(device, non_blocking=True)\n            item_seq_len = batch["item_seq_len"].to(device, non_blocking=True)\n            targets = batch["target"].to(device, non_blocking=True)\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.amp.autocast("cuda", enabled=use_amp):\n                logits = model.full_sort_logits(\n                    item_seq=item_seq,\n                    item_seq_len=item_seq_len,\n                    mask_cold_and_pad=True,\n                )\n                loss = F.cross_entropy(logits, targets)\n\n            scaler.scale(loss).backward()\n            if grad_clip > 0:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)\n            scaler.step(optimizer)\n            scaler.update()\n\n            bsz = item_seq.size(0)\n            running_loss += float(loss.item()) * bsz\n            total_samples += bsz\n            progress.set_postfix(loss=f"{running_loss / max(1, total_samples):.4f}")\n\n        train_loss = running_loss / max(1, total_samples)\n\n        # Đánh giá trên tập Validation Warm\n        val_topk_ids, _ = predict_topk_for_dataset(\n            model=model,\n            dataset=val_dataset,\n            device=device,\n            batch_size=eval_batch_size,\n            topk=topk,\n        )\n        val_metrics = compute_ranking_metrics_from_predictions(\n            val_topk_ids, val_dataset.targets, k=topk\n        )\n        elapsed = time.perf_counter() - t0\n\n        current_score = float(val_metrics[monitor_metric])\n        is_best = current_score > (best_score + min_delta)\n\n        if is_best:\n            best_score = current_score\n            best_epoch = epoch\n            epochs_no_improve = 0\n            torch.save(\n                {\n                    "model_name": model_name.lower(),\n                    "seed": int(seed),\n                    "epoch": int(epoch),\n                    "best_validation_score": float(best_score),\n                    "monitor_metric": monitor_metric,\n                    "model_state_dict": model.state_dict(),\n                    "config": cfg,\n                },\n                checkpoint_path,\n            )\n        else:\n            epochs_no_improve += 1\n\n        row = {\n            "epoch": int(epoch),\n            "train_loss": float(train_loss),\n            f"validation_recall@{topk}": float(val_metrics[f"recall@{topk}"]),\n            f"validation_mrr@{topk}": float(val_metrics[f"mrr@{topk}"]),\n            f"validation_ndcg@{topk}": float(val_metrics[f"ndcg@{topk}"]),\n            "validation_score": current_score,\n            "is_best": bool(is_best),\n            "elapsed_time": float(elapsed),\n        }\n        history_rows.append(row)\n\n        logger.info(\n            f"Epoch {epoch:02d}/{epochs:02d} | Loss: {train_loss:.4f} | "\n            f"Val Recall@{topk}: {val_metrics[f\'recall@{topk}\']:.4f} | "\n            f"Val MRR@{topk}: {val_metrics[f\'mrr@{topk}\']:.4f} | "\n            f"Val NDCG@{topk}: {val_metrics[f\'ndcg@{topk}\']:.4f} | "\n            f"Time: {elapsed:.1f}s {\'[BEST]\' if is_best else \'\'}"\n        )\n\n        if epochs_no_improve >= patience:\n            logger.info(\n                f"Early stopping kích hoạt tại epoch {epoch} "\n                f"(không cải thiện {monitor_metric} sau {patience} epochs). "\n                f"Best epoch = {best_epoch} ({best_score:.4f})."\n            )\n            break\n\n    # 4. Nạp lại Best Checkpoint và kiểm tra tính bất biến của Controlled Cold Embeddings\n    best_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)\n    model.load_state_dict(best_ckpt["model_state_dict"])\n    model.eval()\n\n    # Trích xuất ma trận item embedding (37583, D) từ index 1..37583 (bỏ hàng 0 padding)\n    full_weight_np = model.item_embedding.weight.detach().cpu().numpy()\n    trained_cold_embeddings = full_weight_np[cold_internal_ids]\n    max_cold_drift = float(np.max(np.abs(trained_cold_embeddings - initial_cold_embeddings)))\n    if max_cold_drift > 1e-7:\n        raise AssertionError(\n            f"Rò rỉ gradient vào Controlled Cold Items! max_abs_drift = {max_cold_drift}"\n        )\n    logger.info(\n        f"Kiểm định Option B: Độ lệch tối đa của 7,647 Controlled Cold Embeddings = {max_cold_drift:.8f} (Chưa huấn luyện 100%)."\n    )\n\n    item_embeddings = full_weight_np[1 : max_internal_id + 1].astype(np.float32)\n\n    # 5. Đánh giá trên tập Test chính (All-prefix Warm) & xuất top20.parquet\n    test_topk_ids, test_topk_scores = predict_topk_for_dataset(\n        model=model,\n        dataset=test_dataset,\n        device=device,\n        batch_size=eval_batch_size,\n        topk=topk,\n    )\n    test_metrics_main = compute_ranking_metrics_from_predictions(\n        test_topk_ids, test_dataset.targets, k=topk\n    )\n    test_stratified_df = compute_stratified_ranking_metrics(\n        test_topk_ids,\n        test_dataset.targets,\n        test_dataset.target_groups,\n        k=topk,\n    )\n\n    # Đánh giá bổ sung trên tập Last-item Test (12,791 phiên)\n    test_last_topk_ids, _ = predict_topk_for_dataset(\n        model=model,\n        dataset=test_last_dataset,\n        device=device,\n        batch_size=eval_batch_size,\n        topk=topk,\n    )\n    test_metrics_last_item = compute_ranking_metrics_from_predictions(\n        test_last_topk_ids, test_last_dataset.targets, k=topk\n    )\n\n    # Lưu top20.parquet\n    top20_df = build_top20_dataframe(\n        dataset=test_dataset,\n        topk_internal_ids=test_topk_ids,\n        topk_scores=test_topk_scores,\n        internal_to_raw_id=internal_to_raw,\n    )\n    top20_df.to_parquet(run_dir / "top20.parquet", index=False, compression="zstd")\n\n    # 6. Tính chỉ số Popularity Bias đầu ra & Cold-start Diagnostic\n    bias_summary, item_rec_catalog, group_exposure_df = evaluate_top20_popularity_bias(\n        top20_df=top20_df,\n        item_mapping=item_mapping,\n        k=topk,\n    )\n    cold_diagnostic = compute_cold_start_test_diagnostic(\n        warm_metrics=test_metrics_main,\n        test_events_full=test_events_full,\n        warm_test_examples_count=len(test_dataset),\n        k=topk,\n    )\n\n    # 7. Lưu toàn bộ 8 file chuẩn vào runs/{model}/seed_{seed}/ và bảng item-level\n    save_yaml(cfg, run_dir / "config.yaml")\n\n    history_df = pd.DataFrame(history_rows)\n    history_df.to_csv(run_dir / "training_history.csv", index=False)\n\n    np.save(run_dir / "item_embeddings.npy", item_embeddings)\n\n    embedding_mapping_df = item_mapping.copy()\n    embedding_mapping_df.insert(0, "embedding_row_index", np.arange(len(embedding_mapping_df)))\n    embedding_mapping_df.to_csv(run_dir / "item_embedding_mapping.csv", index=False)\n\n    item_level_df = build_item_level_analysis_table(\n        model_name=model_name,\n        seed=seed,\n        embeddings=item_embeddings,\n        item_rec_catalog=item_rec_catalog,\n    )\n    item_level_path = ITEM_LEVEL_DIR / f"{model_name.lower()}_seed{seed}.parquet"\n    item_level_df.to_parquet(item_level_path, index=False, compression="zstd")\n\n    full_test_metrics_payload = {\n        "model": model_name.lower(),\n        "architecture": cfg.get("architecture", model_name.upper()),\n        "seed": int(seed),\n        "device": str(device),\n        "best_epoch": int(best_epoch),\n        "best_validation_score": float(best_score),\n        "monitor_metric": monitor_metric,\n        "cold_embedding_max_abs_drift": max_cold_drift,\n        "test_warm_all_prefix": test_metrics_main,\n        "test_warm_last_item": test_metrics_last_item,\n        "test_warm_by_popularity_group": test_stratified_df.to_dict(orient="records"),\n        "popularity_bias": bias_summary,\n        "exposure_by_group": group_exposure_df.to_dict(orient="records"),\n        "cold_start_diagnostic": cold_diagnostic,\n    }\n    save_json(full_test_metrics_payload, run_dir / "test_metrics.json")\n\n    logger.info("-" * 72)\n    logger.info(\n        f"[KẾT QUẢ TEST WARM] Recall@{topk}: {test_metrics_main[f\'recall@{topk}\']:.4f} | "\n        f"MRR@{topk}: {test_metrics_main[f\'mrr@{topk}\']:.4f} | "\n        f"NDCG@{topk}: {test_metrics_main[f\'ndcg@{topk}\']:.4f}"\n    )\n    logger.info(\n        f"[TEST LAST-ITEM]    Recall@{topk}: {test_metrics_last_item[f\'recall@{topk}\']:.4f} | "\n        f"MRR@{topk}: {test_metrics_last_item[f\'mrr@{topk}\']:.4f} | "\n        f"NDCG@{topk}: {test_metrics_last_item[f\'ndcg@{topk}\']:.4f}"\n    )\n    logger.info(\n        f"[POPULARITY BIAS]   Coverage_warm@{topk}: {bias_summary[f\'coverage_warm@{topk}\']:.4f} | "\n        f"Gini_warm@{topk}: {bias_summary[f\'gini_warm@{topk}\']:.4f} | "\n        f"ARP@{topk}: {bias_summary[f\'arp@{topk}\']:.2f}"\n    )\n    logger.info(f"Đã xuất đầy đủ kết quả vào: {run_dir} và {item_level_path}")\n    logger.info("=" * 72)\n\n    return {\n        "run_dir": run_dir,\n        "history_df": history_df,\n        "test_metrics": full_test_metrics_payload,\n        "test_stratified_df": test_stratified_df,\n        "group_exposure_df": group_exposure_df,\n        "item_level_df": item_level_df,\n    }\n', 'src/btl_k03/paths.py': 'from __future__ import annotations\n\nfrom pathlib import Path\n\n\ndef find_project_root(start: Path | None = None) -> Path:\n    """Tìm thư mục gốc của dự án dựa trên file BTL-K03-tong-quan-de-tai.md."""\n    current = (start or Path.cwd()).resolve()\n    for candidate in (current, *current.parents):\n        if (candidate / "BTL-K03-tong-quan-de-tai.md").exists():\n            return candidate\n    raise FileNotFoundError(\n        "Không tìm thấy thư mục gốc dự án BTL-K03. Hãy chạy từ bên trong repository."\n    )\n\n\nPROJECT_ROOT = find_project_root()\nCONFIGS_DIR = PROJECT_ROOT / "configs"\nDATA_DIR = PROJECT_ROOT / "data"\nRAW_DIGI_DIR = DATA_DIR / "raw" / "diginetica"\nINTERIM_DIGI_DIR = DATA_DIR / "interim" / "diginetica"\nPROCESSED_DIGI_DIR = DATA_DIR / "processed" / "diginetica"\nRECBOLE_DIGI_DIR = DATA_DIR / "processed" / "recbole" / "diginetica"\n\nRUNS_DIR = PROJECT_ROOT / "runs"\nRESULTS_DIR = PROJECT_ROOT / "results"\nITEM_LEVEL_DIR = RESULTS_DIR / "item_level"\nTABLES_DIR = RESULTS_DIR / "tables"\nFIGURES_DIR = RESULTS_DIR / "figures"\nSTAT_TESTS_DIR = RESULTS_DIR / "statistical_tests"\n\n\ndef get_run_dir(model_name: str, seed: int) -> Path:\n    """Trả về đường dẫn thư mục lưu kết quả huấn luyện cho từng mô hình và seed."""\n    path = RUNS_DIR / model_name.lower() / f"seed_{int(seed)}"\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef ensure_output_dirs() -> None:\n    """Đảm bảo tất cả các thư mục đầu ra đều tồn tại."""\n    for directory in (\n        RUNS_DIR,\n        ITEM_LEVEL_DIR,\n        TABLES_DIR,\n        FIGURES_DIR,\n        STAT_TESTS_DIR,\n    ):\n        directory.mkdir(parents=True, exist_ok=True)\n', 'src/btl_k03/utils.py': 'from __future__ import annotations\n\nimport copy\nimport json\nimport logging\nimport os\nimport random\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport yaml\n\nfrom .paths import CONFIGS_DIR, PROCESSED_DIGI_DIR\n\n\ndef set_global_seed(seed: int, deterministic_cudnn: bool = True) -> None:\n    """Cố định toàn bộ random seed cho Python, NumPy và PyTorch (nếu có)."""\n    seed = int(seed)\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed(seed)\n            torch.cuda.manual_seed_all(seed)\n        if deterministic_cudnn and hasattr(torch.backends, "cudnn"):\n            torch.backends.cudnn.deterministic = True\n            torch.backends.cudnn.benchmark = False\n    except ImportError:\n        pass\n\n\ndef load_yaml(path: Path) -> dict[str, Any]:\n    with path.open("r", encoding="utf-8") as stream:\n        data = yaml.safe_load(stream)\n    return data or {}\n\n\ndef save_yaml(data: dict[str, Any], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as stream:\n        yaml.safe_dump(data, stream, allow_unicode=True, sort_keys=False)\n\n\ndef load_json(path: Path) -> dict[str, Any]:\n    with path.open("r", encoding="utf-8") as stream:\n        return json.load(stream)\n\n\ndef save_json(data: dict[str, Any], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as stream:\n        json.dump(data, stream, ensure_ascii=False, indent=2)\n\n\ndef load_experiment_config(model_name: str) -> dict[str, Any]:\n    """Đọc và hợp nhất configs/data.yaml, configs/common.yaml và configs/{model_name}.yaml."""\n    data_cfg = load_yaml(CONFIGS_DIR / "data.yaml")\n    common_cfg = load_yaml(CONFIGS_DIR / "common.yaml")\n    model_cfg = load_yaml(CONFIGS_DIR / f"{model_name.lower()}.yaml")\n\n    merged = copy.deepcopy(common_cfg)\n    merged["data_config"] = data_cfg\n    merged["model_name"] = model_cfg.get("model_name", model_name.lower())\n    merged["architecture"] = model_cfg.get("architecture", model_name.upper())\n\n    model_params = copy.deepcopy(model_cfg.get("model_params", {}))\n    model_params.setdefault("embedding_size", common_cfg.get("embedding_size", 64))\n    merged["model_params"] = model_params\n\n    training_cfg = copy.deepcopy(common_cfg.get("training", {}))\n    training_cfg.update(model_cfg.get("training_overrides", {}))\n    merged["training"] = training_cfg\n\n    return merged\n\n\ndef load_item_mapping(processed_dir: Path = PROCESSED_DIGI_DIR) -> pd.DataFrame:\n    """Đọc bảng ánh xạ item chuẩn và sắp xếp đúng theo internal_item_id tăng dần (1..N)."""\n    mapping = pd.read_csv(processed_dir / "item_mapping.csv")\n    mapping = mapping.sort_values("internal_item_id", kind="mergesort").reset_index(drop=True)\n    return mapping\n\n\ndef create_run_logger(log_path: Path, logger_name: str) -> logging.Logger:\n    """Tạo logger ghi đồng thời ra file training.log và console."""\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    logger = logging.getLogger(logger_name)\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    logger.propagate = False\n\n    formatter = logging.Formatter(\n        fmt="%(asctime)s | %(levelname)s | %(message)s",\n        datefmt="%Y-%m-%d %H:%M:%S",\n    )\n\n    file_handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n\n    return logger\n', 'src/btl_k03/popularity_metrics.py': 'from __future__ import annotations\n\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\n\ndef compute_gini_coefficient(counts: np.ndarray | list[float] | pd.Series) -> float:\n    """\n    Tính hệ số Gini chuẩn trên vector tần suất được gợi ý (rec_count) của toàn bộ catalog hợp lệ,\n    bao gồm cả các item có rec_count = 0 (theo đúng yêu cầu Mục 9.3 tài liệu đề tài).\n    Giá trị thuộc [0, 1 - 1/N]: gần 0 là phân bổ đồng đều, gần 1 là tập trung cực đoan.\n    """\n    values = np.asarray(counts, dtype=np.float64).ravel()\n    if values.size == 0:\n        return 0.0\n    if np.any(values < 0):\n        raise ValueError("Tần suất gợi ý (rec_count) không được âm.")\n    total = float(values.sum())\n    if total == 0.0 or values.size == 1:\n        return 0.0\n\n    sorted_vals = np.sort(values)\n    n = sorted_vals.size\n    index = np.arange(1, n + 1, dtype=np.float64)\n    gini = float((np.sum((2.0 * index - n - 1.0) * sorted_vals)) / (n * total))\n    return max(0.0, gini)\n\n\ndef compute_lorenz_curve(\n    counts: np.ndarray | list[float] | pd.Series,\n) -> tuple[np.ndarray, np.ndarray]:\n    """Trả về tọa độ (item_share_cum, exposure_share_cum) để vẽ đường cong Lorenz."""\n    values = np.sort(np.asarray(counts, dtype=np.float64).ravel())\n    n = values.size\n    if n == 0 or values.sum() == 0:\n        return np.array([0.0, 1.0]), np.array([0.0, 1.0])\n    x = np.concatenate([[0.0], np.arange(1, n + 1, dtype=np.float64) / n])\n    y = np.concatenate([[0.0], np.cumsum(values) / values.sum()])\n    return x, y\n\n\ndef compute_catalog_coverage(\n    recommended_items: np.ndarray | list[int] | pd.Series,\n    catalog_size: int,\n) -> float:\n    """\n    Tính Catalog Coverage@K = (số item khác nhau xuất hiện ít nhất 1 lần trong các danh sách top-K)\n    chia cho tổng số item hợp lệ trong catalog.\n    """\n    if catalog_size <= 0:\n        raise ValueError("catalog_size phải lớn hơn 0.")\n    unique_items = pd.unique(np.asarray(recommended_items).ravel())\n    return float(len(unique_items) / float(catalog_size))\n\n\ndef compute_arp(\n    recommended_items: np.ndarray | list[int] | pd.Series,\n    item_popularity_map: dict[int, float] | pd.Series,\n    use_log1p: bool = False,\n) -> float:\n    """\n    Tính Average Recommendation Popularity (ARP):\n    Độ phổ biến trung bình trong tập train của tất cả các vị trí item xuất hiện trong top-K.\n    """\n    rec_array = np.asarray(recommended_items).ravel()\n    if rec_array.size == 0:\n        return 0.0\n\n    if isinstance(item_popularity_map, pd.Series):\n        pop_values = item_popularity_map.reindex(rec_array).fillna(0.0).to_numpy(dtype=np.float64)\n    else:\n        pop_values = np.array(\n            [float(item_popularity_map.get(int(item), 0.0)) for item in rec_array],\n            dtype=np.float64,\n        )\n\n    if use_log1p:\n        pop_values = np.log1p(pop_values)\n    return float(np.mean(pop_values))\n\n\ndef compute_ranking_metrics_from_predictions(\n    top_k_ids: np.ndarray,\n    target_ids: np.ndarray,\n    k: int = 20,\n) -> dict[str, float]:\n    """\n    Tính Recall@K (Hit Rate@K), MRR@K và NDCG@K từ ma trận dự đoán top_k_ids (N, K)\n    và vector nhãn thực tế target_ids (N,).\n    """\n    preds = np.asarray(top_k_ids)[:, :k]\n    targets = np.asarray(target_ids).reshape(-1, 1)\n\n    if preds.shape[0] == 0:\n        return {f"recall@{k}": 0.0, f"mrr@{k}": 0.0, f"ndcg@{k}": 0.0}\n\n    hits_matrix = preds == targets\n    hit_any = hits_matrix.any(axis=1)\n    recall = float(np.mean(hit_any))\n\n    ranks = np.argmax(hits_matrix, axis=1) + 1  # 1-indexed rank\n    mrr_scores = np.where(hit_any, 1.0 / ranks, 0.0)\n    ndcg_scores = np.where(hit_any, 1.0 / np.log2(ranks + 1.0), 0.0)\n\n    return {\n        f"recall@{k}": recall,\n        f"mrr@{k}": float(np.mean(mrr_scores)),\n        f"ndcg@{k}": float(np.mean(ndcg_scores)),\n    }\n\n\ndef compute_stratified_ranking_metrics(\n    top_k_ids: np.ndarray,\n    target_ids: np.ndarray,\n    target_groups: np.ndarray | pd.Series,\n    k: int = 20,\n) -> pd.DataFrame:\n    """Tính Recall@K, MRR@K, NDCG@K tổng thể và tách theo từng nhóm target_popularity_group."""\n    preds = np.asarray(top_k_ids)[:, :k]\n    targets = np.asarray(target_ids)\n    groups = np.asarray(target_groups)\n\n    rows = []\n    overall = compute_ranking_metrics_from_predictions(preds, targets, k=k)\n    rows.append(\n        {\n            "target_group": "all_warm",\n            "num_examples": int(len(targets)),\n            "example_share_pct": 100.0,\n            **overall,\n        }\n    )\n\n    for grp in ("popular", "medium", "rare"):\n        mask = groups == grp\n        count = int(mask.sum())\n        if count > 0:\n            grp_metrics = compute_ranking_metrics_from_predictions(\n                preds[mask], targets[mask], k=k\n            )\n        else:\n            grp_metrics = {f"recall@{k}": 0.0, f"mrr@{k}": 0.0, f"ndcg@{k}": 0.0}\n        rows.append(\n            {\n                "target_group": grp,\n                "num_examples": count,\n                "example_share_pct": float(count * 100.0 / max(1, len(targets))),\n                **grp_metrics,\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef evaluate_top20_popularity_bias(\n    top20_df: pd.DataFrame,\n    item_mapping: pd.DataFrame,\n    k: int = 20,\n) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:\n    """\n    Tính toàn bộ chỉ số thiên lệch độ phổ biến ở đầu ra từ bảng top20.parquet và item_mapping.csv:\n    - Coverage@20 (trên warm candidate catalog 29,936 item và trên full model catalog 37,583 item)\n    - Gini@20 (bao gồm cả các item có rec_count = 0)\n    - ARP@20 (theo tần suất train gốc và theo log1p(train_frequency))\n    - Bảng item_rec_stats (rec_count và exposure_share cho từng item)\n    - Bảng group_exposure_df (tỷ trọng lượt hiển thị theo nhóm popular/medium/rare/cold)\n    """\n    filtered_top = top20_df.loc[top20_df["rank"] <= k].copy()\n    total_Recommended_slots = len(filtered_top)\n    num_queries = filtered_top["query_id"].nunique() if "query_id" in filtered_top.columns else int(total_Recommended_slots // max(1, k))\n\n    rec_counts_series = filtered_top["item_id"].value_counts()\n\n    catalog = item_mapping[\n        [\n            "internal_item_id",\n            "item_id",\n            "category_id",\n            "internal_category_id",\n            "train_frequency",\n            "popularity_group",\n            "is_controlled_cold",\n            "is_trainable_warm",\n            "candidate_for_main_evaluation",\n        ]\n    ].copy()\n\n    catalog["rec_count"] = (\n        catalog["item_id"].map(rec_counts_series).fillna(0).astype(np.int64)\n    )\n    catalog["exposure_share"] = (\n        catalog["rec_count"] / float(max(1, total_Recommended_slots))\n    )\n\n    warm_catalog = catalog.loc[catalog["candidate_for_main_evaluation"]].copy()\n    n_warm_items = len(warm_catalog)\n    n_full_items = len(catalog)\n\n    unique_rec_warm = int((warm_catalog["rec_count"] > 0).sum())\n    unique_rec_full = int((catalog["rec_count"] > 0).sum())\n\n    coverage_warm = float(unique_rec_warm / max(1, n_warm_items))\n    coverage_full = float(unique_rec_full / max(1, n_full_items))\n\n    gini_warm = compute_gini_coefficient(warm_catalog["rec_count"].to_numpy())\n    gini_full = compute_gini_coefficient(catalog["rec_count"].to_numpy())\n\n    freq_lookup = catalog.set_index("item_id")["train_frequency"]\n    arp_raw = compute_arp(filtered_top["item_id"], freq_lookup, use_log1p=False)\n    arp_log1p = compute_arp(filtered_top["item_id"], freq_lookup, use_log1p=True)\n\n    train_total_interactions = float(warm_catalog["train_frequency"].sum())\n    group_rows = []\n    for grp in ("popular", "medium", "rare", "cold"):\n        sub = catalog.loc[catalog["popularity_group"] == grp]\n        item_cnt = int(len(sub))\n        rec_sum = int(sub["rec_count"].sum())\n        train_sum = int(sub["train_frequency"].sum())\n        active_items = int((sub["rec_count"] > 0).sum())\n        exposure_pct = float(rec_sum * 100.0 / max(1, total_Recommended_slots))\n        train_share_pct = float(train_sum * 100.0 / max(1.0, train_total_interactions))\n        catalog_share_warm_pct = (\n            float(item_cnt * 100.0 / max(1, n_warm_items)) if grp != "cold" else 0.0\n        )\n        group_rows.append(\n            {\n                "popularity_group": grp,\n                "catalog_items": item_cnt,\n                "warm_catalog_share_pct": catalog_share_warm_pct,\n                "train_interactions": train_sum,\n                "train_interaction_share_pct": train_share_pct,\n                "recommended_slots": rec_sum,\n                "exposure_share_pct": exposure_pct,\n                "exposure_to_train_ratio": (\n                    float(exposure_pct / train_share_pct) if train_share_pct > 0 else 0.0\n                ),\n                "unique_items_recommended": active_items,\n                "within_group_coverage_pct": float(\n                    active_items * 100.0 / max(1, item_cnt)\n                ),\n                "mean_rec_count_per_item": float(sub["rec_count"].mean()) if item_cnt > 0 else 0.0,\n                "median_rec_count_per_item": float(sub["rec_count"].median()) if item_cnt > 0 else 0.0,\n            }\n        )\n\n    group_exposure_df = pd.DataFrame(group_rows)\n\n    summary = {\n        "topk": int(k),\n        "num_queries": int(num_queries),\n        "total_recommendation_slots": int(total_Recommended_slots),\n        "warm_catalog_size": int(n_warm_items),\n        "full_catalog_size": int(n_full_items),\n        "unique_recommended_items_warm": unique_rec_warm,\n        "unique_recommended_items_full": unique_rec_full,\n        f"coverage_warm@{k}": coverage_warm,\n        f"coverage_full@{k}": coverage_full,\n        f"gini_warm@{k}": gini_warm,\n        f"gini_full@{k}": gini_full,\n        f"arp@{k}": arp_raw,\n        f"arp_log1p@{k}": arp_log1p,\n        "popular_group_exposure_share_pct": float(\n            group_exposure_df.loc[\n                group_exposure_df["popularity_group"] == "popular", "exposure_share_pct"\n            ].iloc[0]\n        ),\n        "rare_group_exposure_share_pct": float(\n            group_exposure_df.loc[\n                group_exposure_df["popularity_group"] == "rare", "exposure_share_pct"\n            ].iloc[0]\n        ),\n    }\n\n    return summary, catalog, group_exposure_df\n\n\ndef compute_cold_start_test_diagnostic(\n    warm_metrics: dict[str, float],\n    test_events_full: pd.DataFrame,\n    warm_test_examples_count: int,\n    k: int = 20,\n) -> dict[str, Any]:\n    """\n    Phân tích đối chứng giữa đánh giá trên tập Warm-item chính thức và tập Full Test (có chứa\n    20% Controlled Cold + Natural Cold items). Giúp trả lời chặt chẽ câu hỏi của giảng viên về\n    hướng đánh giá Warm-item vs. Cold-start.\n    """\n    ordered = test_events_full.sort_values(\n        ["session_id", "timeframe", "row_order"], kind="mergesort"\n    )\n    session_lengths = ordered.groupby("session_id").size()\n    valid_sessions = set(session_lengths[session_lengths >= 2].index)\n    ordered = ordered.loc[ordered["session_id"].isin(valid_sessions)].copy()\n\n    ordered["pos"] = ordered.groupby("session_id").cumcount()\n    targets_full = ordered.loc[ordered["pos"] >= 1].copy()\n\n    total_full_targets = len(targets_full)\n    controlled_cold_targets = int(targets_full["is_controlled_cold"].fillna(False).sum())\n    natural_cold_targets = int(targets_full["is_natural_cold"].fillna(False).sum())\n    warm_trainable_targets = int(\n        (\n            targets_full["internal_item_id"].notna()\n            & (~targets_full["is_controlled_cold"].fillna(False))\n        ).sum()\n    )\n    post_holdout_attrition_targets = int(\n        total_full_targets\n        - controlled_cold_targets\n        - natural_cold_targets\n        - warm_trainable_targets\n    )\n\n    # Đối với baseline ID-based, mọi target rơi vào cold item đều có xác suất trúng = 0\n    warm_retention_ratio = float(warm_test_examples_count / max(1, total_full_targets))\n\n    return {\n        "full_test_sessions_len_ge_2": int(len(valid_sessions)),\n        "full_test_all_prefix_targets": int(total_full_targets),\n        "warm_evaluated_targets": int(warm_test_examples_count),\n        "controlled_cold_targets_in_test": controlled_cold_targets,\n        "controlled_cold_target_share_pct": float(\n            controlled_cold_targets * 100.0 / max(1, total_full_targets)\n        ),\n        "natural_cold_targets_in_test": natural_cold_targets,\n        "natural_cold_target_share_pct": float(\n            natural_cold_targets * 100.0 / max(1, total_full_targets)\n        ),\n        "other_unseen_or_filtered_targets": post_holdout_attrition_targets,\n        "warm_target_coverage_in_full_test_pct": float(warm_retention_ratio * 100.0),\n        f"warm_recall@{k}": float(warm_metrics.get(f"recall@{k}", 0.0)),\n        f"warm_mrr@{k}": float(warm_metrics.get(f"mrr@{k}", 0.0)),\n        f"warm_ndcg@{k}": float(warm_metrics.get(f"ndcg@{k}", 0.0)),\n        f"effective_full_test_recall@{k}_with_cold_zero": float(\n            warm_metrics.get(f"recall@{k}", 0.0) * warm_retention_ratio\n        ),\n        f"effective_full_test_mrr@{k}_with_cold_zero": float(\n            warm_metrics.get(f"mrr@{k}", 0.0) * warm_retention_ratio\n        ),\n        f"effective_full_test_ndcg@{k}_with_cold_zero": float(\n            warm_metrics.get(f"ndcg@{k}", 0.0) * warm_retention_ratio\n        ),\n    }\n', 'src/btl_k03/embedding_analysis.py': 'from __future__ import annotations\n\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\nfrom sklearn.decomposition import PCA\nfrom sklearn.manifold import TSNE\n\n\ndef compute_embedding_norms(embeddings: np.ndarray) -> np.ndarray:\n    """Tính chuẩn L2 (Euclidean norm) cho từng hàng vector item embedding."""\n    matrix = np.asarray(embeddings, dtype=np.float64)\n    return np.linalg.norm(matrix, axis=1)\n\n\ndef compute_category_centroid_metrics(\n    embeddings: np.ndarray,\n    item_mapping: pd.DataFrame,\n) -> pd.DataFrame:\n    """\n    Tính tâm category (centroid) chỉ từ các WARM item trong category đó (vì cold item chưa được học),\n    sau đó đo khoảng cách Euclidean và độ tương đồng Cosine từ mọi item (cả warm và cold)\n    đến tâm category của chính nó.\n    """\n    emb = np.asarray(embeddings, dtype=np.float64)\n    if len(emb) != len(item_mapping):\n        raise ValueError(\n            f"Số hàng embedding ({len(emb)}) không khớp số dòng item_mapping ({len(item_mapping)})."\n        )\n\n    norms = np.linalg.norm(emb, axis=1, keepdims=True)\n    unit_emb = emb / np.clip(norms, 1e-12, None)\n\n    df = item_mapping.copy().reset_index(drop=True)\n    df["embedding_norm"] = norms.ravel()\n\n    warm_mask = df["is_trainable_warm"].to_numpy(dtype=bool)\n    categories = df["category_id"].to_numpy()\n\n    # Tính centroid của từng category dựa trên các warm item thuộc category đó\n    unique_cats = np.unique(categories)\n    cat_centroids: dict[int, np.ndarray] = {}\n    cat_unit_centroids: dict[int, np.ndarray] = {}\n    cat_warm_counts: dict[int, int] = {}\n\n    for cat_id in unique_cats:\n        cat_warm_idx = np.where((categories == cat_id) & warm_mask)[0]\n        cat_warm_counts[int(cat_id)] = int(len(cat_warm_idx))\n        if len(cat_warm_idx) > 0:\n            centroid = emb[cat_warm_idx].mean(axis=0)\n            c_norm = float(np.linalg.norm(centroid))\n            unit_centroid = centroid / max(c_norm, 1e-12)\n            cat_centroids[int(cat_id)] = centroid\n            cat_unit_centroids[int(cat_id)] = unit_centroid\n\n    distances = np.full(len(df), np.nan, dtype=np.float64)\n    cosines = np.full(len(df), np.nan, dtype=np.float64)\n    warm_peers = np.zeros(len(df), dtype=np.int64)\n\n    for idx, cat_id in enumerate(categories):\n        cid = int(cat_id)\n        warm_peers[idx] = cat_warm_counts.get(cid, 0)\n        if cid in cat_centroids:\n            distances[idx] = float(np.linalg.norm(emb[idx] - cat_centroids[cid]))\n            cosines[idx] = float(np.dot(unit_emb[idx], cat_unit_centroids[cid]))\n\n    df["category_warm_peers"] = warm_peers\n    df["distance_to_category_centroid"] = distances\n    df["cosine_to_category_centroid"] = cosines\n    return df\n\n\ndef build_item_level_analysis_table(\n    model_name: str,\n    seed: int,\n    embeddings: np.ndarray,\n    item_rec_catalog: pd.DataFrame,\n) -> pd.DataFrame:\n    """\n    Tạo bảng phân tích trung tâm cấp độ item (Mục 11 tài liệu tổng quan đề tài),\n    kết nối tần suất train, nhóm popularity, embedding norm, khoảng cách tâm category\n    với số lần xuất hiện trong top-20 (rec_count) và exposure_share.\n    """\n    ordered_catalog = item_rec_catalog.sort_values(\n        "internal_item_id", kind="mergesort"\n    ).reset_index(drop=True)\n    with_geometry = compute_category_centroid_metrics(embeddings, ordered_catalog)\n\n    result = pd.DataFrame(\n        {\n            "model": model_name.lower(),\n            "seed": int(seed),\n            "internal_item_id": with_geometry["internal_item_id"].astype(np.int64),\n            "item_id": with_geometry["item_id"].astype(np.int64),\n            "category_id": with_geometry["category_id"].astype(np.int64),\n            "internal_category_id": with_geometry["internal_category_id"].astype(np.int64),\n            "train_freq": with_geometry["train_frequency"].astype(np.int64),\n            "popularity_group": with_geometry["popularity_group"].astype(str),\n            "is_cold": with_geometry["is_controlled_cold"].astype(bool),\n            "is_trainable_warm": with_geometry["is_trainable_warm"].astype(bool),\n            "candidate_for_main_evaluation": with_geometry[\n                "candidate_for_main_evaluation"\n            ].astype(bool),\n            "embedding_norm": with_geometry["embedding_norm"].astype(np.float64),\n            "rec_count": with_geometry["rec_count"].astype(np.int64),\n            "exposure_share": with_geometry["exposure_share"].astype(np.float64),\n            "distance_to_category_centroid": with_geometry[\n                "distance_to_category_centroid"\n            ].astype(np.float64),\n            "cosine_to_category_centroid": with_geometry[\n                "cosine_to_category_centroid"\n            ].astype(np.float64),\n            "category_warm_peers": with_geometry["category_warm_peers"].astype(np.int64),\n        }\n    )\n    return result\n\n\ndef spearman_with_bootstrap_ci(\n    x: np.ndarray | pd.Series,\n    y: np.ndarray | pd.Series,\n    n_bootstrap: int = 1000,\n    confidence: float = 0.95,\n    seed: int = 2024,\n) -> dict[str, float]:\n    """\n    Tính hệ số tương quan hạng Spearman kèm p-value và khoảng tin cậy Bootstrap 95%\n    (theo đúng yêu cầu Mục 10.2 & Mục 15 của đề tài).\n    """\n    x_arr = np.asarray(x, dtype=np.float64)\n    y_arr = np.asarray(y, dtype=np.float64)\n    valid = np.isfinite(x_arr) & np.isfinite(y_arr)\n    x_arr = x_arr[valid]\n    y_arr = y_arr[valid]\n\n    n = len(x_arr)\n    if n < 3 or np.all(x_arr == x_arr[0]) or np.all(y_arr == y_arr[0]):\n        return {\n            "n": int(n),\n            "rho": 0.0,\n            "p_value": 1.0,\n            "ci_low": 0.0,\n            "ci_high": 0.0,\n        }\n\n    res = stats.spearmanr(x_arr, y_arr)\n    rho = float(res.statistic)\n    p_val = float(res.pvalue)\n\n    if n_bootstrap > 0:\n        rng = np.random.default_rng(seed)\n        rx = stats.rankdata(x_arr)\n        ry = stats.rankdata(y_arr)\n        boot_rhos = np.empty(n_bootstrap, dtype=np.float64)\n        for b in range(n_bootstrap):\n            idx = rng.integers(0, n, size=n)\n            bx = rx[idx]\n            by = ry[idx]\n            std_x = bx.std()\n            std_y = by.std()\n            if std_x > 0 and std_y > 0:\n                boot_rhos[b] = np.corrcoef(bx, by)[0, 1]\n            else:\n                boot_rhos[b] = 0.0\n        alpha = (1.0 - confidence) / 2.0\n        ci_low = float(np.quantile(boot_rhos, alpha))\n        ci_high = float(np.quantile(boot_rhos, 1.0 - alpha))\n    else:\n        ci_low, ci_high = rho, rho\n\n    return {\n        "n": int(n),\n        "rho": rho,\n        "p_value": p_val,\n        "ci_low": ci_low,\n        "ci_high": ci_high,\n    }\n\n\ndef analyze_three_core_relationships(\n    item_table: pd.DataFrame,\n    n_bootstrap: int = 500,\n    seed: int = 2024,\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\n    """\n    Phân tích 3 mối quan hệ cốt lõi trên tập warm items (Mục 12.1 & 12.2 tài liệu đề tài):\n    1. train_freq <-> embedding_norm\n    2. train_freq <-> rec_count\n    3. embedding_norm <-> rec_count\n    Kèm phân tích tách theo từng bin tần suất (rare, medium, popular) và hồi quy tuyến tính\n    kiểm soát biến nhiễu log(1 + train_freq).\n    """\n    warm = item_table.loc[item_table["is_trainable_warm"]].copy()\n    all_items = item_table.copy()\n\n    core_rows = []\n    # 1. Warm items: 3 core relationships\n    pairs = [\n        ("warm_only", "train_freq", "embedding_norm", warm["train_freq"], warm["embedding_norm"]),\n        ("warm_only", "train_freq", "rec_count", warm["train_freq"], warm["rec_count"]),\n        ("warm_only", "embedding_norm", "rec_count", warm["embedding_norm"], warm["rec_count"]),\n        ("all_catalog_incl_cold", "train_freq", "embedding_norm", all_items["train_freq"], all_items["embedding_norm"]),\n    ]\n\n    for scope, var_x, var_y, s_x, s_y in pairs:\n        stat_dict = spearman_with_bootstrap_ci(\n            s_x, s_y, n_bootstrap=n_bootstrap, seed=seed\n        )\n        core_rows.append(\n            {\n                "model": str(item_table["model"].iloc[0]),\n                "seed": int(item_table["seed"].iloc[0]),\n                "scope": scope,\n                "pair": f"{var_x} <-> {var_y}",\n                "var_x": var_x,\n                "var_y": var_y,\n                **stat_dict,\n            }\n        )\n\n    # 2. Within-bin correlation: embedding_norm <-> rec_count inside each popularity_group\n    bin_rows = []\n    for grp in ("rare", "medium", "popular"):\n        sub = warm.loc[warm["popularity_group"] == grp]\n        stat_norm_rec = spearman_with_bootstrap_ci(\n            sub["embedding_norm"], sub["rec_count"], n_bootstrap=n_bootstrap, seed=seed\n        )\n        stat_freq_norm = spearman_with_bootstrap_ci(\n            sub["train_freq"], sub["embedding_norm"], n_bootstrap=n_bootstrap, seed=seed\n        )\n        bin_rows.append(\n            {\n                "model": str(item_table["model"].iloc[0]),\n                "seed": int(item_table["seed"].iloc[0]),\n                "popularity_group": grp,\n                "n_items": int(len(sub)),\n                "norm_rec_rho": stat_norm_rec["rho"],\n                "norm_rec_p_value": stat_norm_rec["p_value"],\n                "norm_rec_ci_low": stat_norm_rec["ci_low"],\n                "norm_rec_ci_high": stat_norm_rec["ci_high"],\n                "freq_norm_rho": stat_freq_norm["rho"],\n                "freq_norm_p_value": stat_freq_norm["p_value"],\n            }\n        )\n\n    # 3. Multiple OLS regression: log(1 + rec_count) = beta_0 + beta_1 * log(1 + train_freq) + beta_2 * norm\n    y_reg = np.log1p(warm["rec_count"].to_numpy(dtype=np.float64))\n    x_freq = np.log1p(warm["train_freq"].to_numpy(dtype=np.float64))\n    x_norm = warm["embedding_norm"].to_numpy(dtype=np.float64)\n\n    # Chuẩn hóa Z-score để có thêm hệ số beta chuẩn hóa (standardized beta)\n    x_design = np.column_stack([np.ones_like(x_freq), x_freq, x_norm])\n    coeffs, _, _, _ = np.linalg.lstsq(x_design, y_reg, rcond=None)\n    y_pred = x_design @ coeffs\n    residuals = y_reg - y_pred\n    dof = max(1, len(y_reg) - 3)\n    mse = float(np.sum(residuals**2) / dof)\n    cov_beta = mse * np.linalg.pinv(x_design.T @ x_design)\n    se_beta = np.sqrt(np.clip(np.diag(cov_beta), 1e-15, None))\n    t_stats = coeffs / se_beta\n    p_vals = 2.0 * (1.0 - stats.t.cdf(np.abs(t_stats), df=dof))\n    ss_tot = float(np.sum((y_reg - y_reg.mean()) ** 2))\n    r_squared = float(1.0 - np.sum(residuals**2) / max(ss_tot, 1e-12))\n\n    regression_summary = {\n        "model": str(item_table["model"].iloc[0]),\n        "seed": int(item_table["seed"].iloc[0]),\n        "n_warm_items": int(len(warm)),\n        "intercept_beta0": float(coeffs[0]),\n        "log1p_freq_beta1": float(coeffs[1]),\n        "log1p_freq_se": float(se_beta[1]),\n        "log1p_freq_t": float(t_stats[1]),\n        "log1p_freq_p": float(p_vals[1]),\n        "norm_beta2": float(coeffs[2]),\n        "norm_se": float(se_beta[2]),\n        "norm_t": float(t_stats[2]),\n        "norm_p": float(p_vals[2]),\n        "r_squared": r_squared,\n    }\n\n    return pd.DataFrame(core_rows), pd.DataFrame(bin_rows), regression_summary\n\n\ndef sample_category_cosine_pairs(\n    embeddings: np.ndarray,\n    item_mapping: pd.DataFrame,\n    n_pairs_per_group: int = 50000,\n    seed: int = 2024,\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    """\n    Lấy mẫu ngẫu nhiên có kiểm soát (cố định seed) các cặp item để tính Cosine Similarity\n    trên 4 nhóm:\n    1. warm_same_category: hai item warm cùng category\n    2. warm_diff_category: hai item warm khác category\n    3. cold_warm_same_category: một item cold và một item warm cùng category\n    4. cold_warm_diff_category: một item cold và một item warm khác category\n    Tránh tạo ma trận 37,583 x 37,583 (~1.4 tỷ cặp) theo đúng hướng dẫn Mục 10.4.\n    """\n    rng = np.random.default_rng(seed)\n    emb = np.asarray(embeddings, dtype=np.float64)\n    norms = np.linalg.norm(emb, axis=1, keepdims=True)\n    unit_emb = emb / np.clip(norms, 1e-12, None)\n\n    df = item_mapping.sort_values("internal_item_id", kind="mergesort").reset_index(drop=True)\n    cats = df["category_id"].to_numpy()\n    is_warm = df["is_trainable_warm"].to_numpy(dtype=bool)\n    is_cold = df["is_controlled_cold"].to_numpy(dtype=bool)\n\n    warm_indices = np.where(is_warm)[0]\n    cold_indices = np.where(is_cold)[0]\n\n    # Gom index theo category\n    warm_by_cat: dict[int, np.ndarray] = {}\n    cold_by_cat: dict[int, np.ndarray] = {}\n    for c in np.unique(cats):\n        cid = int(c)\n        w_idx = np.where(is_warm & (cats == c))[0]\n        c_idx = np.where(is_cold & (cats == c))[0]\n        if len(w_idx) >= 2:\n            warm_by_cat[cid] = w_idx\n        if len(c_idx) >= 1 and len(w_idx) >= 1:\n            cold_by_cat[cid] = (c_idx, w_idx)\n\n    # 1. Warm - Warm cùng category\n    valid_warm_cats = list(warm_by_cat.keys())\n    cat_weights = np.array([len(warm_by_cat[c]) for c in valid_warm_cats], dtype=np.float64)\n    cat_probs = cat_weights / cat_weights.sum()\n\n    chosen_cats = rng.choice(valid_warm_cats, size=n_pairs_per_group, p=cat_probs)\n    idx_a_same = np.empty(n_pairs_per_group, dtype=np.int64)\n    idx_b_same = np.empty(n_pairs_per_group, dtype=np.int64)\n    for i, cid in enumerate(chosen_cats):\n        pool = warm_by_cat[int(cid)]\n        pair = rng.choice(pool, size=2, replace=False)\n        idx_a_same[i], idx_b_same[i] = pair[0], pair[1]\n    cos_warm_same = np.sum(unit_emb[idx_a_same] * unit_emb[idx_b_same], axis=1)\n\n    # 2. Warm - Warm khác category\n    idx_a_diff = np.empty(n_pairs_per_group, dtype=np.int64)\n    idx_b_diff = np.empty(n_pairs_per_group, dtype=np.int64)\n    filled = 0\n    while filled < n_pairs_per_group:\n        batch = min(n_pairs_per_group - filled + 2048, n_pairs_per_group * 2)\n        cand_a = rng.choice(warm_indices, size=batch, replace=True)\n        cand_b = rng.choice(warm_indices, size=batch, replace=True)\n        valid = cats[cand_a] != cats[cand_b]\n        cand_a, cand_b = cand_a[valid], cand_b[valid]\n        take = min(len(cand_a), n_pairs_per_group - filled)\n        idx_a_diff[filled : filled + take] = cand_a[:take]\n        idx_b_diff[filled : filled + take] = cand_b[:take]\n        filled += take\n    cos_warm_diff = np.sum(unit_emb[idx_a_diff] * unit_emb[idx_b_diff], axis=1)\n\n    # 3. Cold - Warm cùng category\n    valid_cw_cats = list(cold_by_cat.keys())\n    cw_weights = np.array([len(cold_by_cat[c][0]) for c in valid_cw_cats], dtype=np.float64)\n    cw_probs = cw_weights / cw_weights.sum()\n    chosen_cw_cats = rng.choice(valid_cw_cats, size=n_pairs_per_group, p=cw_probs)\n    idx_c_same = np.empty(n_pairs_per_group, dtype=np.int64)\n    idx_w_same = np.empty(n_pairs_per_group, dtype=np.int64)\n    for i, cid in enumerate(chosen_cw_cats):\n        c_pool, w_pool = cold_by_cat[int(cid)]\n        idx_c_same[i] = rng.choice(c_pool)\n        idx_w_same[i] = rng.choice(w_pool)\n    cos_cw_same = np.sum(unit_emb[idx_c_same] * unit_emb[idx_w_same], axis=1)\n\n    # 4. Cold - Warm khác category\n    idx_c_diff = np.empty(n_pairs_per_group, dtype=np.int64)\n    idx_w_diff = np.empty(n_pairs_per_group, dtype=np.int64)\n    filled = 0\n    while filled < n_pairs_per_group:\n        batch = min(n_pairs_per_group - filled + 2048, n_pairs_per_group * 2)\n        cand_c = rng.choice(cold_indices, size=batch, replace=True)\n        cand_w = rng.choice(warm_indices, size=batch, replace=True)\n        valid = cats[cand_c] != cats[cand_w]\n        cand_c, cand_w = cand_c[valid], cand_w[valid]\n        take = min(len(cand_c), n_pairs_per_group - filled)\n        idx_c_diff[filled : filled + take] = cand_c[:take]\n        idx_w_diff[filled : filled + take] = cand_w[:take]\n        filled += take\n    cos_cw_diff = np.sum(unit_emb[idx_c_diff] * unit_emb[idx_w_diff], axis=1)\n\n    groups_data = [\n        ("warm_same_category", cos_warm_same),\n        ("warm_diff_category", cos_warm_diff),\n        ("cold_warm_same_category", cos_cw_same),\n        ("cold_warm_diff_category", cos_cw_diff),\n    ]\n\n    summary_rows = []\n    for name, arr in groups_data:\n        mean_val = float(np.mean(arr))\n        std_val = float(np.std(arr, ddof=1))\n        se_val = std_val / np.sqrt(len(arr))\n        summary_rows.append(\n            {\n                "pair_group": name,\n                "n_pairs": int(len(arr)),\n                "mean_cosine": mean_val,\n                "std_cosine": std_val,\n                "median_cosine": float(np.median(arr)),\n                "ci95_low": float(mean_val - 1.96 * se_val),\n                "ci95_high": float(mean_val + 1.96 * se_val),\n            }\n        )\n\n    u_warm, p_warm = stats.mannwhitneyu(cos_warm_same, cos_warm_diff, alternative="two-sided")\n    t_warm, tp_warm = stats.ttest_ind(cos_warm_same, cos_warm_diff, equal_var=False)\n\n    u_cold, p_cold = stats.mannwhitneyu(cos_cw_same, cos_cw_diff, alternative="two-sided")\n\n    tests_dict = {\n        "warm_same_vs_diff_mean_gap": float(np.mean(cos_warm_same) - np.mean(cos_warm_diff)),\n        "warm_same_vs_diff_ttest_t": float(t_warm),\n        "warm_same_vs_diff_ttest_p": float(tp_warm),\n        "warm_same_vs_diff_mannwhitney_u": float(u_warm),\n        "warm_same_vs_diff_mannwhitney_p": float(p_warm),\n        "cold_warm_same_vs_diff_mean_gap": float(np.mean(cos_cw_same) - np.mean(cos_cw_diff)),\n        "cold_warm_same_vs_diff_mannwhitney_u": float(u_cold),\n        "cold_warm_same_vs_diff_mannwhitney_p": float(p_cold),\n    }\n\n    return pd.DataFrame(summary_rows), tests_dict\n\n\ndef compute_tsne_sample(\n    embeddings: np.ndarray,\n    item_table: pd.DataFrame,\n    sample_size: int = 8000,\n    perplexity: int = 30,\n    seed: int = 2024,\n    n_top_categories: int = 8,\n) -> pd.DataFrame:\n    """\n    Lấy mẫu phân tầng 5,000 - 10,000 item qua 4 nhóm (popular, medium, rare, cold)\n    và chạy t-SNE giảm chiều xuống 2D (kèm PCA tiền xử lý nếu chiều > 32).\n    """\n    rng = np.random.default_rng(seed)\n    df = item_table.sort_values("internal_item_id", kind="mergesort").reset_index(drop=True)\n    emb = np.asarray(embeddings, dtype=np.float32)\n\n    groups = ["popular", "medium", "rare", "cold"]\n    per_group = sample_size // len(groups)\n    sampled_indices = []\n\n    for grp in groups:\n        grp_idx = np.where(df["popularity_group"].to_numpy() == grp)[0]\n        take = min(len(grp_idx), per_group)\n        chosen = rng.choice(grp_idx, size=take, replace=False)\n        sampled_indices.append(chosen)\n\n    sampled_idx = np.sort(np.concatenate(sampled_indices))\n    sub_df = df.iloc[sampled_idx].copy().reset_index(drop=True)\n    sub_emb = emb[sampled_idx]\n\n    if sub_emb.shape[1] > 32:\n        pca = PCA(n_components=32, random_state=seed)\n        sub_emb_input = pca.fit_transform(sub_emb)\n    else:\n        sub_emb_input = sub_emb\n\n    tsne = TSNE(\n        n_components=2,\n        perplexity=perplexity,\n        init="pca",\n        learning_rate="auto",\n        max_iter=1000,\n        random_state=seed,\n    )\n    coords = tsne.fit_transform(sub_emb_input)\n\n    sub_df["tsne_1"] = coords[:, 0]\n    sub_df["tsne_2"] = coords[:, 1]\n    sub_df["tsne_perplexity"] = int(perplexity)\n    sub_df["tsne_seed"] = int(seed)\n\n    # Đánh dấu các category lớn nhất để tô màu biểu đồ t-SNE theo category dễ quan sát\n    top_cats = (\n        df.loc[df["is_trainable_warm"], "category_id"]\n        .value_counts()\n        .head(n_top_categories)\n        .index.tolist()\n    )\n    sub_df["category_display"] = sub_df["category_id"].apply(\n        lambda c: f"Cat {int(c)}" if c in top_cats else "Other Categories"\n    )\n\n    return sub_df\n', 'src/btl_k03/score_calibration.py': '"""Inference-time popularity scoring control; this is not a novel PH method."""\nfrom pathlib import Path\nimport json\nimport math\n\nimport torch\nfrom torch import nn\n\n\nclass PopularityScoreCalibrator(nn.Module):\n    """Apply a validation-selected, frozen coefficient to full-catalog logits.\n\n    Model parameters/embeddings stay unchanged. The buffer adds one float per\n    catalog entry. Candidate masking remains the caller\'s responsibility.\n    """\n    def __init__(self, train_frequencies, coefficient):\n        super().__init__()\n        frequency = torch.as_tensor(train_frequencies, dtype=torch.float32).clone()\n        coefficient = float(coefficient)\n        if frequency.ndim != 1 or not torch.isfinite(frequency).all() or (frequency < 0).any():\n            raise ValueError("Expected finite nonnegative train frequencies")\n        if not math.isfinite(coefficient) or coefficient < 0:\n            raise ValueError("Expected a finite nonnegative coefficient")\n        self.register_buffer("log_popularity_penalty", torch.log1p(frequency))\n        self.coefficient = coefficient\n\n    def forward(self, logits):\n        if logits.shape[-1] != len(self.log_popularity_penalty):\n            raise ValueError("Logits and training catalog sizes differ")\n        return logits - self.coefficient * self.log_popularity_penalty.to(device=logits.device, dtype=logits.dtype)\n\n    @classmethod\n    def from_validation_policy(cls, policy_file: Path, item_mapping):\n        policy = json.loads(Path(policy_file).read_text(encoding="utf-8"))\n        if policy.get("selected_on") != "validation" or policy.get("test_seen_during_selection") is not False:\n            raise ValueError("Expected a policy locked on validation before test evaluation")\n        size = int(item_mapping.internal_item_id.max()) + 1\n        frequency = torch.zeros(size)\n        ids = torch.tensor(item_mapping.internal_item_id.to_numpy().copy(), dtype=torch.long)\n        frequency[ids] = torch.tensor(item_mapping.train_frequency.to_numpy().copy(), dtype=torch.float32)\n        return cls(frequency, policy["coefficient"])\n', 'src/btl_k03/artifacts.py': '"""Read embeddings from an export or the authoritative saved checkpoint."""\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef load_item_embeddings(run_dir: Path) -> np.ndarray:\n    """Return item rows 1..N (padding excluded), as the old NPY export did."""\n    run_dir = Path(run_dir)\n    export = run_dir / "item_embeddings.npy"\n    if export.exists():\n        return np.load(export, allow_pickle=False)\n    checkpoint = torch.load(run_dir / "checkpoint.pth", map_location="cpu", weights_only=False)\n    weight = checkpoint["model_state_dict"]["item_embedding.weight"]\n    return weight.detach().cpu().numpy()[1:].astype(np.float32, copy=True)\n', 'src/btl_k03/statistics.py': '"""Paired session-cluster comparisons shared by experiment reports."""\nimport numpy as np\n\n\ndef clustered_delta(control, treatment, metric, group, iterations, rng):\n    keys = ["source_row", "session_id", "target", "group"]\n    aligned = control.merge(treatment, on=keys, suffixes=("_control", "_treatment"), validate="one_to_one")\n    if len(aligned) != len(control) or len(aligned) != len(treatment):\n        raise ValueError("Cannot compare different evaluation queries")\n    if group != "all_warm":\n        aligned = aligned[aligned.group.eq(group)]\n    if aligned.empty:\n        return {"delta": None, "ci_low": None, "ci_high": None, "n_queries": 0, "n_sessions": 0}\n    aligned = aligned.assign(difference=aligned[f"{metric}_treatment"] - aligned[f"{metric}_control"])\n    clusters = aligned.groupby("session_id").difference.agg(["sum", "count"])\n    sums, counts = clusters["sum"].to_numpy(), clusters["count"].to_numpy()\n    bootstrap = []\n    for _ in range(iterations):\n        ids = rng.integers(0, len(clusters), size=len(clusters))\n        bootstrap.append(sums[ids].sum() / counts[ids].sum())\n    return {"delta": float(sums.sum() / counts.sum()), "ci_low": float(np.quantile(bootstrap, .025)),\n            "ci_high": float(np.quantile(bootstrap, .975)), "n_queries": len(aligned), "n_sessions": len(clusters)}\n', 'src/btl_k03/topology/__init__.py': '"""\nModule Topological Regularization & Persistence Geometry for Session Recommendation.\nNghiên cứu: Hiệu chỉnh thiên lệch độ phổ biến trong không gian embedding bằng Tô-pô Lân cận Differentiable.\n"""\n\nfrom .persistence_loss import CosineTailAlignmentLoss, TopologicalPersistenceLoss\nfrom .persistent_homology import vietoris_rips_persistence, diagram_wasserstein_squared\nfrom .topo_eval import evaluate_topological_properties, compute_anisotropy_spectrum\nfrom .topo_trainer import train_and_evaluate_topology_seed\n\n__all__ = [\n    "TopologicalPersistenceLoss",\n    "CosineTailAlignmentLoss",\n    "vietoris_rips_persistence",\n    "diagram_wasserstein_squared",\n    "evaluate_topological_properties",\n    "compute_anisotropy_spectrum",\n    "train_and_evaluate_topology_seed",\n]\n', 'src/btl_k03/topology/persistence_loss.py': '"""Actual PH regularisation and an explicitly named cosine-only control."""\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom .persistent_homology import diagram_wasserstein_squared, vietoris_rips_persistence\n\n\ndef session_dispersion(session_vectors: torch.Tensor) -> torch.Tensor:\n    """Off-diagonal squared cosine; does not promise isotropy when B > D."""\n    if len(session_vectors) < 2:\n        return session_vectors.sum() * 0.0\n    unit = F.normalize(session_vectors, dim=-1)\n    gram = unit @ unit.T\n    off_diagonal = ~torch.eye(len(unit), dtype=torch.bool, device=unit.device)\n    return gram[off_diagonal].square().mean()\n\n\nclass CosineTailAlignmentLoss(nn.Module):\n    """Pairwise angular hinge control. This is NOT persistent homology."""\n    def __init__(self, tau=0.45, lambda_anisotropy=0.05, tail_threshold=20):\n        super().__init__()\n        self.tau = tau\n        self.lambda_anisotropy = lambda_anisotropy\n        self.tail_threshold = tail_threshold\n\n    def forward(self, session_vectors, target_embeddings, item_frequencies, target_ids):\n        frequency = item_frequencies[target_ids]\n        tail = (frequency > 0) & (frequency < self.tail_threshold)\n        distance = 1 - F.cosine_similarity(session_vectors, target_embeddings, dim=-1)\n        alignment = F.relu(distance[tail] - self.tau).mean() if tail.any() else session_vectors.sum() * 0\n        return alignment + self.lambda_anisotropy * session_dispersion(session_vectors)\n\n\nclass TopologicalPersistenceLoss(nn.Module):\n    """Match local H0/H1 diagrams of sessions to detached target-item diagrams.\n\n    Anchor clouds on rare TRAIN targets, selecting distinct target-ID neighbors\n    by target-embedding distance in the same training batch. The paired cloud\n    uses corresponding session vectors. L2-normalise both; filtration uses\n    Euclidean chord distance [0, 2]. Reference/selection are detached, gradients\n    flow through sessions, including their input embedding path.\n\n    H0 preserves connectivity; H1 preserves actual loops killed by triangles.\n    Long bars do NOT establish semantic quality. Tail accuracy is a hypothesis.\n    """\n    def __init__(self, neighbors=8, max_clouds=8, max_dimension=1, h1_weight=1.0,\n                 lambda_anisotropy=0.05, tail_threshold=20):\n        super().__init__()\n        if not 3 <= neighbors <= 32 or max_clouds < 1:\n            raise ValueError("neighbors must be 3..32 and max_clouds >= 1")\n        if max_dimension not in (0, 1) or h1_weight < 0 or lambda_anisotropy < 0 or tail_threshold <= 0:\n            raise ValueError("Invalid PH loss parameters")\n        self.neighbors, self.max_clouds = neighbors, max_clouds\n        self.max_dimension, self.h1_weight = max_dimension, h1_weight\n        self.lambda_anisotropy, self.tail_threshold = lambda_anisotropy, tail_threshold\n        self.last_stats: dict[str, float] = {}\n        # Independent RNG: PH cloud sampling must not change subsequent dropout.\n        self.sampling_generator = torch.Generator().manual_seed(2026)\n\n    def forward(self, session_vectors, target_embeddings, item_frequencies, target_ids):\n        if session_vectors.shape != target_embeddings.shape or len(target_ids) != len(session_vectors):\n            raise ValueError("Sessions, targets and embeddings must have matching batch shapes")\n        student = F.normalize(session_vectors.float(), dim=-1)\n        reference = F.normalize(target_embeddings.detach().float(), dim=-1)\n        frequencies = item_frequencies[target_ids]\n        eligible = (frequencies > 0) & (frequencies < self.tail_threshold)\n        anchors = eligible.nonzero().flatten()\n        if len(anchors) > self.max_clouds:\n            selection = torch.randperm(len(anchors), generator=self.sampling_generator)[:self.max_clouds]\n            anchors = anchors[selection.to(anchors.device)]\n        representatives = {}\n        # Avoid per-element device synchronisation in the batch selection loop.\n        target_list = target_ids.detach().cpu().tolist()\n        frequency_list = frequencies.detach().cpu().tolist()\n        for i, target in enumerate(target_list):\n            if frequency_list[i] > 0:\n                representatives.setdefault(target, i)\n        losses0, losses1 = [], []\n        bars1 = 0\n        for anchor in anchors.tolist():\n            candidates = [i for target, i in representatives.items() if target != target_list[anchor]]\n            if len(candidates) < 2:\n                continue\n            candidate_ids = torch.tensor(candidates, device=student.device)\n            distances = (reference[candidate_ids] - reference[anchor]).square().sum(-1)\n            nearest = candidate_ids[torch.argsort(distances, stable=True)[:self.neighbors - 1]]\n            ids = torch.cat((nearest.new_tensor([anchor]), nearest))\n            predicted = vietoris_rips_persistence(student[ids], self.max_dimension)\n            expected = vietoris_rips_persistence(reference[ids], self.max_dimension)\n            # Normalise W2 sum by number of finite H0 bars; same scale for H1.\n            losses0.append(diagram_wasserstein_squared(predicted[0], expected[0]) / (len(ids) - 1))\n            if self.max_dimension == 1:\n                losses1.append(diagram_wasserstein_squared(predicted[1], expected[1]) / (len(ids) - 1))\n                bars1 += len(predicted[1]) + len(expected[1])\n        zero = student.sum() * 0.0\n        loss0 = torch.stack(losses0).mean() if losses0 else zero\n        loss1 = torch.stack(losses1).mean() if losses1 else zero\n        dispersion = session_dispersion(student)\n        self.last_stats = {"ph_h0": float(loss0.detach()), "ph_h1": float(loss1.detach()),\n                           "ph_clouds": len(losses0), "ph_h1_bars": bars1,\n                           "dispersion": float(dispersion.detach())}\n        return loss0 + self.h1_weight * loss1 + self.lambda_anisotropy * dispersion\n', 'src/btl_k03/topology/persistent_homology.py': '"""Exact small-cloud Vietoris--Rips PH over F_2, with live Torch endpoints.\n\nVertices enter at zero; a simplex enters at its longest Euclidean edge. Boundary\nreduction and diagram assignment are discrete, detached operations. Selected\nbirth/death edge lengths retain autograd away from filtration/assignment ties.\n"""\nfrom __future__ import annotations\n\nfrom itertools import combinations\n\nimport numpy as np\nimport torch\nfrom scipy.optimize import linear_sum_assignment\n\n\ndef vietoris_rips_persistence(points: torch.Tensor, max_dimension: int = 1) -> dict[int, torch.Tensor]:\n    """H0/H1 diagrams [birth, death], including essential H0 = [0, inf].\n\n    Triangles kill H1 classes; graph cycles are not reported as persistent holes.\n    Full filtration, no cutoff, at most 32 vertices. Omit zero-length intervals.\n    Unpaired triangles are outside the requested dimensions (no H2 claim).\n    """\n    if points.ndim != 2 or not 1 <= len(points) <= 32:\n        raise ValueError("points must have shape [1..32, dimension]")\n    if max_dimension not in (0, 1):\n        raise ValueError("Only H0 and H1 are supported")\n    if not points.is_floating_point() or not torch.isfinite(points).all():\n        raise ValueError("points must be finite floating-point coordinates")\n    distances = torch.cdist(points, points, compute_mode="donot_use_mm_for_euclid_dist")\n    detached = distances.detach().cpu().numpy()\n    simplices = []\n    for size in range(1, max_dimension + 3):\n        for vertices in combinations(range(len(points)), size):\n            edge = max(combinations(vertices, 2), key=lambda e: detached[e]) if size > 1 else None\n            value = float(detached[edge]) if edge is not None else 0.0\n            simplices.append((value, size - 1, vertices, edge))\n    simplices.sort(key=lambda s: (s[0], s[1], s[2]))\n    index = {s[2]: i for i, s in enumerate(simplices)}\n    reduced: dict[int, set[int]] = {}\n    pivot_to_column: dict[int, int] = {}\n    births, paired_births = set(), set()\n    pairs: dict[int, list[torch.Tensor]] = {d: [] for d in range(max_dimension + 1)}\n    zero = points.sum() * 0.0\n\n    def endpoint(i):\n        edge = simplices[i][3]\n        return distances[edge] if edge is not None else zero\n\n    for column, (_, dimension, vertices, _) in enumerate(simplices):\n        boundary = {index[face] for face in combinations(vertices, len(vertices) - 1)} if dimension else set()\n        while boundary and max(boundary) in pivot_to_column:\n            boundary ^= reduced[pivot_to_column[max(boundary)]]\n        if not boundary:\n            births.add(column)\n            continue\n        pivot = max(boundary)\n        reduced[column] = boundary\n        pivot_to_column[pivot] = column\n        paired_births.add(pivot)\n        homology_dimension = simplices[pivot][1]\n        if homology_dimension <= max_dimension and simplices[column][0] > simplices[pivot][0]:\n            pairs[homology_dimension].append(torch.stack((endpoint(pivot), endpoint(column))))\n    for birth in sorted(births - paired_births):\n        dimension = simplices[birth][1]\n        if dimension <= max_dimension:\n            pairs[dimension].append(torch.stack((endpoint(birth), zero.new_tensor(float("inf")))))\n    return {d: torch.stack(rows) if rows else points.new_empty((0, 2)) for d, rows in pairs.items()}\n\n\ndef finite_diagram(diagram: torch.Tensor) -> torch.Tensor:\n    return diagram[torch.isfinite(diagram).all(dim=1)]\n\n\ndef diagram_wasserstein_squared(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\n    """Squared W2, Euclidean ground metric, optimal matching to the diagonal.\n\n    Explicitly exclude essential bars. Unnormalised sum; caller may divide by\n    cloud size. Empty diagrams are valid and do not receive artificial bars.\n    """\n    a, b = finite_diagram(a), finite_diagram(b)\n    n, m = len(a), len(b)\n    zero = a.sum() * 0.0 + b.sum() * 0.0\n    if n == 0:\n        return ((b[:, 1] - b[:, 0]).square() / 2).sum() + zero\n    if m == 0:\n        return ((a[:, 1] - a[:, 0]).square() / 2).sum() + zero\n    cost = a.new_full((n + m, n + m), float("inf"))\n    cost[:n, :m] = (a[:, None, :] - b[None, :, :]).square().sum(dim=-1)\n    ai, bi = torch.arange(n, device=a.device), torch.arange(m, device=a.device)\n    cost[ai, m + ai] = (a[:, 1] - a[:, 0]).square() / 2\n    cost[n + bi, bi] = (b[:, 1] - b[:, 0]).square() / 2\n    cost[n:, m:] = 0.0\n    row, col = linear_sum_assignment(cost.detach().cpu().numpy().astype(np.float64))\n    return cost[row, col].sum()\n', 'src/btl_k03/topology/topo_trainer.py': '"""Reproducible CE/cosine/PH controls using the current SBR model APIs."""\nfrom __future__ import annotations\n\nimport copy\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\n\nfrom ..models import SessionPrefixDataset, build_model_from_config, build_top20_dataframe, predict_topk_for_dataset\nfrom ..paths import PROCESSED_DIGI_DIR, RUNS_DIR\nfrom ..popularity_metrics import compute_stratified_ranking_metrics, evaluate_top20_popularity_bias\nfrom ..utils import create_run_logger, load_experiment_config, load_item_mapping, save_json, save_yaml, set_global_seed\nfrom .persistence_loss import CosineTailAlignmentLoss, TopologicalPersistenceLoss, session_dispersion\nfrom .role_gradients import ROLE_METHODS, backward_role_loss\n\nMETHODS = ("baseline", "dispersion", "cosine", "ph_h0", "ph_h0_h1", *ROLE_METHODS)\n\n\ndef _sample(frame, limit, seed):\n    if limit is not None and limit < 1:\n        raise ValueError("Sample limits must be positive or None")\n    if limit is not None and limit < len(frame):\n        frame = frame.sample(n=limit, random_state=seed).sort_index()\n    # Retain original row index so prediction pairs are traceable across runs.\n    return frame.copy()\n\n\ndef _evaluate(model, frame, mapping, device, batch_size, topk, output, split):\n    dataset = SessionPrefixDataset(frame, max_seq_length=model._experiment_max_seq_length)\n    predictions, scores = predict_topk_for_dataset(model, dataset, device, batch_size, topk)\n    if not model.trainable_warm_mask[torch.as_tensor(predictions, device=device)].all():\n        raise AssertionError("Cold/padding item leaked into recommendations")\n    stratified = compute_stratified_ranking_metrics(predictions, dataset.targets, dataset.target_groups, k=topk)\n    records = stratified.to_dict(orient="records")\n    for row in records:\n        if row["num_examples"] == 0:\n            for metric in ("recall", "mrr", "ndcg"):\n                row[f"{metric}@{topk}"] = None\n    hits = predictions == dataset.targets[:, None]\n    hit = hits.any(axis=1)\n    ranks = hits.argmax(axis=1) + 1\n    per_query = pd.DataFrame({"source_row": frame.index.to_numpy(), "session_id": dataset.session_ids,\n                              "target": dataset.targets, "group": dataset.target_groups,\n                              "hit": hit.astype(int), "reciprocal_rank": np.where(hit, 1 / ranks, 0),\n                              "ndcg": np.where(hit, 1 / np.log2(ranks + 1), 0)})\n    per_query.to_csv(output / f"{split}_per_query.csv.gz", index=False)\n    lookup = np.zeros(int(mapping.internal_item_id.max()) + 1, dtype=np.int64)\n    lookup[mapping.internal_item_id.to_numpy()] = mapping.item_id.to_numpy()\n    top_frame = build_top20_dataframe(dataset, predictions, scores, lookup)\n    top_frame.to_parquet(output / f"{split}_top20.parquet", index=False)\n    bias, _, exposure = evaluate_top20_popularity_bias(top_frame, mapping, k=topk)\n    return {"ranking_by_group": records, "popularity_bias": bias,\n            "exposure_by_group": exposure.to_dict(orient="records")}\n\n\ndef train_and_evaluate_topology_seed(\n    seed: int, config: dict | None = None, processed_dir: Path = PROCESSED_DIGI_DIR,\n    max_epochs_override: int | None = None, device_str: str | None = None,\n    *, model_name: str = "sasrec", method: str = "ph_h0_h1", output_dir: Path | None = None,\n    train_limit: int | None = None, validation_limit: int | None = None,\n    evaluate_test: bool = False, init_checkpoint: Path | None = None,\n) -> dict:\n    """Train in a fresh directory; select on validation MRR, test only if asked.\n\n    Limited runs are exploratory pilots, never comparable to full-data baselines.\n    An optional common checkpoint permits a controlled fine-tuning pilot. Adam\n    state is deliberately fresh for every method; this is not training resume.\n    """\n    if method not in METHODS:\n        raise ValueError(f"method must be one of {METHODS}")\n    cfg = copy.deepcopy(config) if config is not None else load_experiment_config(model_name)\n    model_name = cfg.get("model_name", model_name).lower()\n    cfg.setdefault("topology_params", {})\n    cfg.update(seed=int(seed), method=method)\n    if max_epochs_override is not None:\n        cfg["training"]["epochs"] = int(max_epochs_override)\n    training, topology = cfg["training"], cfg["topology_params"]\n    epochs = int(training.get("epochs", 30))\n    interval = int(topology.get("regularization_interval", 1))\n    if interval < 1:\n        raise ValueError("regularization_interval must be positive")\n    if epochs < 1 or float(topology.get("lambda_topo", 0.15)) < 0:\n        raise ValueError("epochs must be positive and lambda_topo non-negative")\n    device = torch.device(device_str or ("cuda" if torch.cuda.is_available() else "cpu"))\n    set_global_seed(seed)\n    output = Path(output_dir) if output_dir is not None else RUNS_DIR / "persistent_homology" / model_name / method / f"seed_{seed}"\n    output.mkdir(parents=True, exist_ok=False)\n    logger = create_run_logger(output / "training.log", f"ph.{model_name}.{method}.{seed}")\n    mapping = load_item_mapping(processed_dir)\n    train = _sample(pd.read_parquet(processed_dir / cfg["data_files"]["train_examples"]), train_limit, seed)\n    validation = _sample(pd.read_parquet(processed_dir / cfg["data_files"]["validation_examples"]), validation_limit, seed + 1)\n    if train.empty or validation.empty:\n        raise ValueError("Training and validation data must be nonempty")\n    model = build_model_from_config(cfg, mapping, device)\n    if init_checkpoint is not None:\n        checkpoint = torch.load(init_checkpoint, map_location=device, weights_only=False)\n        if checkpoint.get("config", {}).get("model_name") != model_name:\n            raise ValueError("Initial checkpoint architecture does not match")\n        checkpoint_mapping = pd.read_csv(init_checkpoint.parent / "item_embedding_mapping.csv").sort_values("internal_item_id")\n        if not mapping[["internal_item_id", "item_id"]].reset_index(drop=True).equals(\n            checkpoint_mapping[["internal_item_id", "item_id"]].reset_index(drop=True)\n        ):\n            raise ValueError("Initial checkpoint item IDs differ from current mapping")\n        model.load_state_dict(checkpoint["model_state_dict"])\n        expected_mask = torch.zeros_like(model.trainable_warm_mask)\n        expected_mask[torch.tensor(mapping.loc[mapping.is_trainable_warm, "internal_item_id"].to_numpy(), device=device)] = True\n        if not torch.equal(model.trainable_warm_mask, expected_mask):\n            raise ValueError("Initial checkpoint warm mask differs from current mapping")\n    model._experiment_max_seq_length = int(cfg.get("max_seq_length", 50))\n    for frame in (train, validation):\n        ids = torch.tensor(frame.target_internal_item_id.to_numpy(), device=device)\n        if not model.trainable_warm_mask[ids].all():\n            raise ValueError("Non-warm training/validation target")\n    frozen = ~model.trainable_warm_mask\n    initial_frozen = model.item_embedding.weight.detach()[frozen].clone()\n    frequencies = torch.zeros(model.num_items_with_pad, dtype=torch.long, device=device)\n    frequencies[torch.tensor(mapping.internal_item_id.to_numpy(), device=device)] = torch.tensor(mapping.train_frequency.to_numpy(), device=device)\n    if method == "cosine":\n        criterion = CosineTailAlignmentLoss(tau=float(topology.get("cosine_tau", 0.45)),\n            lambda_anisotropy=float(topology.get("lambda_anisotropy", 0.05)), tail_threshold=int(topology.get("tail_threshold", 20)))\n    else:\n        criterion = TopologicalPersistenceLoss(neighbors=int(topology.get("neighbors", 8)),\n            max_clouds=int(topology.get("max_clouds", 8)), max_dimension=0 if method == "ph_h0" else 1,\n            h1_weight=float(topology.get("h1_weight", 1)), lambda_anisotropy=float(topology.get("lambda_anisotropy", 0.05)),\n            tail_threshold=int(topology.get("tail_threshold", 20)))\n        criterion.sampling_generator.manual_seed(seed)\n    batch_size = int(training.get("train_batch_size", 512))\n    eval_batch_size = int(training.get("eval_batch_size", 1024))\n    topk = int(cfg["evaluation"].get("topk", 20))\n    if not 1 <= topk <= int(model.trainable_warm_mask.sum()):\n        raise ValueError("topk exceeds the warm candidate set")\n    loader = DataLoader(SessionPrefixDataset(train, model._experiment_max_seq_length), batch_size=batch_size,\n                        shuffle=True, generator=torch.Generator().manual_seed(seed))\n    optimizer = torch.optim.Adam(model.parameters(), lr=float(training.get("learning_rate", 0.001)),\n                                 weight_decay=float(training.get("weight_decay", 0)))\n    # PH pairing/assignment runs in FP32. No implicit mixed-precision changes\n    # between methods, especially for small persistence intervals.\n    cfg["training"]["use_amp"] = False\n    cfg["experiment"] = {"train_examples": len(train), "validation_examples": len(validation),\n        "train_limit": train_limit, "validation_limit": validation_limit,\n        "initial_checkpoint": str(init_checkpoint.resolve()) if init_checkpoint else None,\n        "evaluate_test": evaluate_test, "reference": "current_batch_target_embeddings_stop_gradient",\n        "filtration": "vietoris_rips_euclidean_chord", "coefficient_field": 2,\n        "precision": "float32", "device": str(device)}\n    save_yaml(cfg, output / "config.yaml")\n    pd.DataFrame({"source_row": train.index, "session_id": train.session_id.to_numpy()}).to_csv(output / "train_sample.csv.gz", index=False)\n    best, best_epoch, stale = -float("inf"), 0, 0\n    history = []\n    start = time.perf_counter()\n    logger.info("%s %s seed=%s train=%s validation=%s device=%s", model_name, method, seed, len(train), len(validation), device)\n    for epoch in range(1, epochs + 1):\n        epoch_start = time.perf_counter()\n        model.train()\n        totals = {"ce": 0.0, "regularizer": 0.0, "ph_h0": 0.0, "ph_h1": 0.0,\n                  "ph_clouds": 0.0, "ph_h1_bars": 0.0, "adjusted_items": 0.0,\n                  "input_active_items": 0.0, "role_conflict_items": 0.0,\n                  "gradient_adjustment_norm": 0.0, "relative_gradient_change": 0.0}\n        samples = 0\n        for batch_index, batch in enumerate(loader):\n            optimizer.zero_grad(set_to_none=True)\n            targets = batch["target"].to(device)\n            session = model.encode_session(batch["item_seq"].to(device), batch["item_seq_len"].to(device))\n            output_weight = model.item_embedding.weight.detach().requires_grad_() if method in ROLE_METHODS else model.item_embedding.weight\n            logits = (session @ output_weight.T).masked_fill(~model.trainable_warm_mask[None], -1e9)\n            ce = F.cross_entropy(logits, targets)\n            apply_regularizer = batch_index % interval == 0\n            if method == "baseline" or (method in ROLE_METHODS and method != "role_ph") or not apply_regularizer:\n                regularizer = session.sum() * 0\n            elif method == "dispersion":\n                regularizer = float(topology.get("lambda_anisotropy", 0.05)) * session_dispersion(session)\n            else:\n                regularizer = criterion(session, model.item_embedding(targets), frequencies, targets)\n            # Periodic regularization saves exact-PH pairing work on GPU. The\n            # multiplier preserves the nominal per-step strength in expectation\n            # over shuffled batches; it is not identical to every-step training.\n            loss = ce if method in ROLE_METHODS else ce + float(topology.get("lambda_topo", 0.15)) * interval * regularizer\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite training loss")\n            if method in ROLE_METHODS:\n                effective_method = "role_sum" if method == "role_ph" and not apply_regularizer else method\n                role_stats = backward_role_loss(ce, model.item_embedding.weight, output_weight,\n                    model.trainable_warm_mask, frequencies, effective_method,\n                    strength=float(topology.get("gradient_strength", .5)),\n                    tail_threshold=int(topology.get("tail_threshold", 20)), topology_loss=regularizer)\n                for key in ("adjusted_items", "input_active_items", "role_conflict_items", "gradient_adjustment_norm", "relative_gradient_change"):\n                    totals[key] += role_stats[key]\n            else:\n                loss.backward()\n            clip = float(training.get("grad_clip_norm", 5))\n            if clip > 0:\n                torch.nn.utils.clip_grad_norm_(model.parameters(), clip, error_if_nonfinite=True)\n            optimizer.step()\n            # Gradient masking alone does not protect against weight decay.\n            with torch.no_grad():\n                model.item_embedding.weight[frozen] = initial_frozen\n            count = len(targets)\n            totals["ce"] += float(ce.detach()) * count\n            totals["regularizer"] += float(regularizer.detach()) * count\n            if (method.startswith("ph_") or method == "role_ph") and apply_regularizer:\n                for key in ("ph_h0", "ph_h1"):\n                    totals[key] += criterion.last_stats[key] * count\n                for key in ("ph_clouds", "ph_h1_bars"):\n                    totals[key] += criterion.last_stats[key]\n            samples += count\n        val_dataset = SessionPrefixDataset(validation, model._experiment_max_seq_length)\n        predicted, _ = predict_topk_for_dataset(model, val_dataset, device, eval_batch_size, topk)\n        metrics = compute_stratified_ranking_metrics(predicted, val_dataset.targets, val_dataset.target_groups, k=topk)\n        score = float(metrics.iloc[0][f"mrr@{topk}"])\n        if score > best + float(cfg["early_stopping"].get("min_delta", 0.0001)):\n            best, best_epoch, stale = score, epoch, 0\n            torch.save({"model_state_dict": model.state_dict(), "config": cfg, "epoch": epoch,\n                        "best_validation_mrr": best}, output / "checkpoint.pth")\n        else:\n            stale += 1\n        row = {"epoch": epoch, **{k: v / samples if k in ("ce", "regularizer", "ph_h0", "ph_h1") else v for k, v in totals.items()},\n               "validation_mrr": score, "validation_recall": float(metrics.iloc[0][f"recall@{topk}"]),\n               "seconds": time.perf_counter() - epoch_start}\n        history.append(row)\n        pd.DataFrame(history).to_csv(output / "training_history.csv", index=False)\n        logger.info("epoch=%s CE=%.5f PH0=%.5f PH1=%.5f val_MRR=%.5f time=%.1fs", epoch, row["ce"], row["ph_h0"], row["ph_h1"], score, row["seconds"])\n        if stale >= int(cfg["early_stopping"].get("patience", 5)):\n            break\n    model.load_state_dict(torch.load(output / "checkpoint.pth", map_location=device, weights_only=False)["model_state_dict"])\n    drift = float((model.item_embedding.weight.detach()[frozen] - initial_frozen).abs().max())\n    if drift != 0:\n        raise AssertionError(f"Frozen embedding drift: {drift}")\n    result = {"model": model_name, "method": method, "seed": seed, "best_epoch": best_epoch,\n              "frozen_embedding_max_abs_drift": drift, "seconds": time.perf_counter() - start,\n              "status": "exploratory_pilot" if train_limit or validation_limit or init_checkpoint else "full_training",\n              "experiment": cfg["experiment"], "output_dir": str(output.resolve())}\n    result["validation"] = _evaluate(model, validation, mapping, device, eval_batch_size, topk, output, "validation")\n    # Load test data only after selecting the checkpoint and only when requested.\n    if evaluate_test:\n        test = pd.read_parquet(processed_dir / cfg["data_files"]["test_examples"])\n        result["test"] = _evaluate(model, test, mapping, device, eval_batch_size, topk, output, "test")\n    np.save(output / "item_embeddings.npy", model.item_embedding.weight.detach().cpu().numpy()[1:])\n    mapping.to_csv(output / "item_embedding_mapping.csv", index=False)\n    result["training_and_selection_seconds"] = result["seconds"]\n    result["seconds"] = time.perf_counter() - start\n    save_json(result, output / "metrics.json")\n    return result\n', 'src/btl_k03/topology/topo_eval.py': 'import numpy as np\nimport torch\nfrom scipy.spatial.distance import pdist, squareform\n\ndef compute_anisotropy_spectrum(item_embeddings: np.ndarray) -> dict:\n    """\n    Đo độ mất đẳng hướng (Anisotropy) của không gian embedding.\n    Nếu singular values phân rã quá nhanh -> Không gian bị bẹp/nón (Anisotropic).\n    Nếu singular values dàn đều -> Không gian đẳng hướng hoàn hảo (Isotropic).\n    """\n    # Trừ vector trung bình (Centering)\n    mean_vec = np.mean(item_embeddings, axis=0, keepdims=True)\n    centered = item_embeddings - mean_vec\n    \n    # SVD\n    _, s, _ = np.linalg.svd(centered, full_matrices=False)\n    variance_explained = (s ** 2) / np.sum(s ** 2)\n    \n    # Anisotropy ratio = singular value lớn nhất / tổng singular values\n    top1_ratio = variance_explained[0]\n    top5_ratio = np.sum(variance_explained[:5])\n    \n    return {\n        "top1_singular_ratio": float(top1_ratio),\n        "top5_singular_ratio": float(top5_ratio),\n        "singular_values": s[:10].tolist()\n    }\n\ndef evaluate_topological_properties(embeddings: np.ndarray, item_frequencies: np.ndarray, tail_threshold: int = 20) -> dict:\n    """\n    Đánh giá tô-pô lân cận giữa nhóm Head và nhóm Tail (Item A vs Item B).\n    """\n    tail_indices = np.where((item_frequencies < tail_threshold) & (item_frequencies > 0))[0]\n    head_indices = np.where(item_frequencies >= tail_threshold)[0]\n    \n    anisotropy_metrics = compute_anisotropy_spectrum(embeddings)\n    \n    # Tính L2 norm phân bố giữa Head và Tail\n    tail_norms = np.linalg.norm(embeddings[tail_indices], axis=1) if len(tail_indices) > 0 else np.array([0])\n    head_norms = np.linalg.norm(embeddings[head_indices], axis=1) if len(head_indices) > 0 else np.array([0])\n    \n    return {\n        "anisotropy": anisotropy_metrics,\n        "tail_mean_norm": float(np.mean(tail_norms)),\n        "head_mean_norm": float(np.mean(head_norms)),\n        "norm_ratio_head_to_tail": float(np.mean(head_norms) / (np.mean(tail_norms) + 1e-8)),\n    }\n', 'src/btl_k03/topology/role_gradients.py': '"""Selective adjustment of the two CE paths through a tied embedding table.\n\nPH supplies a local gradient direction, not a claim of semantic correctness.\nThe projection acts BEFORE Adam; it does not guarantee norm preservation or\nloss descent after Adam preconditioning, momentum and clipping.\n"""\nfrom __future__ import annotations\n\nimport torch\n\nROLE_METHODS = ("role_sum", "role_input_scale", "role_projection", "role_tangent", "role_ph")\n\n\ndef combine_role_gradients(weight, grad_input, grad_output, warm_mask, frequencies,\n                           method, strength=.5, tail_threshold=20, topology_gradient=None):\n    if method not in ROLE_METHODS or not 0 <= strength <= 1:\n        raise ValueError("Invalid role-gradient method or projection strength")\n    gi, go = grad_input.detach(), grad_output.detach()\n    active = gi.norm(dim=1) > 1e-12\n    eligible = warm_mask & (frequencies > 0) & (frequencies < tail_threshold) & active\n    conflict = (gi * go).sum(-1) < 0\n    selected = torch.zeros_like(eligible)\n    if method == "role_sum":\n        adjusted = gi + go\n    elif method == "role_input_scale":\n        adjusted = (1 + strength) * gi + go\n        selected = active & warm_mask\n    else:\n        unit = torch.nn.functional.normalize(weight.detach(), dim=-1)\n        guide = gi if method != "role_ph" else topology_gradient\n        if guide is None:\n            raise ValueError("role_ph needs the live PH objective gradient")\n        guide = guide.detach()\n        output = go\n        if method in ("role_tangent", "role_ph"):\n            guide = guide - (guide * unit).sum(-1, keepdim=True) * unit\n            output = go - (go * unit).sum(-1, keepdim=True) * unit\n        dot = (output * guide).sum(-1, keepdim=True)\n        square_norm = guide.square().sum(-1, keepdim=True)\n        selected = eligible & conflict & (dot[:, 0] < 0) & (square_norm[:, 0] > 1e-24)\n        projection = dot.clamp_max(0) / square_norm.clamp_min(1e-24) * guide\n        adjusted = gi + go - strength * projection * selected[:, None]\n    adjusted = adjusted.masked_fill(~warm_mask[:, None], 0)\n    change = (adjusted - (gi + go)).norm()\n    stats = {"input_active_items": int(active.sum()), "role_conflict_items": int((conflict & active & warm_mask).sum()),\n             "adjusted_items": int(selected.sum()), "gradient_adjustment_norm": float(change),\n             "relative_gradient_change": float(change / (gi + go).norm().clamp_min(1e-12))}\n    return adjusted, stats\n\n\ndef backward_role_loss(ce_loss, weight, output_leaf, warm_mask, frequencies,\n                       method, strength=.5, tail_threshold=20, topology_loss=None):\n    """One CE forward/dropout realization; separate output leaf and input E.\n\n    CE backward gives every encoder parameter its ordinary CE gradient. Only E\n    is replaced. autograd.grad(PH, E) does not add a PH gradient to the encoder.\n    Output-leaf gradients and input-E gradients sum exactly to tied CE gradient.\n    """\n    topology_gradient = None\n    if method == "role_ph":\n        if topology_loss is None:\n            raise ValueError("Missing PH loss")\n        topology_gradient = torch.autograd.grad(topology_loss, weight, retain_graph=True)[0]\n    ce_loss.backward()\n    adjusted, stats = combine_role_gradients(weight, weight.grad, output_leaf.grad,\n        warm_mask, frequencies, method, strength, tail_threshold, topology_gradient)\n    weight.grad = adjusted\n    return stats\n', 'scripts/train.py': '"""Run controlled, measured PH experiments; never reuse legacy result paths."""\nfrom pathlib import Path\nimport argparse\nfrom datetime import datetime\nimport sys\n\nimport pandas as pd\nimport torch\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\n\nfrom btl_k03.topology.topo_trainer import METHODS, train_and_evaluate_topology_seed\nfrom btl_k03.utils import load_experiment_config, load_yaml\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--models", nargs="+", choices=["gru4rec", "narm", "sasrec"], default=["gru4rec"])\n    parser.add_argument("--methods", nargs="+", choices=METHODS, default=["baseline", "cosine", "ph_h0", "ph_h0_h1"])\n    parser.add_argument("--seeds", nargs="+", type=int, default=[2024, 2025])\n    parser.add_argument("--epochs", type=int, default=2)\n    parser.add_argument("--train-limit", type=int, default=2048)\n    parser.add_argument("--validation-limit", type=int, default=2048)\n    parser.add_argument("--full-data", action="store_true", help="Ignore sample limits; set epochs explicitly for full training")\n    parser.add_argument("--evaluate-test", action="store_true", help="Evaluate test AFTER validation checkpoint selection")\n    parser.add_argument("--batch-size", type=int, default=128)\n    parser.add_argument("--learning-rate", type=float)\n    parser.add_argument("--threads", type=int, default=2)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--config", type=Path, default=ROOT / "configs/topology/role_gradient_pilot.yaml")\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--init-existing", action="store_true", help="Fine-tune every method from the same model/seed baseline checkpoint; exploratory only")\n    args = parser.parse_args()\n    if args.threads < 1 or args.batch_size < 1:\n        parser.error("threads and batch-size must be positive")\n    torch.set_num_threads(args.threads)\n    root = args.output or ROOT / "results/persistent_homology" / datetime.now().strftime("%Y%m%d_%H%M%S")\n    root.mkdir(parents=True, exist_ok=False)\n    rows = []\n    for name in args.models:\n        for seed in args.seeds:\n            for method in args.methods:\n                config = load_experiment_config(name)\n                config["topology_params"] = load_yaml(args.config)["topology_params"]\n                config["training"].update(train_batch_size=args.batch_size, eval_batch_size=args.batch_size)\n                if args.learning_rate is not None:\n                    if args.learning_rate <= 0:\n                        parser.error("learning-rate must be positive")\n                    config["training"]["learning_rate"] = args.learning_rate\n                result = train_and_evaluate_topology_seed(seed, config=config, model_name=name, method=method,\n                    max_epochs_override=args.epochs, device_str=args.device,\n                    train_limit=None if args.full_data else args.train_limit,\n                    validation_limit=None if args.full_data else args.validation_limit,\n                    evaluate_test=args.evaluate_test, output_dir=root / name / method / f"seed_{seed}",\n                    init_checkpoint=ROOT / "runs" / name / f"seed_{seed}" / "checkpoint.pth" if args.init_existing else None)\n                for split in ("validation", "test"):\n                    if split not in result:\n                        continue\n                    for group in result[split]["ranking_by_group"]:\n                        rows.append({"model": name, "method": method, "seed": seed, "split": split,\n                            "status": result["status"], "best_epoch": result["best_epoch"],\n                            "seconds": result["seconds"], "frozen_drift": result["frozen_embedding_max_abs_drift"],\n                            **group, **result[split]["popularity_bias"]})\n                pd.DataFrame(rows).to_csv(root / "comparison.csv", index=False)\n    print(f"Measured results: {root.resolve()}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/analyze.py': '"""Extend the original frequency/norm/category analysis with actual H0/H1.\n\nOnly warm item embeddings are used. Geometry is observational; neither long\nbars nor incremental predictive R2 establish a causal mechanism or relevance.\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.stats import spearmanr\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import mean_absolute_error, r2_score\nfrom sklearn.model_selection import KFold\nfrom sklearn.neighbors import NearestNeighbors\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\nfrom btl_k03.topology.persistent_homology import finite_diagram, vietoris_rips_persistence\nfrom btl_k03.artifacts import load_item_embeddings\n\n\ndef analyze(run, output, per_group=200, neighbors=12, seed=2024):\n    output.mkdir(parents=True, exist_ok=False)\n    mapping = pd.read_csv(run / "item_embedding_mapping.csv").sort_values("internal_item_id")\n    embedding = load_item_embeddings(run)\n    if len(mapping) != len(embedding) or not np.array_equal(mapping.internal_item_id, np.arange(1, len(mapping) + 1)):\n        raise ValueError("Embedding row/internal ID mapping is inconsistent")\n    warm = mapping.loc[mapping.is_trainable_warm].copy()\n    unit = normalize(embedding[warm.internal_item_id.to_numpy() - 1])\n    sample = pd.concat([part.sample(min(per_group, len(part)), random_state=seed)\n                        for group, part in warm.groupby("popularity_group") if group in ("popular", "medium", "rare")])\n    lookup = pd.Series(np.arange(len(warm)), index=warm.internal_item_id)\n    positions = lookup.loc[sample.internal_item_id].to_numpy()\n    k = min(neighbors, len(warm))\n    nn = NearestNeighbors(n_neighbors=k, metric="euclidean", algorithm="brute", n_jobs=2).fit(unit)\n    distances, indices = nn.kneighbors(unit[positions])\n    rows = []\n    for (_, item), point_indices, nn_dist in zip(sample.iterrows(), indices, distances):\n        diagrams = vietoris_rips_persistence(torch.tensor(unit[point_indices], dtype=torch.float32))\n        d0, d1 = finite_diagram(diagrams[0]).numpy(), finite_diagram(diagrams[1]).numpy()\n        life0, life1 = d0[:, 1] - d0[:, 0], d1[:, 1] - d1[:, 0]\n        rows.append({"internal_item_id": int(item.internal_item_id), "item_id": int(item.item_id),\n            "category_id": str(item.category_id), "popularity_group": item.popularity_group,\n            "train_frequency": int(item.train_frequency), "log_frequency": float(np.log1p(item.train_frequency)),\n            "embedding_norm": float(np.linalg.norm(embedding[int(item.internal_item_id) - 1])),\n            "mean_neighbor_distance": float(nn_dist[1:].mean()),\n            "h0_mean_lifetime": float(life0.mean()) if len(life0) else 0,\n            "h0_max_lifetime": float(life0.max()) if len(life0) else 0,\n            "h1_bar_count": len(life1), "h1_total_lifetime": float(life1.sum()),\n            "h1_max_lifetime": float(life1.max()) if len(life1) else 0})\n    frame = pd.DataFrame(rows)\n    topo = ["h0_mean_lifetime", "h0_max_lifetime", "h1_bar_count", "h1_total_lifetime", "h1_max_lifetime"]\n    relationships = []\n    for measure in topo:\n        for variable in ("train_frequency", "embedding_norm", "mean_neighbor_distance"):\n            if frame[measure].nunique() > 1 and frame[variable].nunique() > 1:\n                rho, p = spearmanr(frame[measure], frame[variable])\n                relationships.append({"measure": measure, "variable": variable, "rho": float(rho), "p_value": float(p)})\n    report = {"scope": "original_embedding_geometry_study", "warm_only": True, "filtration": "Rips_chord_F2_H0_H1",\n              "n_sampled_items": len(frame), "neighbors": k,\n              "interpretation": "Observational diagnostics; PH stability is not relevance, quality or causality"}\n    predictions = run / "validation_top20.parquet"\n    if predictions.exists():\n        counts = pd.read_parquet(predictions, columns=["internal_item_id"]).internal_item_id.value_counts()\n        frame["validation_rec_count"] = frame.internal_item_id.map(counts).fillna(0)\n        y = np.log1p(frame.validation_rec_count.to_numpy())\n        base = ["log_frequency", "embedding_norm", "mean_neighbor_distance"]\n        # Same item folds for nested feature comparisons. This tests incremental\n        # prediction of exposure on validation, never causal attribution.\n        comparison = []\n        folds = list(KFold(n_splits=5, shuffle=True, random_state=seed).split(frame))\n        for name, numeric in (("frequency_norm_density_category", base), ("plus_persistent_homology", base + topo)):\n            predicted = np.zeros(len(frame))\n            for train, test in folds:\n                features = ColumnTransformer([("numeric", StandardScaler(), numeric),\n                    ("category", OneHotEncoder(handle_unknown="ignore"), ["category_id"])])\n                model = make_pipeline(features, Ridge(alpha=10, solver="lsqr"))\n                model.fit(frame.iloc[train], y[train])\n                predicted[test] = model.predict(frame.iloc[test])\n            comparison.append({"features": name, "out_of_fold_r2": float(r2_score(y, predicted)),\n                               "out_of_fold_mae": float(mean_absolute_error(y, predicted))})\n        report["validation_exposure_prediction"] = comparison\n        report["r2_increment_from_ph"] = comparison[1]["out_of_fold_r2"] - comparison[0]["out_of_fold_r2"]\n        report["caveat"] = "Item folds share an embedding neighborhood graph; exploratory prediction, not independent causal evidence"\n    frame.to_csv(output / "item_ph_geometry.csv", index=False)\n    frame.groupby("popularity_group")[topo + ["embedding_norm", "mean_neighbor_distance"]].mean().to_csv(output / "group_geometry.csv")\n    pd.DataFrame(relationships).to_csv(output / "geometry_correlations.csv", index=False)\n    (output / "interpretation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--run", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--per-group", type=int, default=200)\n    parser.add_argument("--neighbors", type=int, default=12)\n    args = parser.parse_args()\n    if args.per_group < 2 or not 3 <= args.neighbors <= 32:\n        parser.error("per-group must be >= 2 and neighbors 3..32")\n    torch.set_num_threads(2)\n    from threadpoolctl import threadpool_limits\n    with threadpool_limits(limits=2):\n        analyze(args.run, args.output, args.per_group, args.neighbors)\n', 'scripts/report.py': '"""Report measured role-gradient effects against CE, simple scaling and epoch 0.\n\nNo test labels are read. Compare identical validation queries, cluster-resample\nsessions, and explicitly record small effects and absent gains.\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport sys\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\nfrom btl_k03.models import build_model_from_config\nfrom btl_k03.artifacts import load_item_embeddings\nfrom btl_k03.topology.topo_trainer import _evaluate, _sample\nfrom btl_k03.utils import load_item_mapping, load_yaml\nfrom btl_k03.statistics import clustered_delta\n\n\ndef report(roots, output, iterations=1000):\n    output.mkdir(parents=True, exist_ok=False)\n    comparisons = []\n    files = {}\n    for root in roots:\n        table = pd.read_csv(root / "comparison.csv")\n        comparisons.append(table)\n        for model, method, seed in table[["model", "method", "seed"]].drop_duplicates().itertuples(index=False, name=None):\n            key = (model, method, seed)\n            if key in files:\n                raise ValueError(f"Duplicate experiment: {key}")\n            files[key] = root / model / method / f"seed_{seed}"\n    combined = pd.concat(comparisons, ignore_index=True)\n    combined.to_csv(output / "all_measured_results.csv", index=False)\n    mapping = load_item_mapping()\n    processed = ROOT / "data/processed/diginetica"\n    validation = pd.read_parquet(processed / "validation_examples.parquet")\n    models = combined[["model", "seed"]].drop_duplicates()\n    differences, rows, diagnostics = [], [], []\n    rng = np.random.default_rng(20261121)\n    for model, seed in models.itertuples(index=False, name=None):\n        baseline = files[(model, "baseline", seed)]\n        config = load_yaml(baseline / "config.yaml")\n        reference = None\n        if config["experiment"].get("initial_checkpoint"):\n            checkpoint_path = Path(config["experiment"]["initial_checkpoint"])\n            checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n            original = build_model_from_config(checkpoint["config"], mapping, torch.device("cpu"))\n            original.load_state_dict(checkpoint["model_state_dict"])\n            original._experiment_max_seq_length = int(config.get("max_seq_length", 50))\n            reference = output / "untouched_checkpoint" / model / f"seed_{seed}"\n            reference.mkdir(parents=True)\n            sample = _sample(validation, config["experiment"]["validation_limit"], seed + 1)\n            initial_metrics = _evaluate(original, sample, mapping, torch.device("cpu"), 128, 20, reference, "validation")\n            (reference / "metrics.json").write_text(json.dumps(initial_metrics, indent=2), encoding="utf-8")\n        available = {method: path for (name, method, item_seed), path in files.items() if name == model and item_seed == seed}\n        baseline_embeddings = load_item_embeddings(baseline)\n        queries = {method: pd.read_csv(path / "validation_per_query.csv.gz") for method, path in available.items()}\n        if reference is not None:\n            queries["untouched"] = pd.read_csv(reference / "validation_per_query.csv.gz")\n        controls = [method for method in ("baseline", "role_input_scale", "role_tangent", "untouched") if method in queries]\n        for method, query in queries.items():\n            for group in ("all_warm", "rare"):\n                subset = query if group == "all_warm" else query[query.group.eq(group)]\n                base = queries["baseline"] if group == "all_warm" else queries["baseline"][queries["baseline"].group.eq(group)]\n                # Explicit merge guards against positional comparisons of\n                # different samples and retains paired hit transitions.\n                aligned = base.merge(subset, on=["source_row", "session_id", "target", "group"],\n                    suffixes=("_base", "_new"), validate="one_to_one")\n                if len(aligned) != len(base) or len(aligned) != len(subset):\n                    raise ValueError("Validation query samples differ")\n                rows.append({"model": model, "seed": seed, "method": method, "group": group,\n                    "n_queries": len(subset), "hits": int(subset.hit.sum()), "recall": float(subset.hit.mean()),\n                    "mrr": float(subset.reciprocal_rank.mean()), "new_hits_vs_ce": int(((aligned.hit_base == 0) & (aligned.hit_new == 1)).sum()),\n                    "lost_hits_vs_ce": int(((aligned.hit_base == 1) & (aligned.hit_new == 0)).sum())})\n            if method in available:\n                history = pd.read_csv(available[method] / "training_history.csv")\n                diagnostics.append({"model": model, "seed": seed, "method": method,\n                    "adjusted_item_batch_events": float(history.get("adjusted_items", pd.Series([0])).sum()),\n                    "input_active_item_batch_events": float(history.get("input_active_items", pd.Series([0])).sum()),\n                    "ph_h0_mean": float(history.ph_h0.mean()), "ph_h1_mean": float(history.ph_h1.mean()),\n                    "rare_mean_embedding_distance_from_ce": float(np.linalg.norm(\n                        (load_item_embeddings(available[method]) - baseline_embeddings)[mapping.popularity_group.eq("rare").to_numpy()], axis=1).mean())})\n            if method not in ("role_ph", "ph_h0_h1", "role_input_scale"):\n                continue\n            for control in controls:\n                if method == control:\n                    continue\n                for group, metric in (("rare", "hit"), ("all_warm", "hit"), ("rare", "reciprocal_rank")):\n                    delta = clustered_delta(queries[control], query, metric, group, iterations, rng)\n                    differences.append({"model": model, "seed": seed, "method": method, "control": control,\n                                        "group": group, "metric": metric, **delta})\n        print(f"Evaluated paired controls: {model} seed={seed}", flush=True)\n    pd.DataFrame(rows).to_csv(output / "ranking_effects.csv", index=False)\n    pd.DataFrame(diagnostics).to_csv(output / "mechanism_diagnostics.csv", index=False)\n    delta_frame = pd.DataFrame(differences)\n    delta_frame.to_csv(output / "paired_session_bootstrap.csv", index=False)\n    decisive = delta_frame[delta_frame.method.eq("role_ph") & delta_frame.control.isin(["baseline", "role_input_scale", "untouched"])\n                            & delta_frame.group.eq("rare") & delta_frame.metric.eq("hit")]\n    passes = bool(len(decisive)) and decisive.ci_low.gt(0).all()\n    verdict = {"method": "role_ph", "status": "exploratory_positive_signal" if passes else "added_value_not_demonstrated",\n        "criterion": "positive lower session-bootstrap rare Recall bound against CE, input scaling and untouched checkpoint for every measured model/seed",\n        "models": models.to_dict(orient="records"), "test_used": False,\n        "scope": "Small-sample fine-tuning from validation-selected checkpoints; not a full-training or confirmatory result",\n        "inference": "Architecture and inference parameter count unchanged; inference latency not measured",\n        "novelty": "Not established; code implementation is not proof of a new scientific contribution"}\n    (output / "verdict.json").write_text(json.dumps(verdict, indent=2), encoding="utf-8")\n    plot = delta_frame[delta_frame.control.eq("baseline") & delta_frame.group.eq("rare") & delta_frame.metric.eq("hit")]\n    if len(plot):\n        figure, axis = plt.subplots(figsize=(12, max(4, len(plot) * .25)))\n        for index, row in enumerate(plot.itertuples(index=False)):\n            axis.errorbar(row.delta * 100, index, xerr=[[max(0, row.delta - row.ci_low) * 100], [max(0, row.ci_high - row.delta) * 100]],\n                fmt="o", capsize=3, color="tab:blue" if row.method == "role_ph" else "tab:gray")\n        axis.set_yticks(range(len(plot)), [f"{r.model} / {r.seed} / {r.method}" for r in plot.itertuples(index=False)])\n        axis.axvline(0, color="black", linestyle="--", linewidth=1)\n        axis.set(xlabel="Rare Recall difference vs CE (percentage points), paired session bootstrap 95% CI",\n                 title="Measured exploratory effects; intervals do not include training uncertainty")\n        figure.tight_layout()\n        figure.savefig(output / "rare_recall_effects.png", dpi=180)\n        plt.close(figure)\n    print(json.dumps(verdict, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--roots", type=Path, nargs="+", required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--iterations", type=int, default=1000)\n    args = parser.parse_args()\n    torch.set_num_threads(2)\n    from threadpoolctl import threadpool_limits\n    with threadpool_limits(limits=2):\n        report(args.roots, args.output, args.iterations)\n', 'scripts/calibrate.py': '"""Strong, explicitly NON-NOVEL popularity-adjusted scoring control.\n\nChoose lambda using validation ONLY under an overall Recall guardrail, freeze\nit to JSON, then evaluate baseline and chosen scoring on test once. This control\nmeasures what the more complex PH proposal must beat on the original SBR task.\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\nfrom btl_k03.models import SessionPrefixDataset, build_model_from_config\nfrom btl_k03.popularity_metrics import compute_stratified_ranking_metrics\nfrom btl_k03.utils import load_item_mapping\nfrom btl_k03.statistics import clustered_delta\n\n\n@torch.no_grad()\ndef predictions_for_grid(model, frame, coefficients, penalty, batch_size=128):\n    model.eval()\n    dataset = SessionPrefixDataset(frame, 50)\n    predictions = {coefficient: np.empty((len(frame), 20), dtype=np.int64) for coefficient in coefficients}\n    offset = 0\n    for batch in DataLoader(dataset, batch_size=batch_size, shuffle=False):\n        logits = model.full_sort_logits(batch["item_seq"], batch["item_seq_len"])\n        count = len(batch["target"])\n        for coefficient in coefficients:\n            adjusted = logits - coefficient * penalty[None]\n            predictions[coefficient][offset:offset + count] = adjusted.topk(20, dim=-1).indices.numpy()\n        offset += count\n    return predictions, dataset\n\n\ndef per_query(frame, prediction):\n    hits = prediction == frame.target_internal_item_id.to_numpy()[:, None]\n    hit = hits.any(1)\n    rank = hits.argmax(1) + 1\n    return pd.DataFrame({"source_row": frame.index, "session_id": frame.session_id.to_numpy(),\n        "target": frame.target_internal_item_id.to_numpy(), "group": frame.target_popularity_group.to_numpy(),\n        "hit": hit.astype(int), "reciprocal_rank": np.where(hit, 1 / rank, 0), "ndcg": np.where(hit, 1 / np.log2(rank + 1), 0)})\n\n\ndef choose_coefficient(rows, max_overall_drop):\n    baseline = next(row for row in rows if row["coefficient"] == 0)\n    eligible = [row for row in rows if row["overall_recall"] >= baseline["overall_recall"] - max_overall_drop]\n    return max(eligible, key=lambda row: (row["rare_recall"], row["rare_mrr"], row["overall_mrr"], -row["coefficient"]))["coefficient"]\n\n\ndef run(output, models, seeds, max_drop=.005, checkpoint_root=None):\n    output.mkdir(parents=True, exist_ok=False)\n    grid = [0., .05, .1, .2, .4]\n    protocol = {"score": "original_dot_product - lambda * log(1 + train_frequency)", "novel_method": False,\n                "grid": grid, "selection": "max rare Recall then rare MRR within validation overall Recall guardrail",\n                "max_validation_overall_drop": max_drop, "models": models, "seeds": seeds,\n                "test_policy": "read outcomes only after saving validation-selected lambda; no retuning",\n                "limitation": "Diginetica test has been viewed in prior repository analyses; this is an offline benchmark, not an independent new dataset or live business outcome"}\n    (output / "protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")\n    mapping = load_item_mapping()\n    processed = ROOT / "data/processed/diginetica"\n    validation = pd.read_parquet(processed / "validation_examples.parquet")\n    results, evidence = [], []\n    for model_name in models:\n        for seed in seeds:\n            directory = output / model_name / f"seed_{seed}"\n            directory.mkdir(parents=True)\n            checkpoint_file = (checkpoint_root / model_name / "baseline" / f"seed_{seed}/checkpoint.pth"\n                               if checkpoint_root else ROOT / "runs" / model_name / f"seed_{seed}/checkpoint.pth")\n            checkpoint = torch.load(checkpoint_file, map_location="cpu", weights_only=False)\n            model = build_model_from_config(checkpoint["config"], mapping, torch.device("cpu"))\n            model.load_state_dict(checkpoint["model_state_dict"])\n            frequency = torch.zeros(model.num_items_with_pad)\n            frequency[torch.tensor(mapping.internal_item_id.to_numpy())] = torch.tensor(mapping.train_frequency.to_numpy(), dtype=torch.float32)\n            penalty = torch.log1p(frequency)\n            predictions, dataset = predictions_for_grid(model, validation, grid, penalty)\n            validation_rows = []\n            for coefficient in grid:\n                metrics = compute_stratified_ranking_metrics(predictions[coefficient], dataset.targets, dataset.target_groups, k=20).set_index("target_group")\n                validation_rows.append({"coefficient": coefficient, "overall_recall": float(metrics.loc["all_warm", "recall@20"]),\n                    "overall_mrr": float(metrics.loc["all_warm", "mrr@20"]), "rare_recall": float(metrics.loc["rare", "recall@20"]),\n                    "rare_mrr": float(metrics.loc["rare", "mrr@20"])})\n            chosen = choose_coefficient(validation_rows, max_drop)\n            pd.DataFrame(validation_rows).to_csv(directory / "validation_search.csv", index=False)\n            (directory / "locked_policy.json").write_text(json.dumps({"coefficient": chosen, "selected_on": "validation", "test_seen_during_selection": False}, indent=2), encoding="utf-8")\n            del predictions\n            # Chosen coefficient is fixed above. No tuning follows this point.\n            test = pd.read_parquet(processed / "test_examples.parquet")\n            test_predictions, dataset = predictions_for_grid(model, test, sorted(set([0., chosen])), penalty)\n            query_frames = {}\n            for label, coefficient in (("baseline", 0.), ("popularity_calibration", chosen)):\n                prediction = test_predictions[coefficient]\n                if not model.trainable_warm_mask[torch.tensor(prediction)].all():\n                    raise AssertionError("Cold/padding candidate leaked")\n                stats = compute_stratified_ranking_metrics(prediction, dataset.targets, dataset.target_groups, k=20)\n                for row in stats.to_dict(orient="records"):\n                    results.append({"model": model_name, "seed": seed, "method": label, "coefficient": coefficient, **row})\n                query_frames[label] = per_query(test, prediction)\n                query_frames[label].to_csv(directory / f"test_{label}_per_query.csv.gz", index=False)\n            rng = np.random.default_rng(seed)\n            for group, metric in (("rare", "hit"), ("all_warm", "hit"), ("rare", "reciprocal_rank")):\n                estimate = clustered_delta(query_frames["baseline"], query_frames["popularity_calibration"], metric, group, 2000, rng)\n                evidence.append({"model": model_name, "seed": seed, "coefficient": chosen, "group": group, "metric": metric, **estimate})\n            pd.DataFrame(results).to_csv(output / "test_comparison.csv", index=False)\n            pd.DataFrame(evidence).to_csv(output / "paired_test_bootstrap.csv", index=False)\n            print(f"{model_name} seed={seed} locked_lambda={chosen}; test evaluation complete", flush=True)\n    summary = pd.DataFrame(evidence)\n    rare = summary[summary.group.eq("rare") & summary.metric.eq("hit")]\n    overall = summary[summary.group.eq("all_warm") & summary.metric.eq("hit")]\n    verdict = {"novel_method": False, "rare_positive_ci_cases": int(rare.ci_low.gt(0).sum()),\n        "measured_cases": len(rare), "overall_within_drop_in_point_estimate_cases": int(overall.delta.ge(-max_drop).sum()),\n        "interpretation": "Measured control, not proof of PH benefit or novelty; no guaranteed online/revenue lift"}\n    (output / "verdict.json").write_text(json.dumps(verdict, indent=2), encoding="utf-8")\n    print(json.dumps(verdict, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--models", nargs="+", choices=["gru4rec", "narm", "sasrec"], default=["gru4rec", "narm", "sasrec"])\n    parser.add_argument("--seeds", nargs="+", type=int, default=[2024, 2025])\n    parser.add_argument("--checkpoint-root", type=Path)\n    args = parser.parse_args()\n    torch.set_num_threads(2)\n    from threadpoolctl import threadpool_limits\n    with threadpool_limits(limits=2):\n        run(args.output, args.models, args.seeds, checkpoint_root=args.checkpoint_root)\n', 'tests/test_persistent_homology.py': 'from pathlib import Path\nimport sys\n\nimport numpy as np\nimport pandas as pd\nimport pytest\nimport torch\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\n\nfrom btl_k03.topology.persistent_homology import (\n    diagram_wasserstein_squared, finite_diagram, vietoris_rips_persistence,\n)\nfrom btl_k03.topology.persistence_loss import TopologicalPersistenceLoss\n\n\ndef test_square_has_real_cycle_killed_by_triangles():\n    points = torch.tensor([[0., 0.], [1., 0.], [1., 1.], [0., 1.]], dtype=torch.double)\n    diagrams = vietoris_rips_persistence(points)\n    torch.testing.assert_close(finite_diagram(diagrams[0])[:, 1], torch.ones(3, dtype=torch.double))\n    torch.testing.assert_close(diagrams[1], torch.tensor([[1., np.sqrt(2)]], dtype=torch.double))\n    assert torch.isinf(diagrams[0][:, 1]).sum() == 1\n\n\ndef test_h1_has_nonzero_live_edge_gradient():\n    points = torch.tensor([[0., 0.], [1.1, 0.], [1., 1.2], [0., 1.]], requires_grad=True)\n    diagram = vietoris_rips_persistence(points)[1]\n    assert len(diagram) == 1\n    (diagram[:, 1] - diagram[:, 0]).sum().backward()\n    assert torch.isfinite(points.grad).all() and points.grad.norm() > 0\n\n\n@pytest.mark.parametrize("seed", [7, 19, 2024])\ndef test_diagrams_match_independent_ripser(seed):\n    ripser = pytest.importorskip("ripser").ripser\n    points = np.random.default_rng(seed).normal(size=(10, 3))\n    actual = vietoris_rips_persistence(torch.tensor(points, dtype=torch.double))\n    expected = ripser(points, maxdim=1, coeff=2)["dgms"]\n    for dim in (0, 1):\n        got, wanted = actual[dim].detach().numpy(), expected[dim]\n        got = got[np.lexsort((got[:, 1], got[:, 0]))]\n        wanted = wanted[np.lexsort((wanted[:, 1], wanted[:, 0]))]\n        np.testing.assert_allclose(got, wanted, rtol=1e-5, atol=2e-6)\n\n\ndef test_permutation_invariance_and_scaling():\n    points = torch.tensor(np.random.default_rng(91).normal(size=(8, 3)), dtype=torch.double)\n    original = vietoris_rips_persistence(points)\n    moved = vietoris_rips_persistence(points[[4, 0, 7, 1, 5, 2, 6, 3]] * 3 + 10)\n    for dim in (0, 1):\n        a = finite_diagram(original[dim]) * 3\n        b = finite_diagram(moved[dim])\n        assert float(diagram_wasserstein_squared(a, b)) < 1e-20\n\n\ndef test_diagonal_assignment_and_empty_diagrams():\n    bar = torch.tensor([[1., 3.]], dtype=torch.double, requires_grad=True)\n    empty = torch.empty((0, 2), dtype=torch.double)\n    assert float(diagram_wasserstein_squared(bar, empty).detach()) == 2.0\n    far = torch.tensor([[20., 22.]], dtype=torch.double)\n    # Match both to the diagonal rather than to each other.\n    loss = diagram_wasserstein_squared(bar, far)\n    assert float(loss.detach()) == 4.0\n    loss.backward()\n    torch.testing.assert_close(bar.grad, torch.tensor([[-2., 2.]], dtype=torch.double))\n    assert float(diagram_wasserstein_squared(empty, empty)) == 0\n\n\ndef test_persistence_loss_gradient_matches_finite_differences():\n    points = torch.tensor(np.random.default_rng(30).normal(size=(6, 3)), dtype=torch.double, requires_grad=True)\n    reference = vietoris_rips_persistence(torch.tensor(np.random.default_rng(32).normal(size=(6, 3)), dtype=torch.double))\n\n    def objective(value):\n        diagrams = vietoris_rips_persistence(value)\n        return sum(diagram_wasserstein_squared(diagrams[d], reference[d]) for d in (0, 1))\n\n    assert torch.autograd.gradcheck(objective, (points,), eps=1e-6, atol=1e-4, rtol=1e-3)\n\n\ndef test_tail_cloud_loss_has_live_session_gradient_and_detached_reference():\n    torch.manual_seed(24)\n    session = torch.randn(8, 6, requires_grad=True)\n    target = torch.randn(8, 6, requires_grad=True)\n    criterion = TopologicalPersistenceLoss(neighbors=6, max_clouds=3, lambda_anisotropy=0)\n    loss = criterion(session, target, torch.tensor([0, 2, 4, 5, 3, 6, 10, 30, 40]), torch.arange(1, 9))\n    loss.backward()\n    assert criterion.last_stats["ph_clouds"] == 3\n    assert torch.isfinite(session.grad).all() and session.grad.norm() > 0\n    assert target.grad is None\n\n\ndef test_no_tail_and_duplicate_targets_are_safe():\n    session = torch.randn(4, 6, requires_grad=True)\n    criterion = TopologicalPersistenceLoss(lambda_anisotropy=0)\n    loss = criterion(session, torch.randn(4, 6), torch.tensor([0, 100]), torch.ones(4, dtype=torch.long))\n    loss.backward()\n    assert float(loss.detach()) == 0 and criterion.last_stats["ph_clouds"] == 0\n    loss = criterion(session, torch.randn(4, 6), torch.tensor([0, 2]), torch.ones(4, dtype=torch.long))\n    assert float(loss.detach()) == 0\n\n\ndef test_singleton_and_coincident_points():\n    for points in (torch.ones(1, 3), torch.ones(4, 3)):\n        diagrams = vietoris_rips_persistence(points)\n        assert len(finite_diagram(diagrams[0])) == 0\n        assert len(diagrams[1]) == 0\n        assert torch.isinf(diagrams[0][:, 1]).sum() == 1\n\n\n@pytest.mark.parametrize("method", ["baseline", "dispersion", "cosine", "ph_h0", "ph_h0_h1", "role_sum", "role_ph"])\n@pytest.mark.parametrize("model_name", ["gru4rec", "narm", "sasrec"])\ndef test_trainer_uses_current_api_and_preserves_frozen_rows(tmp_path, method, model_name):\n    from btl_k03.topology.topo_trainer import train_and_evaluate_topology_seed\n    from btl_k03.utils import load_experiment_config\n\n    data = tmp_path / "data"\n    data.mkdir()\n    mapping = pd.DataFrame({"internal_item_id": [1, 2, 3, 4], "item_id": [11, 22, 33, 44],\n        "category_id": [1, 1, 2, 2], "internal_category_id": [1, 1, 2, 2], "train_frequency": [4, 3, 2, 0],\n        "popularity_group": ["rare", "rare", "rare", "cold"], "is_controlled_cold": [False, False, False, True],\n        "is_trainable_warm": [True, True, True, False], "candidate_for_main_evaluation": [True, True, True, False]})\n    mapping.to_csv(data / "item_mapping.csv", index=False)\n    examples = pd.DataFrame({"session_id": [1, 2, 3, 4, 5, 6], "target_internal_item_id": [1, 2, 3, 1, 2, 3],\n        "target_item_id": [11, 22, 33, 11, 22, 33], "target_popularity_group": ["rare"] * 6,\n        "prefix_items": [[2, 3], [1], [2, 1], [3], [3, 1], [1, 2]]})\n    for split in ("train", "validation"):\n        examples.to_parquet(data / f"{split}_examples.parquet")\n    config = load_experiment_config(model_name)\n    config["max_seq_length"] = 4\n    config["model_params"].update(embedding_size=4, hidden_size=8, inner_size=8)\n    config["training"].update(epochs=1, train_batch_size=6, eval_batch_size=6, weight_decay=0.1)\n    config["evaluation"]["topk"] = 2\n    config["topology_params"] = {"neighbors": 3, "max_clouds": 2, "lambda_topo": 0.15}\n    result = train_and_evaluate_topology_seed(10, config, data, method=method, output_dir=tmp_path / "run", device_str="cpu")\n    assert result["frozen_embedding_max_abs_drift"] == 0\n    assert "test" not in result\n    assert (tmp_path / "run/validation_top20.parquet").exists()\n    with pytest.raises(FileExistsError):\n        train_and_evaluate_topology_seed(10, config, data, method=method, output_dir=tmp_path / "run")\n\n\ndef test_cloud_sampling_does_not_change_dropout_rng():\n    torch.manual_seed(101)\n    session = torch.randn(8, 6, requires_grad=True)\n    target = torch.randn(8, 6)\n    state = torch.random.get_rng_state().clone()\n    criterion = TopologicalPersistenceLoss(max_clouds=2)\n    criterion(session, target, torch.tensor([0] + [2] * 8), torch.arange(1, 9))\n    assert torch.equal(state, torch.random.get_rng_state())\n', 'tests/test_role_gradients.py': 'from pathlib import Path\nimport sys\n\nimport pandas as pd\nimport pytest\nimport torch\nfrom torch.nn import functional as F\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\nfrom btl_k03.models import build_model_from_config\nfrom btl_k03.utils import load_experiment_config\nfrom btl_k03.topology.role_gradients import ROLE_METHODS, backward_role_loss, combine_role_gradients\nfrom btl_k03.topology.persistence_loss import TopologicalPersistenceLoss\n\n\n@pytest.mark.parametrize("name", ["gru4rec", "narm", "sasrec"])\ndef test_split_sum_reproduces_every_parameter_gradient_with_dropout(name):\n    mapping = pd.DataFrame({"internal_item_id": [1, 2, 3, 4], "is_trainable_warm": [True, True, True, False]})\n    config = load_experiment_config(name)\n    config["max_seq_length"] = 4\n    config["model_params"].update(embedding_size=4, hidden_size=8, inner_size=8)\n    normal = build_model_from_config(config, mapping, torch.device("cpu"))\n    split = build_model_from_config(config, mapping, torch.device("cpu"))\n    split.load_state_dict(normal.state_dict())\n    sequence = torch.tensor([[1, 2, 0, 0], [2, 3, 1, 0], [1, 0, 0, 0]])\n    length = torch.tensor([2, 3, 1])\n    targets = torch.tensor([3, 2, 1])\n    torch.manual_seed(99)\n    F.cross_entropy(normal.full_sort_logits(sequence, length), targets).backward()\n    torch.manual_seed(99)\n    session = split.encode_session(sequence, length)\n    output = split.item_embedding.weight.detach().requires_grad_()\n    logits = (session @ output.T).masked_fill(~split.trainable_warm_mask[None], -1e9)\n    loss = F.cross_entropy(logits, targets)\n    backward_role_loss(loss, split.item_embedding.weight, output, split.trainable_warm_mask,\n                       torch.tensor([0, 2, 2, 2, 0]), "role_sum")\n    for (key, a), (_, b) in zip(normal.named_parameters(), split.named_parameters()):\n        torch.testing.assert_close(a.grad, b.grad, rtol=1e-5, atol=1e-7, msg=lambda msg: key + \': \' + msg)\n\n\n@pytest.mark.parametrize("method", ROLE_METHODS)\ndef test_zero_strength_is_baseline_and_frozen_rows_stay_zero(method):\n    torch.manual_seed(7)\n    weight, gi, go = torch.randn(3, 4), torch.randn(3, 4), torch.randn(3, 4)\n    warm = torch.tensor([False, True, True])\n    gi[~warm] = 0\n    go[~warm] = 0\n    actual, _ = combine_role_gradients(weight, gi, go, warm, torch.tensor([0, 2, 2]),\n        method, strength=0, topology_gradient=torch.randn(3, 4))\n    torch.testing.assert_close(actual, gi + go)\n\n\ndef test_tangent_projection_changes_direction_only_before_optimizer():\n    weight = torch.tensor([[1., 0.], [1., 0.]])\n    gi = torch.tensor([[.2, 1.], [0., 1.]])\n    go = torch.tensor([[.3, -2.], [0., -2.]])\n    warm = torch.tensor([True, False])\n    adjusted, stats = combine_role_gradients(weight, gi, go, warm, torch.tensor([2, 0]),\n        "role_ph", strength=.5, topology_gradient=torch.tensor([[0., 1.], [0., 1.]]))\n    torch.testing.assert_close(adjusted[0], torch.tensor([.5, 0.]))\n    assert stats["adjusted_items"] == 1 and adjusted[1].eq(0).all()\n\n\ndef test_ph_guide_is_live_and_encoder_gets_only_ce_gradient():\n    mapping = pd.DataFrame({"internal_item_id": [1, 2, 3, 4], "is_trainable_warm": [True, True, True, False]})\n    config = load_experiment_config("gru4rec")\n    config["model_params"].update(embedding_size=4, hidden_size=8, embedding_dropout=0, hidden_dropout=0)\n    model = build_model_from_config(config, mapping, torch.device("cpu"))\n    sequence = torch.tensor([[1, 2], [2, 3], [3, 1]])\n    targets = torch.tensor([3, 1, 2])\n    session = model.encode_session(sequence, torch.tensor([2, 2, 2]))\n    output = model.item_embedding.weight.detach().requires_grad_()\n    loss = F.cross_entropy((session @ output.T).masked_fill(~model.trainable_warm_mask[None], -1e9), targets)\n    expected = torch.autograd.grad(loss, model.dense.weight, retain_graph=True)[0]\n    frequencies = torch.tensor([0, 2, 2, 2, 0])\n    ph = TopologicalPersistenceLoss(neighbors=3, max_clouds=2, lambda_anisotropy=0)(\n        session, model.item_embedding(targets), frequencies, targets)\n    stats = backward_role_loss(loss, model.item_embedding.weight, output, model.trainable_warm_mask,\n                              frequencies, "role_ph", topology_loss=ph)\n    torch.testing.assert_close(model.dense.weight.grad, expected)\n    assert torch.isfinite(model.item_embedding.weight.grad).all()\n    assert stats["input_active_items"] > 0\n', 'configs/common.yaml': '# Cấu hình dùng chung cho cả ba mô hình SBR: GRU4Rec, NARM, SASRec\n# Đảm bảo tính công bằng tuyệt đối khi so sánh chất lượng gợi ý, popularity bias và hình học embedding\n\nproject_name: BTL-K03-Popularity-Bias-Embedding-Geometry\ndataset: diginetica\n\n# Hai random seed bắt buộc theo yêu cầu đề tài (Mục 11, 15)\nseeds:\n  - 2024\n  - 2025\n\n# Không gian biểu diễn dùng chung để so sánh công bằng L2 norm, cosine và t-SNE\nembedding_size: 64\nmax_seq_length: 50\npadding_idx: 0\n\n# Dữ liệu đầu vào từ bước 02_data_preprocessing.ipynb\ndata_files:\n  processed_dir: data/processed/diginetica\n  item_mapping: item_mapping.csv\n  category_mapping: category_mapping.csv\n  popularity_groups: popularity_groups.csv\n  cold_items: cold_items.csv\n  dataset_statistics: dataset_statistics.json\n  # Dữ liệu huấn luyện và đánh giá chính (All-prefix augmentation giúp đạt Recall@20 chuẩn benchmark ~45-52%)\n  train_examples: train_examples.parquet\n  validation_examples: validation_examples.parquet\n  test_examples: test_examples.parquet\n  # Dữ liệu đánh giá bổ sung theo phiên cuối (Last-item) và kiểm toán Cold-start 20% (Full events)\n  validation_last_item_examples: validation_last_item_examples.parquet\n  test_last_item_examples: test_last_item_examples.parquet\n  validation_events_full: validation_events_full.parquet\n  test_events_full: test_events_full.parquet\n\n# Giao thức Cold-start 20% (Option B) trong lúc huấn luyện và suy diễn\ncold_start_protocol:\n  keep_cold_in_embedding_table: true\n  freeze_cold_item_gradients: true\n  mask_cold_items_in_loss: true\n  mask_cold_items_in_ranking: true\n  mask_fill_value: -1.0e9\n  verify_zero_drift_after_training: true\n\n# Cấu hình tối ưu hóa và vòng lặp huấn luyện\ntraining:\n  loss_type: CE                # Full Cross-Entropy trên tập warm catalog (cho chất lượng cao nhất trên Diginetica)\n  optimizer: adam\n  learning_rate: 0.001\n  weight_decay: 0.0\n  grad_clip_norm: 5.0\n  train_batch_size: 512\n  eval_batch_size: 1024\n  epochs: 30\n  num_workers: 0\n  pin_memory: true\n  use_amp: true                # Tự động bật Mixed Precision (FP16) khi chạy trên GPU CUDA để tăng tốc gấp 2-3 lần\n\n# Cấu hình dừng sớm (Early Stopping) dựa trên tập Validation Warm\nearly_stopping:\n  monitor: mrr@20\n  mode: max\n  patience: 5\n  min_delta: 0.0001\n\n# Cấu hình đánh giá đầu ra và thiên lệch độ phổ biến\nevaluation:\n  topk: 20\n  candidate_set: warm_only     # 29,936 warm items (candidate_for_main_evaluation == True)\n  accuracy_metrics:\n    - recall@20\n    - mrr@20\n    - ndcg@20\n  popularity_bias_metrics:\n    - coverage@20\n    - gini@20\n    - arp@20\n  evaluate_last_item_split: true\n  evaluate_cold_start_impact: true\n\n# Cấu hình phân tích hình học không gian embedding (SV2)\nembedding_geometry:\n  pair_sample_size_per_group: 50000\n  bootstrap_iterations: 1000\n  confidence_level: 0.95\n  tsne:\n    sample_size: 8000\n    perplexity: 30\n    n_iter: 1000\n    init: pca\n    metric: euclidean\n    random_seeds:\n      - 2024\n      - 2025\n', 'configs/data.yaml': '\ndataset: diginetica\n\nraw_dir: data/raw/diginetica\ninterim_dir: data/interim/diginetica\nprocessed_dir: data/processed/diginetica\nrecbole_dir: data/processed/recbole/diginetica\n\nordering:\n  within_session: [timeframe, row_order]\n  split_session_by: last_event_date\n  keep_repeated_clicks: true\n  keep_midnight_crossing_sessions: true\n\nfiltering:\n  min_session_length: 2\n  min_item_frequency: 5\n  fixed_point: true\n\ntemporal_split:\n  strategy: last_n_calendar_days\n  validation_days: 7\n  test_days: 7\n\ncold_start:\n  protocol: controlled_item_holdout_option_b_assumption\n  ratio: 0.20\n  unit: item\n  stratify_by: category\n  seed: 2024\n  denominator: pre_holdout_filtered_train_catalog\n  keep_in_item_mapping: true\n  usage: visualization_only\n  exclude_from_main_warm_evaluation: true\n  require_one_warm_item_per_category: true\n  min_pre_holdout_items_per_category: 5\n  assumption_note: >-\n    Mục 1.6/Option B không có trong tài liệu được cung cấp. Giao thức này là\n    giả định minh bạch dựa trên mô tả cold-start 20% và embedding cold chưa\n    huấn luyện; nếu nhận được quy định gốc thì quy định gốc sẽ thay thế.\n\npopularity_groups:\n  source: final_train_frequency_only\n  method: tie_aware_item_frequency_quantiles\n  rare_quantile: 0.20\n  popular_quantile: 0.80\n\nexports:\n  parquet_compression: zstd\n  padding_internal_item_id: 0\n', 'configs/gru4rec.yaml': '# Cấu hình mô hình GRU4Rec (Hidasi et al., ICLR 2016)\n# Kiến trúc chuẩn RecBole: Item Embedding -> Dropout -> GRU -> Dense Projection -> Full-Catalog Dot Product\n\nmodel_name: gru4rec\narchitecture: GRU4Rec\n\nmodel_params:\n  embedding_size: 64\n  hidden_size: 128\n  num_layers: 1\n  embedding_dropout: 0.25\n  hidden_dropout: 0.20\n  init_std: 0.02\n\ntraining_overrides:\n  learning_rate: 0.001\n  weight_decay: 0.0\n  train_batch_size: 512\n  eval_batch_size: 1024\n  epochs: 30\n', 'configs/narm.yaml': '# Cấu hình mô hình NARM — Neural Attentive Session-based Recommendation (Li et al., CIKM 2017)\n# Kiến trúc chuẩn RecBole: Global GRU Encoder + Local Attention Encoder -> Bi-linear/Dense Projection -> Full-Catalog Dot Product\n\nmodel_name: narm\narchitecture: NARM\n\nmodel_params:\n  embedding_size: 64\n  hidden_size: 128\n  n_layers: 1\n  embedding_dropout: 0.25\n  ct_dropout: 0.50\n  init_std: 0.02\n\ntraining_overrides:\n  learning_rate: 0.001\n  weight_decay: 0.0\n  train_batch_size: 512\n  eval_batch_size: 1024\n  epochs: 30\n', 'configs/sasrec.yaml': '# Cấu hình mô hình SASRec — Self-Attentive Sequential Recommendation (Kang & McAuley, ICDM 2018)\n# Kiến trúc chuẩn RecBole: Item + Positional Embedding -> Causal Multi-Head Transformer Blocks -> Full-Catalog Dot Product\n\nmodel_name: sasrec\narchitecture: SASRec\n\nmodel_params:\n  embedding_size: 64\n  n_layers: 2\n  n_heads: 2\n  inner_size: 256\n  hidden_dropout_prob: 0.20\n  attn_dropout_prob: 0.20\n  hidden_act: gelu\n  layer_norm_eps: 1.0e-12\n  initializer_range: 0.02\n\ntraining_overrides:\n  learning_rate: 0.001\n  weight_decay: 0.0\n  train_batch_size: 512\n  eval_batch_size: 1024\n  epochs: 30\n  use_amp: false\n', 'configs/topology/role_gradient_study.yaml': '# Full-data successor to the local exploratory pilot; no audience/traffic task.\nmodels: [gru4rec, narm, sasrec]\nseeds: [2024, 2025]\nmethods: [baseline, role_input_scale, role_tangent, ph_h0_h1, role_ph]\nepochs: 30\ntopology_params:\n  neighbors: 6\n  max_clouds: 4\n  h1_weight: 1.0\n  lambda_topo: 0.15\n  lambda_anisotropy: 0.0\n  tail_threshold: 20\n  cosine_tau: 0.45\n  gradient_strength: 0.5\n  regularization_interval: 1\nprimary_outcome: rare_recall@20\nguardrail: overall_recall_drop_at_most_0.005\nselection: validation_mrr@20\nscope: tied_embedding_gradient_roles_and_persistent_homology\n'}
DATA_HASHES = {'item_mapping.csv': '7ef8636a4e74c9471f0e2d9805ae5b5c3fe3186dcb5e094f6fd2a86d75097f4a', 'train_examples.parquet': 'eb41c516b26ea163ce6d9cceefd6b6817d6c897f49dcc23d4ab0136d69225d04', 'validation_examples.parquet': '52cc7af596b5b708162781b9f24267bf49c4a0fd5e43f3d5c8df9302639e194c', 'test_examples.parquet': '54b5014dbfbb8a43507b06655421bca92930e8563aad41295cb6deaa37260668'}
for relative, contents in PAYLOAD.items():
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(contents, encoding='utf-8')
candidates = [p.parent for p in Path('/kaggle/input').rglob('item_mapping.csv')
              if (p.parent / 'train_examples.parquet').exists()]
source = None
for candidate in candidates:
    if all((candidate / name).exists() and hashlib.sha256((candidate / name).read_bytes()).hexdigest() == digest
           for name, digest in DATA_HASHES.items()):
        source = candidate
        break
if source is None:
    raise RuntimeError('Attached data do not match the reviewed Diginetica split/mapping. Do not train on a different protocol.')
target_data = WORK / 'data/processed/diginetica'
target_data.mkdir(parents=True, exist_ok=True)
for file in source.iterdir():
    if file.is_file():
        shutil.copy2(file, target_data / file.name)
os.chdir(WORK)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ripser'], check=True)
subprocess.run([sys.executable, '-B', '-m', 'pytest', 'tests/test_persistent_homology.py', 'tests/test_role_gradients.py', '-q'], check=True)
print('Diginetica data hashes and PH tests verified:', source)


## Huấn luyện và đối chứng

30 run = 3 mô hình × 2 seed × 5 phương án, tối đa 30 epoch/run với early stopping.
Thời gian phụ thuộc GPU/quota; log và checkpoint được ghi sau từng run/epoch.
Không tự thay đổi config để tìm kết quả thuận lợi. Nếu PH không hơn đối chứng,
kết quả đó vẫn được giữ trong bảng đánh giá.


In [ ]:
import concurrent.futures, torch, yaml
study = yaml.safe_load((WORK / 'configs/topology/role_gradient_study.yaml').read_text(encoding='utf-8'))
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before starting the full study.')
OUTPUT = WORK / 'results/ph_embedding_study'
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'locked_study.json').write_text(json.dumps(study, indent=2), encoding='utf-8')
jobs = [(model, seed) for model in study['models'] for seed in study['seeds']]
gpu_count = min(torch.cuda.device_count(), 2)

def worker(slot):
    for model, seed in jobs[slot::gpu_count]:
        destination = OUTPUT / f'{model}_{seed}'
        log = OUTPUT / f'{model}_{seed}.log'
        command = [sys.executable, '-B', 'scripts/train.py',
            '--models', model, '--seeds', str(seed), '--methods', *study['methods'],
            '--epochs', str(study['epochs']), '--full-data', '--evaluate-test',
            '--batch-size', '512', '--threads', '2', '--device', f'cuda:{slot}',
            '--config', 'configs/topology/role_gradient_study.yaml', '--output', str(destination)]
        environment = dict(os.environ, OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
        print('START', model, seed, 'GPU', slot, flush=True)
        with log.open('w', encoding='utf-8') as stream:
            subprocess.run(command, cwd=WORK, env=environment, stdout=stream, stderr=subprocess.STDOUT, check=True)
        if 'role_ph' in study['methods']:
            subprocess.run([sys.executable, '-B', 'scripts/report.py',
                '--roots', str(destination), '--output', str(destination / 'role_report')], cwd=WORK, check=True)
        for method in ('baseline', 'ph_h0_h1'):
            run = destination / model / method / f'seed_{seed}'
            subprocess.run([sys.executable, '-B', 'scripts/analyze.py',
                '--run', str(run), '--output', str(run / 'geometry_ph'), '--per-group', '200'], cwd=WORK, check=True)
        subprocess.run([sys.executable, '-B', 'scripts/calibrate.py',
            '--models', model, '--seeds', str(seed), '--checkpoint-root', str(destination),
            '--output', str(destination / 'popularity_score_control')], cwd=WORK, check=True)
        print('FINISHED', model, seed, flush=True)

with concurrent.futures.ThreadPoolExecutor(max_workers=gpu_count) as pool:
    futures = [pool.submit(worker, slot) for slot in range(gpu_count)]
    for future in futures:
        future.result()


In [ ]:
import pandas as pd
tables = []
for path in OUTPUT.glob('*/comparison.csv'):
    tables.append(pd.read_csv(path))
comparison = pd.concat(tables, ignore_index=True)
comparison.to_csv(OUTPUT / 'all_models_comparison.csv', index=False)
display(comparison.loc[comparison.target_group.isin(['all_warm', 'rare']),
    ['model', 'method', 'seed', 'split', 'num_examples', 'recall@20', 'mrr@20', 'frozen_drift']])
print('All measured outputs:', OUTPUT)
print('No result implies revenue, semantic quality or causal effects. Compare PH with cosine and dispersion before choosing it.')
